In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


import kagglesdk.kaggle_env as kaggle_env

if not hasattr(kaggle_env, "get_web_endpoint"):
    kaggle_env.get_web_endpoint = kaggle_env.get_endpoint

import kagglehub
import keras
import keras_cv

print("GPUs:", gpus)

In [ ]:
class CFG:
    verbose = 2  # Verbosity
    seed = 49  # Random seed
    neg_sample = 0.01 # Downsample negative class
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [256, 256]  # Input image size
    epochs = 20 # Training epochs
    batch_size = 64  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
BASE_PATH = "/mnt/disk1/home/cgiovannoni/isic24-dum"
DATA_PATH = f"{BASE_PATH}/data/isic24"
PROC_PATH = f"{BASE_PATH}/models/enhanced/enhanced/preprocessed_dataset"
ART_PATH = f"{BASE_PATH}/artifacts/enhanced"
FEATURE_SPACE_PATH = f"{ART_PATH}/feature_space"
RUNS_PATH = f"{ART_PATH}/runs"
SUMMARIES_PATH = f"{ART_PATH}/summaries"

In [ ]:
import pandas as pd

# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [ ]:
print(df['target'].value_counts())

In [ ]:
print("Class Distribution Before Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

# Sampling
majority_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
minority_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([majority_df, minority_df], axis=0).sample(frac=1.0)

print("\nClass Distribution After Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)
    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)

    # Decode before cache (so we don't decode every epoch)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed, reshuffle_each_iteration=True)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        opt.experimental_slack = True
        ds = ds.with_options(opt)

    ds = ds.batch(batch_size, drop_remainder=drop_remainder)

    # Keep augmentation AFTER cache so it's still random per epoch
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTO)

    ds = ds.prefetch(AUTO)
    return ds

def _to_feature_dict(df):
    feat_dict = {}
    for c in df.columns:
        s = df[c]
        if s.dtype == object or str(s.dtype).startswith('string'):
            feat_dict[c] = s.astype(str).to_numpy()
        else:
            feat_dict[c] = s.to_numpy()
    return feat_dict

In [ ]:
from tqdm import tqdm

# Build datasets
print("# Training:")
training_features = _to_feature_dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(
    training_ids, training_validation_hdf5, training_features, training_labels,
    batch_size=CFG.batch_size, shuffle=True, augment=True
)

print("# Validation:")
validation_features = _to_feature_dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(
    validation_ids, training_validation_hdf5, validation_features, validation_labels,
    batch_size=CFG.batch_size, shuffle=False, augment=False
)

print("# Testing:")
testing_features = _to_feature_dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(
    testing_ids, testing_hdf5, testing_features,
    batch_size=CFG.batch_size, shuffle=False, augment=False, cache=False
)

In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

# ---------------------------------------------------------------------------
# REPRODUCIBILITY FIX (2026-07-14).
# `training_ds` is built with shuffle=True, reshuffle_each_iteration=True and
# experimental_deterministic=False. Discretization (streaming quantile sketch) and
# StringLookup (vocab ordered by frequency) are ORDER-DEPENDENT, so adapt() produced
# slightly different statistics on every notebook session -- which is why reloading a
# checkpoint post-hoc does not reproduce the cached predictions (the error grows with
# alpha_tab: exact at alpha=0, max |dp| = 0.24 at alpha=1).
#
# Load the saved FeatureSpace when it exists; only adapt (and save) when it does not.
# Set FORCE_READAPT_FEATURE_SPACE = True to deliberately re-adapt.
# ---------------------------------------------------------------------------
FORCE_READAPT_FEATURE_SPACE = False

_fs_file = os.path.join(FEATURE_SPACE_PATH, "feature_space.keras")

if os.path.exists(_fs_file) and not FORCE_READAPT_FEATURE_SPACE:
    feature_space = keras.saving.load_model(_fs_file)
    print(f"FeatureSpace LOADED from {_fs_file} (deterministic; not re-adapted)")
else:
    training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
    feature_space.adapt(training_ds_with_no_labels)
    os.makedirs(FEATURE_SPACE_PATH, exist_ok=True)
    feature_space.save(_fs_file)
    print(f"FeatureSpace ADAPTED and saved to {_fs_file}")

print(feature_space.get_encoded_features().shape)

In [ ]:
# The previous cell already LOADS the canonical FeatureSpace when it exists, and only
# adapt()s + saves when it does not. Re-saving unconditionally here is a footgun: if the
# adapt branch ever runs (FORCE_READAPT=True, or the file goes missing), this line silently
# overwrites the canonical statistics the June checkpoints were trained with -- and
# artifacts/ is .gitignored, so there is NO recovery. Kept read-only on purpose.
filepath = os.path.join(FEATURE_SPACE_PATH, "feature_space.keras")
print(f"FeatureSpace in use: '{filepath}'")
print(f"  encoded dims: {feature_space.get_encoded_features().shape}")
print("  (not re-written here — the previous cell owns saving)")

In [ ]:
# Apply Feature Processing
augment_fn = build_augmenter()

auto = tf.data.AUTOTUNE

training_ds = training_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
training_ds = training_ds.map(augment_fn, num_parallel_calls=auto)
training_ds = training_ds.prefetch(auto)

validation_ds = validation_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
validation_ds = validation_ds.prefetch(auto)

testing_ds = testing_ds.map(
    lambda x: {'images': x['images'], 'features': feature_space(x['features'])},
    num_parallel_calls=auto,
)
testing_ds = testing_ds.prefetch(auto)

In [ ]:
for x, y in training_ds.take(1):
    print(type(x))
    print(type(x["features"]))
    if isinstance(x["features"], dict):
        print({k: v.shape for k, v in x["features"].items()})
    else:
        print(x["features"].shape, x["features"].dtype)

In [ ]:
# optimizer, loss & metric
adam = keras.optimizers.Adam(learning_rate=1e-4)
auc = keras.metrics.AUC(name='auc')
bce = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

## Modelling

### callbacks

In [ ]:
import math
import matplotlib.pyplot as plt

def get_lr_callback(batch_size=8, mode='cos', epochs=20, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode=CFG.lr_mode, plot=False)

#### shap

In [ ]:

_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])


def aggregate_raw_importance(encoded_feature_names, mean_abs_vec):
    agg = {}
    for j, val in enumerate(mean_abs_vec):
        name = encoded_feature_names[j]
        if "__" in name:
            raw = name.split("__", 1)[0]
        elif name.startswith("age_approx_bin_"):
            raw = "age_approx"
        else:
            raw = name
        agg.setdefault(raw, 0.0)
        agg[raw] += float(abs(val))
    return agg



def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    # ax.bar_label(container, fmt="{:.4f}", padding=4, fontsize=7)
    # adaptive barplot decimal precision
    for container in ax.containers:
        labels = []
        for v in container.datavalues:
            if abs(v) < 1e-3:
                labels.append(f"{v:.2e}")
            else:
                labels.append(f"{v:.4f}")
        ax.bar_label(container, labels=labels, padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]

def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
    ):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


import shap
from typing import Tuple, Dict, Any

class ShapEngine:
    """
    SHAP engine for the tabular 'features' input of the multimodal model.

    - Original model signature: model({"images": images, "features": features}) -> (batch, 1)
    - Internal wrapper: wrapped_model([features, images]) -> (batch, 1)
    - Explainer: shap.GradientExplainer on wrapped_model
    - We keep SHAP values only for the 'features' input (index 0 in [features, images]).
    """

    def __init__(
        self,
        background_size: int = 128,
        max_explain_samples: int = 64,
        top_k: int = 10,
        positive_threshold: float = 0.5,
        use_gradient_explainer: bool = True,
    ):
        self.background_size = background_size
        self.max_explain_samples = max_explain_samples
        self.top_k = top_k
        self.positive_threshold = positive_threshold
        self.use_gradient_explainer = use_gradient_explainer

        self._explainer = None
        self._wrapped_model = None   # Keras model: [features, images] -> original_model(...)
        self._feature_input_index = 0

    # ----------------- internal utils ----------------- #
    def _build_wrapped_model(
        self,
        base_model: tf.keras.Model,
        feature_shape,
        image_shape,
    ):
        """
        Build a Keras model that takes [features, images] and calls the original
        dict-based model internally.
        """
        feat_in = tf.keras.Input(shape=feature_shape[1:], name="shap_features")
        img_in = tf.keras.Input(shape=image_shape[1:], name="shap_images")

        out = base_model({"features": feat_in, "images": img_in})
        self._wrapped_model = tf.keras.Model(
            inputs=[feat_in, img_in],
            outputs=out,
            name="wrapped_for_shap",
        )

    def _build_explainer(
        self,
        base_model: tf.keras.Model,
        bg_features: np.ndarray,
        bg_images: np.ndarray,
    ):
        # Build wrapped model lazily
        if self._wrapped_model is None:
            self._build_wrapped_model(
                base_model,
                feature_shape=bg_features.shape,
                image_shape=bg_images.shape,
            )

        bg_inputs = [bg_features, bg_images]  # [features, images]

        if self.use_gradient_explainer:
            self._explainer = shap.GradientExplainer(self._wrapped_model, bg_inputs)
        else:
            self._explainer = shap.explainers.Deep(self._wrapped_model, bg_inputs)

    def _select_background_and_expl_set(
        self,
        features: np.ndarray,
        images: np.ndarray,
    ):
        n = features.shape[0]
        bg_idx = np.random.choice(n, size=min(self.background_size, n), replace=False)

        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.random.choice(n, size=self.max_explain_samples, replace=False)

        bg_features = features[bg_idx]
        bg_images = images[bg_idx]
        ex_features = features[ex_idx]
        ex_images = images[ex_idx]

        return (bg_features, bg_images), (ex_features, ex_images), ex_idx

    # ----------------- public API ----------------- #
    def explain(
        self,
        validation_data: Tuple[np.ndarray, np.ndarray, np.ndarray],
        model: tf.keras.Model,
        class_index: int = 0,  # irrelevant for scalar output
    ) -> Tuple[Dict[str, Any], np.ndarray, np.ndarray]:
        """
        validation_data: (features, images, labels) (all numpy arrays)

        Returns:
            stats: dict with
                - mean_abs_all: (n_features,)
                - mean_abs_pos: (n_features,)
                - mean_abs_neg: (n_features,)
                - topk_all_indices, topk_pos_indices, topk_neg_indices
                - explained_indices: indices into the original batch
            shap_values_features: (n_explain, n_features)
            preds: (n_explain, 1)
        """

        features, images, labels = validation_data

        (bg_features, bg_images), (ex_features, ex_images), ex_idx = \
            self._select_background_and_expl_set(features, images)

        # Lazy explainer init on wrapped model
        if self._explainer is None:
            self._build_explainer(model, bg_features, bg_images)

        ex_inputs = [ex_features, ex_images]  # [features, images]
        preds = self._wrapped_model(ex_inputs).numpy()  # (n_ex, 1)

        # SHAP values: list [phi_features, phi_images]
        shap_values_all_inputs = self._explainer.shap_values(ex_inputs)
        shap_values_features = shap_values_all_inputs[self._feature_input_index]
        # (n_explain, n_features)

        shap_values_features = np.array(shap_values_features)

        # Handle shapes:
        # - expected: (n_ex, n_features)
        # - you currently get: (n_ex, n_features, 1)
        if shap_values_features.ndim == 3:
          # If last dim is 1 → (n_ex, n_features, 1) → squeeze last
          if shap_values_features.shape[-1] == 1:
              shap_values_features = shap_values_features[..., 0]
          # If second dim is 1 → (n_ex, 1, n_features) → squeeze middle
          elif shap_values_features.shape[1] == 1:
              shap_values_features = shap_values_features[:, 0, :]
        # Now: (n_ex, n_features)

        # Global mean |SHAP|
        mean_abs_all = np.mean(np.abs(shap_values_features), axis=0)  # (n_features,)

        # Positive / negative groups
        pred_scores = preds[:, 0]
        pos_mask = pred_scores >= self.positive_threshold
        neg_mask = ~pos_mask

        if pos_mask.any():
            mean_abs_pos = np.mean(np.abs(shap_values_features[pos_mask]), axis=0)
        else:
            mean_abs_pos = np.zeros_like(mean_abs_all)

        if neg_mask.any():
            mean_abs_neg = np.mean(np.abs(shap_values_features[neg_mask]), axis=0)
        else:
            mean_abs_neg = np.zeros_like(mean_abs_all)

        def topk_indices(x: np.ndarray, k: int):
            # robust 1D top-k
            x = x.reshape(-1)
            k = min(k, x.shape[0])
            return np.argsort(x)[::-1][:k]

        topk_all_idx = topk_indices(mean_abs_all, self.top_k)
        topk_pos_idx = topk_indices(mean_abs_pos, self.top_k)
        topk_neg_idx = topk_indices(mean_abs_neg, self.top_k)

        stats = {
            "mean_abs_all": mean_abs_all,
            "mean_abs_pos": mean_abs_pos,
            "mean_abs_neg": mean_abs_neg,
            "topk_all_indices": topk_all_idx,
            "topk_pos_indices": topk_pos_idx,
            "topk_neg_indices": topk_neg_idx,
            "explained_indices": ex_idx,
        }

        return stats, shap_values_features, preds

from pathlib import Path
from typing import List

import numpy as np
import tensorflow as tf
from keras.callbacks import Callback

class ShapCallback(Callback):
    """
    Runs ShapEngine every N epochs on a frozen validation subset and produces:
      - grouped barplot (Global vs Pos vs Neg) for top-k raw features
      - 2x2 scatter for top-4 numeric raw features (pos vs neg predictions)

    Explanation set is built at on_train_begin by concatenating multiple
    batches from `data_source`, up to `max_samples` samples (and optionally
    up to `max_batches` batches).
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        validation_df,
        feat_cols: List[str],
        encoded_feature_names: List[str],
        base_dir: str,
        background_size: int = 256,
        max_explain_samples: int = 256,
        top_k: int = 15,
        positive_threshold: float = 0.5,
        every_n_epochs: int = 1,
        max_samples: int = 512,   # total samples in explanation set
        max_batches: int | None = None,  # limit num batches to read, or None
        rng_seed: int = 0,
    ):
        super().__init__()
        self.data_source = data_source
        self.validation_df = validation_df
        self.feat_cols = feat_cols
        self.encoded_feature_names = encoded_feature_names

        self.background_size = int(background_size)
        self.max_explain_samples = int(max_explain_samples)
        self.top_k = int(top_k)
        self.positive_threshold = float(positive_threshold)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)

        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        # frozen explanation set
        self._images = None
        self._features = None
        self._labels = None
        self._raw_batch_df = None

        # ShapEngine instance
        self._engine = ShapEngine(
            background_size=self.background_size,
            max_explain_samples=self.max_explain_samples,
            top_k=self.top_k,
            positive_threshold=self.positive_threshold,
        )

    # ---------- helpers ---------- #
    def _freeze_batch_from_dataset(self, ds: tf.data.Dataset):
        """
        Build explanation set by concatenating multiple batches from ds.

        Assumes ds yields (inputs, labels) where inputs is {'images','features'}.
        Also builds the corresponding raw_batch_df from validation_df (shuffle=False).
        """
        X_img_list, X_feat_list, y_list = [], [], []
        n_collected = 0

        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches is not None and b_idx >= self.max_batches:
                break

            if not isinstance(x, dict):
                raise ValueError("Expected dataset elements like ({'images','features'}, y).")

            images_b = x["images"].numpy()
            features_b = x["features"].numpy()
            labels_b = y.numpy()

            remaining = self.max_samples - n_collected
            if remaining <= 0:
                break

            if images_b.shape[0] > remaining:
                images_b = images_b[:remaining]
                features_b = features_b[:remaining]
                labels_b = labels_b[:remaining]

            X_img_list.append(images_b)
            X_feat_list.append(features_b)
            y_list.append(labels_b)

            n_collected += images_b.shape[0]
            if n_collected >= self.max_samples:
                break

        if n_collected == 0:
            raise RuntimeError("No samples collected for SHAP explanation set.")

        self._images = np.concatenate(X_img_list, axis=0)
        self._features = np.concatenate(X_feat_list, axis=0)
        self._labels = np.concatenate(y_list, axis=0)

        # assume validation_ds built from validation_df with shuffle=False
        self._raw_batch_df = (
            self.validation_df.iloc[:n_collected][self.feat_cols].reset_index(drop=True)
        )

    # ---------- Keras hooks ---------- #
    def on_train_begin(self, logs=None):
        self._freeze_batch_from_dataset(self.data_source)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch % self.every_n_epochs) != 0:
            return
        if self._images is None or self._features is None:
            return

        np.random.seed(self.rng_seed)

        # SHAP on frozen set with current weights
        stats, shap_vals, preds = self._engine.explain(
            validation_data=(self._features, self._images, self._labels),
            model=self.model,
            class_index=0,
        )

        mean_abs_all = stats["mean_abs_all"]
        mean_abs_pos = stats["mean_abs_pos"]
        mean_abs_neg = stats["mean_abs_neg"]

        global_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_all
        )
        pos_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_pos
        )
        neg_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_neg
        )

        epoch_name = f"epoch_{epoch:03d}"
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        # Persist arrays + tabular share for post-hoc analysis
        np.savez_compressed(
            epoch_dir / f"shap_arrays_{epoch_name}.npz",
            shap_values=shap_vals,                          # (n_ex, n_features_encoded)
            preds=preds,
            explained_indices=stats["explained_indices"],
            mean_abs_all=mean_abs_all,
            mean_abs_pos=mean_abs_pos,
            mean_abs_neg=mean_abs_neg,
        )

        # Tabular share = sum |SHAP| over tabular dims / total. Here all encoded features
        # ARE tabular (image branch is separate input), so share = mean_abs_all.sum() over all dims.
        # More useful: per-raw-feature share aggregated into "categorical vs numeric vs interaction" buckets,
        # but simplest paper-ready signal is the raw importance vector itself (already saved).

        total_abs = float(mean_abs_all.sum())
        top1_share = float(mean_abs_all.max() / (total_abs + 1e-12))
        top5_share = float(np.sort(mean_abs_all)[::-1][:5].sum() / (total_abs + 1e-12))

        import csv
        shap_csv = self.base_dir / "shap_summary.csv"
        write_header = not shap_csv.exists()
        with open(shap_csv, "a", newline="") as f:
            w = csv.writer(f)
            if write_header:
                w.writerow(["epoch", "total_abs_shap", "top1_share", "top5_share",
                            "n_explained", "n_pos", "n_neg"])
            w.writerow([
                int(epoch), total_abs, top1_share, top5_share,
                int(shap_vals.shape[0]),
                int((preds[:, 0] >= self.positive_threshold).sum()),
                int((preds[:, 0] <  self.positive_threshold).sum()),
            ])

        # 1) grouped barplot: global vs pos vs neg (top-k)
        plot_topk_grouped_raw_global_pos_neg(
            global_imp=global_importance_raw,
            pos_imp=pos_importance_raw,
            neg_imp=neg_importance_raw,
            epoch_num = epoch_name,
            k=self.top_k,
            figsize=(10, 5),
            out_path=str(
                epoch_dir
                / f"shap_grouped_raw_top{self.top_k:02d}_glob_pos_neg_{epoch_name}.png"
            ),
        )

        # 2) scatter grid: top-4 numeric features
        plot_shap_scatter_top4_numeric(
            shap_vals=shap_vals,
            preds=preds,
            encoded_feature_names=self.encoded_feature_names,
            global_importance_raw=global_importance_raw,
            raw_batch_df=self._raw_batch_df,
            explained_indices=stats["explained_indices"],
            positive_threshold=self.positive_threshold,
            epoch_num = epoch_name,
            out_path=str(
                epoch_dir / f"shap_top4_numeric_grid_{epoch_name}.png"
            ),
        )

#### Gradcam

In [ ]:
import os
from pathlib import Path
from typing import Optional, Tuple, List, Union

import cv2
import numpy as np
import tensorflow as tf
from keras import Model


class MultiModalGradCAMEngine:
    """
    Grad-CAM engine for multimodal models (images + tabular features).

    Features:
    - Robust target selection: defaults to the 4D tensor feeding the first GlobalAveragePooling2D.
    - Two modes:
        * 'context'     → uses real tabular features.
        * 'image_only'  → replaces features with a baseline (zeros or provided vector).
    - Output:
        * grid: single grid image (HxWx3, float [0..1]) of horizontally stacked triptychs
        * triptychs: list of per-sample images (orig+pred | heatmap | overlay), each Hx(3W)x3 in [0..1]
        * preds: model predictions for the batch (numpy array)
    - Visualization:
        * Accepts a separate display_images batch or a deprocess_fn to convert model-input images to [0,1].
    """

    def __init__(
        self,
        use_guided_grads: bool = True,
        mode: str = 'context',              # 'context' or 'image_only'
        baseline: Optional[np.ndarray] = None,
        deprocess_fn: Optional[callable] = None,  # fn(img_float) -> float [0,1]
    ):
        assert mode in ('context', 'image_only'), "mode must be 'context' or 'image_only'"
        self.use_guided_grads = use_guided_grads
        self.mode = mode
        self.baseline = baseline
        self.deprocess_fn = deprocess_fn

    def explain(
        self,
        validation_data: Tuple[dict, np.ndarray],                            # ({'images', 'features'}, labels)
        model: tf.keras.Model,
        class_index: int,
        layer_name: Optional[str] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], np.ndarray]:
        """
        Compute Grad-CAMs for a batch and return (grid, triptychs, preds).

        validation_data is # ({'images', 'features'}, labels)

        """
        # --- Unpack ---
        x, labels = validation_data
        images = x['images']
        features = x['features']

        # --- Build grad model ---
        conv_target_tensor = self._resolve_target_tensor(model, layer_name)
        grad_model = Model(inputs=model.inputs, outputs=[conv_target_tensor, model.output])

        # --- Build feeds in model.inputs order ---
        # feeds = [features, images]
        # if self.mode == 'image_only':Fif se
        #     feeds = self._apply_baseline_to_feeds(model, feeds, images, features)

        feeds = self._ordered_feeds(model, images=images, features=features)
        if self.mode == 'image_only':
            feeds = self._apply_baseline_to_feeds(model, feeds, images, features)

        # --- Forward + gradients ---
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(feeds, training=False)

            # target score
            if preds.shape.rank == 2 and preds.shape[-1] == 1:
                # binary sigmoid -> use logit(p) to avoid saturation artifacts
                p = tf.clip_by_value(preds[:, 0], 1e-6, 1.0 - 1e-6)
                target = tf.math.log(p) - tf.math.log(1.0 - p)  # (B,)
            else:
                # multi-class / multi-label (keep prob target; optionally log it)
                target = preds[:, class_index]  # (B,)

            loss = tf.reduce_sum(target)  # scalar

        grads = tape.gradient(loss, conv_out)  # (B, Hc, Wc, C)

        if self.use_guided_grads:
            grads = (
                tf.cast(conv_out > 0, grads.dtype)
                * tf.cast(grads > 0, grads.dtype)
                * grads
            )

        # --- CAMs (B, Hc, Wc) ---
        cams = self._generate_cams(conv_out, grads)  # list of (Hc, Wc)
        cams = np.stack(cams, axis=0)

        # --- Triptychs with display images ---
        triptychs = []
        preds_np = preds.numpy() if isinstance(preds, tf.Tensor) else preds
        for i in range(images.shape[0]):
            pred_text = self._format_pred_text(preds_np[i])
            vis_img = images[i]
            trip = self._compose_triptych(vis_img, cams[i], pred_text)
            triptychs.append(trip)

        grid = self._make_grid(triptychs)
        return grid, triptychs, preds_np

    # ------------------------ Saving helpers ------------------------
    def save_grid(self, grid: np.ndarray, output_dir: str, output_name: str) -> None:
        """Save the grid image to disk (RGB)."""
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        out_path = str(Path(output_dir) / output_name)
        img_u8 = np.uint8(255 * np.clip(grid, 0, 1))
        cv2.imwrite(out_path, cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    def save_triptychs(
        self,
        triptychs: List[np.ndarray],
        base_dir: str,
        epoch_name: str,
        prefix: str = "sample",
    ) -> None:
        """
        Save each triptych as:
          base_dir/epoch_name/{prefix}_{i:03d}.png
        """
        out_dir = Path(base_dir) / epoch_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(triptychs):
            img_u8 = np.uint8(255 * np.clip(img, 0, 1))
            path = out_dir / f"{prefix}_{i:03d}.png"
            cv2.imwrite(str(path), cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    # ------------------------ Internal helpers ------------------------
    def _make_grid(self, triptychs: List[np.ndarray]) -> np.ndarray:
        """Create a grid image; uses tf-explain grid_display if available, else a simple grid."""
        try:
            from tf_explain.utils.display import grid_display
            thumbs = np.stack([np.uint8(255 * np.clip(t, 0, 1)) for t in triptychs], axis=0)
            canvas = grid_display(thumbs)  # returns uint8 RGB
            return canvas.astype(np.float32) / 255.0
        except Exception:
            return self._naive_grid(triptychs)


    def _resolve_target_tensor(self, model: tf.keras.Model, layer_name: Optional[str]):
        """
        Prefer the 4D tensor feeding the first GlobalAveragePooling2D.
        If layer_name is provided, use that layer's output; else fallback to last 4D layer.
        """
        if layer_name is not None:
            return model.get_layer(layer_name).output
        # Prefer the tensor feeding the first GAP
        for l in model.layers:
            if isinstance(l, tf.keras.layers.GlobalAveragePooling2D):
                return l.input
        # Fallback to last 4D tensor in the graph
        for l in reversed(model.layers):
            try:
                shp = l.output_shape
            except Exception:
                continue
            if isinstance(shp, tuple) and len(shp) == 4:
                return l.output
        raise ValueError("No suitable 4D conv tensor found for Grad-CAM target")

    def _ordered_feeds(self, model: tf.keras.Model, images, features):
        """Create a list of inputs respecting model.inputs order (rank-4 → images, rank-2 → features)."""
        feeds = []
        for kt in model.inputs:
            if len(kt.shape) == 4:
                feeds.append(images)
            elif len(kt.shape) == 2:
                feeds.append(features)
            else:
                raise ValueError(f"Unexpected input rank for {kt.name}: {kt.shape}")
        return feeds

    def _apply_baseline_to_feeds(self, model: tf.keras.Model, feeds, images, features):
        """Replace the features tensor in feeds with a baseline (zeros by default)."""
        feat_inp = next(inp for inp in model.inputs if len(inp.shape) == 2)
        F = int(feat_inp.shape[-1])
        if self.baseline is None:
            baseline = tf.zeros((tf.shape(images)[0], F), dtype=getattr(features, 'dtype', tf.float32))
        else:
            base = tf.convert_to_tensor(self.baseline)
            if base.shape.rank == 1:
                baseline = tf.broadcast_to(base[None, :], [tf.shape(images)[0], F])
            else:
                baseline = base
        feeds_out = []
        for kt, fed in zip(model.inputs, feeds):
            feeds_out.append(baseline if len(kt.shape) == 2 else fed)
        return feeds_out

    def _generate_cams(self, conv_outputs: tf.Tensor, grads: tf.Tensor) -> List[np.ndarray]:
        """Compute per-sample weighted sum of conv activations; returns list of 2D CAMs normalized to [0,1]."""
        cams = []
        conv_np = conv_outputs.numpy() if isinstance(conv_outputs, tf.Tensor) else conv_outputs
        grads_np = grads.numpy() if isinstance(grads, tf.Tensor) else grads
        B = conv_np.shape[0]
        for i in range(B):
            # exclude 1-pixel border to avoid padding artifacts
            gr = grads_np[i][1:-1, 1:-1, :]
            co = conv_np[i][1:-1, 1:-1, :]
            weights = gr.mean(axis=(0, 1))  # (C,)
            cam = (co * weights[None, None, :]).sum(axis=-1)
            cam = np.maximum(cam, 0.0)
            cam_min, cam_max = cam.min(), cam.max()
            cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
            cams.append(cam)
        return cams

    def _format_pred_text(self, pred_row):
        """Format prediction text for display: binary → p=..., multi-class → ŷ=k (p=...)."""
        try:
            arr = np.array(pred_row)
        except Exception:
            return f"p={float(pred_row):.3f}"
        if arr.ndim == 0:
            return f"p={float(arr):.3f}"
        arr = arr.ravel()
        if arr.size == 1:
            return f"p={float(arr[0]):.3f}"
        else:
            k = int(np.argmax(arr))
            p = float(arr[k])
            return f"ŷ={k} (p={p:.3f})"

    def _compose_triptych(self, image_like: np.ndarray, cam_2d: np.ndarray, pred_text: str, colormap=cv2.COLORMAP_JET, image_weight=0.8) -> np.ndarray:
        """Create a 3-panel image (orig+text | heatmap | overlay), each panel HxWx3 in [0,1]."""
        img = self._to_display_rgb(image_like)
        H, W = img.shape[:2]
        # Heatmap
        heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam_2d, (W, H))),colormap)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB) / 255.0
        # Overlay
        overlay = image_weight * img + (1.0 - image_weight) * heat
        overlay = np.clip(overlay, 0, 1)
        # Original with prediction text
        orig = (img.copy() * 255).astype(np.uint8)
        txt = pred_text if isinstance(pred_text, str) else str(pred_text)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        orig = orig.astype(np.float32) / 255.0
        return np.hstack([orig, heat, overlay])

    # ---------------- Display helpers ----------------
    def _to_display_rgb(self, image: np.ndarray) -> np.ndarray:
        """
        Convert a model-input image to float RGB in [0,1] for visualization.
        Applies custom deprocess_fn if provided; else uses heuristics.
        """
        arr = image.numpy() if tf.is_tensor(image) else np.asarray(image)

        if self.deprocess_fn is not None:
            out = self.deprocess_fn(arr)
            return np.clip(self._ensure_rgb(out), 0.0, 1.0)

        # 1) uint8 → [0,1]
        if arr.dtype == np.uint8:
            arr = arr.astype(np.float32) / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # 2) Float ranges
        arr = arr.astype(np.float32)
        amin, amax = float(arr.min()), float(arr.max())

        # true [-1,1] only if there are negatives
        if amin < 0.0 and amax <= 1.1:
            arr = (arr + 1.0) / 2.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # already [0,1]
        if 0.0 <= amin and amax <= 1.1:
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # [0,255] floats
        if amin >= -1.0 and amax > 1.5 and amax <= 260:
            arr = arr / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # fallback: per-image min-max
        rng = (amax - amin) if (amax - amin) > 1e-8 else 1.0
        arr = (arr - amin) / rng
        return np.clip(self._ensure_rgb(arr), 0, 1)

    def _ensure_rgb(self, arr: np.ndarray) -> np.ndarray:
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return arr

    def _naive_grid(self, triptychs: List[np.ndarray], cols: int = 2) -> np.ndarray:
        if not triptychs:
            return np.zeros((32, 32, 3), dtype=np.float32)
        H, W, C = triptychs[0].shape
        rows = int(np.ceil(len(triptychs) / cols))
        grid = np.ones((rows * H, cols * W, C), dtype=np.float32)
        for idx, img in enumerate(triptychs):
            r = idx // cols
            c = idx % cols
            grid[r*H:(r+1)*H, c*W:(c+1)*W] = np.clip(img, 0, 1)
        return grid

from pathlib import Path
import numpy as np
import tensorflow as tf
import cv2
from keras.callbacks import Callback

class MultiModalGradCAMCallback(Callback):
    def __init__(self, engine, probe_batch, class_index=0, layer_name=None,
                 save_dir=None, tag="GradCAM/grid"):
        super().__init__()
        self.engine = engine
        self.probe_batch = probe_batch
        self.class_index = class_index
        self.layer_name = layer_name
        self.tag = tag
        self.save_dir = Path(save_dir) if save_dir else None
        if self.save_dir:
            self.save_dir.mkdir(parents=True, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        grid, _, _ = self.engine.explain(
            validation_data=self.probe_batch,
            model=self.model,
            class_index=self.class_index,
            layer_name=self.layer_name,
        )

        # Disk (PNG)
        if self.save_dir:
            out = (np.clip(grid, 0, 1) * 255).astype(np.uint8)
            cv2.imwrite(str(self.save_dir / f"epoch_{epoch:03d}.png"),
                        cv2.cvtColor(out, cv2.COLOR_RGB2BGR))


#### LORE


In [ ]:
!pip install git+https://github.com/kdd-lab/XAI-Lib.git

In [ ]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [ ]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

In [ ]:
import seaborn as sns

def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="{:.2f}", padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]


def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_lore_feature_frequency(
    freq_all: dict, freq_pos: dict, freq_neg: dict,
    k: int = 10, epoch_num=None, mode: str = '', figsize=(10, 5), out_path=None,
):
    """Barplot of rule-feature frequencies: Global / Pos / Neg predictions."""
    items = sorted(freq_all.items(), key=lambda kv: kv[1], reverse=True)[:k]
    if not items:
        return
    names = [n for n, _ in items]
    df = pd.DataFrame({
        'feature': names,
        'Global': [freq_all.get(n, 0) for n in names],
        'Pos preds': [freq_pos.get(n, 0) for n in names],
        'Neg preds': [freq_neg.get(n, 0) for n in names],
    })
    df_melt = df.melt(id_vars='feature', var_name='Class', value_name='Rule frequency')
    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt, y='feature', x='Rule frequency', hue='Class',
        palette=(sns.color_palette('bright')[4], sns.color_palette('bright')[0], sns.color_palette('bright')[3]),
        edgecolor='black',
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', padding=4, fontsize=7)
    title = f'LORE Feature Frequency in Rules (top {k})'
    if mode:
        title += f' [{mode}]'
    if epoch_num is not None:
        title += f' – {epoch_num}'
    plt.title(title)
    plt.xlabel('Rule frequency (# instances)')
    plt.ylabel('')
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


def save_lore_rules_text(rules: list, epoch_num=None, mode: str = '', out_path=None) -> str:
    """Write per-instance LORE rules to a .txt file."""
    header = 'LORE Rules'
    if mode:
        header += f' [{mode}]'
    if epoch_num is not None:
        header += f' – {epoch_num}'
    lines = [header, '=' * len(header), '']

    def _fmt(subset, label):
        lines.append(f'=== {label} ({len(subset)} instances) ===')
        for i, r in subset:
            lines.append(f'[inst {i:02d}] pred={r["pred"]:.3f} label={r["label"]}')
            lines.append(f'  RULE: {r["rule_str"]}')
            for cr in r.get('crules_str', []):
                lines.append(f'  COUNTERFACTUAL: {cr}')
            lines.append('')

    pos = [(i, r) for i, r in enumerate(rules) if r['pred'] >= 0.5]
    neg = [(i, r) for i, r in enumerate(rules) if r['pred'] < 0.5]
    _fmt(pos, 'POSITIVE PREDICTIONS')
    _fmt(neg, 'NEGATIVE PREDICTIONS')

    text = "\n".join(lines)
    if out_path:
        with open(out_path, 'w') as f:
            f.write(text)
    return text


from xailib.explainers.lore_explainer import LoreTabularExplainer
from collections import defaultdict
from typing import Dict, List, Optional


class LoreEngine:
    """
    LORE on raw (pre-FeatureSpace) tabular features.
    Categoricals are label-encoded before LORE, decoded back to strings
    in the predictor before FeatureSpace.
    """

    def __init__(
        self,
        feature_space,
        feat_cols: List[str],
        max_explain_samples: int = 32,
        positive_threshold: float = 0.5,
        lore_config: Optional[Dict] = None,
        lore_random_state: int = 0,
        oos_samples: int = 1000,
    ):
        self.feature_space = feature_space
        self.feat_cols = feat_cols
        self.max_explain_samples = int(max_explain_samples)
        self.positive_threshold = float(positive_threshold)
        # NOTE: 'random_state' inside lore_config is SILENTLY IGNORED by the xailib wrapper --
        # LoreTabularExplainer.fit() passes random_state=self.random_state (a class attribute,
        # default 0) and never reads the config dict. It is set explicitly in _build_explainer.
        self.lore_random_state = int(lore_random_state)
        self.oos_samples = int(oos_samples)
        self.lore_config = lore_config or {
            'neigh_type': 'random',
            'size': 300,
            'verbose': False,
        }
        self._image_encoder = None
        self._head_model = None
        self._predict_fn = None
        # Label encoding maps — built per explain() call from actual data
        self._label_maps = {}   # col → {string: int}
        self._inv_maps = {}     # col → {int: string}
        self._cat_cols = []

    # ── Model splitting ──────────────────────────────────────────────────
    def _build_sub_models(self, model):
        if self._image_encoder is not None:
            return
        # Cut the graph right after the (expensive) image backbone, at the
        # GlobalAveragePooling2D output. This pooled image feature is the single
        # trunk through which *all* image-side computation flows, so the head can
        # rebuild any downstream fusion scheme (plain concat, gamma fusion,
        # learnable scaling, adaptive gating) from [pooled_image, features].
        # The backbone stays in image_encoder (run once, cached); the head holds
        # only the cheap dense/fusion layers, so predicting over the LORE
        # neighbourhood is fast and works for every architecture.
        gap_layers = [l for l in model.layers
                      if isinstance(l, keras.layers.GlobalAveragePooling2D)]
        if not gap_layers:
            raise RuntimeError(
                "LORE: no GlobalAveragePooling2D found in "
                f"'{model.name}' to split the image branch."
            )
        img_emb = gap_layers[0].output            # (None, C) pooled image feature
        feat_in = model.input["features"]
        self._image_encoder = keras.Model(
            inputs=model.input["images"], outputs=img_emb, name="image_encoder",
        )
        # Reuse the intermediate pooled tensor directly as a head input (Keras
        # functional API): architecture-agnostic, no hardcoded layer names.
        self._head_model = keras.Model(
            inputs=[img_emb, feat_in], outputs=model.output, name="head_model",
        )
        self._predict_fn = tf.function(
            lambda embs, enc: self._head_model([embs, enc], training=False),
            reduce_retracing=True,
        )

    # ── Label encoding ───────────────────────────────────────────────────
    def _label_encode_df(self, df):
        """Label-encode categorical columns to integers. Returns encoded copy."""
        # self._cat_cols = [c for c in df.columns if df[c].dtype == 'object']
        self._cat_cols = [c for c in df.columns if c != 'target'
                          and not pd.api.types.is_numeric_dtype(df[c])]
        self._label_maps, self._inv_maps = {}, {}
        out = df.copy()
        for col in self._cat_cols:
            uniq = sorted(df[col].dropna().unique())
            lm = {v: i for i, v in enumerate(uniq)}
            self._label_maps[col] = lm
            self._inv_maps[col] = {i: v for v, i in lm.items()}
            out[col] = df[col].map(lm).astype('float64')
        return out

    # ── Predictor ────────────────────────────────────────────────────────
    def _make_predictor(self, image_embedding, feat_cols_no_target):
        head = self._head_model
        fs = self.feature_space
        emb_const = tf.constant(image_embedding[np.newaxis], dtype=tf.float32)
        cat_cols = self._cat_cols
        inv_maps = self._inv_maps
        predict_fn = self._predict_fn

        def predict(X) -> np.ndarray:
            if isinstance(X, pd.DataFrame):
                df = X.copy()
                cols = [c for c in df.columns if c != 'target']
            else:
                arr = np.asarray(X, dtype=np.float32)
                if arr.ndim == 1:
                    arr = arr.reshape(1, -1)
                df = pd.DataFrame(arr, columns=feat_cols_no_target)
                cols = feat_cols_no_target

            # Decode label-encoded categoricals back to strings
            feat_dict = {}
            for col in cols:
                if col in cat_cols:
                    ints = df[col].values.round().astype(int).clip(0, len(inv_maps[col]) - 1)
                    strings = [inv_maps[col].get(i, inv_maps[col][0]) for i in ints]
                    feat_dict[col] = tf.constant(strings)
                else:
                    feat_dict[col] = tf.constant(df[col].values.astype(np.float32))

            encoded = fs(feat_dict)
            n = encoded.shape[0]
            embs = tf.repeat(emb_const, n, axis=0)
            return predict_fn(embs, encoded).numpy()[:, 0]

        return predict

    # ── LORE wrappers ────────────────────────────────────────────────────
    def _build_explainer(self, predictor, background_df):
        class _BB:
            def __init__(self, fn, thr):
                self.fn, self.thr = fn, thr
            def predict(self, X):
                return (self.fn(X) >= self.thr).astype(int)
            def predict_proba(self, X):
                p = self.fn(X)
                return np.column_stack([1 - p, p])

        bb = _BB(predictor, self.positive_threshold)
        df_bg = background_df.copy().reset_index(drop=True)
        df_bg['target'] = bb.predict(df_bg).astype(np.int64)
        present = set(df_bg['target'].dropna().astype(int).unique().tolist())
        missing = [cls for cls in (0, 1) if cls not in present]
        if missing and len(df_bg) > 0:
            base_row = df_bg.iloc[0].copy()
            filler_rows = []
            for cls in missing:
                row = base_row.copy()
                row['target'] = cls
                filler_rows.append(row)
            df_bg = pd.concat([df_bg, pd.DataFrame(filler_rows)], ignore_index=True)
        exp = LoreTabularExplainer(bb=bb)
        exp.random_state = self.lore_random_state   # FIX: the config key is ignored by the wrapper
        exp.fit(df_bg, 'target', self.lore_config)
        self._predictor = predictor                 # kept for the out-of-sample fidelity below
        return exp

    # def _explain_one(self, lore_exp, x_series):
    #     try:
    #         exp = lore_exp.explain(x_series.values)
    #         rule = exp.getRules()
    #         crules = exp.getCounterfactualRules()
    #         premise = rule.get('premise', [])
    #
    #         def fmt(p):
    #             att, op, thr = p['att'], p['op'], float(p['thr'])
    #             if att in self._cat_cols:
    #                 idx = int(round(thr))
    #                 idx = max(0, min(idx, len(self._inv_maps[att]) - 1))
    #                 label = self._inv_maps[att][idx]
    #                 readable_op = '=' if op in ('>', '>=') else '≠'
    #                 return f"{att} {readable_op} '{label}'"
    #             return f"{att} {op} {thr:.4g}"
    #
    #         rule_str = ' AND '.join(fmt(p) for p in premise)
    #         rule_features = [p['att'] for p in premise]
    #         crules_str = [
    #             ' AND '.join(fmt(p) for p in cr.get('premise', []))
    #             for cr in (crules or [])
    #         ]
    #         return {'rule_str': rule_str, 'rule_features': rule_features, 'crules_str': crules_str}
    #     except Exception as e:
    #         return {'rule_str': f'[error: {e}]', 'rule_features': [], 'crules_str': []}

    def _explain_one(self, lore_exp, x_series):
        try:
            exp = lore_exp.explain(x_series.values)
            rule = exp.getRules()
            crules = exp.getCounterfactualRules()
            premise = rule.get('premise', [])

            def fmt(p):
                att, op, thr = p['att'], p['op'], float(p['thr'])
                if att in self._cat_cols:
                    idx = int(round(thr))
                    idx = max(0, min(idx, len(self._inv_maps[att]) - 1))
                    label = self._inv_maps[att][idx]
                    readable_op = '=' if op in ('>', '>=') else '!='
                    return f"{att} {readable_op} '{label}'"
                return f"{att} {op} {thr:.4g}"

            rule_str = ' AND '.join(fmt(p) for p in premise)
            rule_features = [p['att'] for p in premise]
            crules_str = [
                ' AND '.join(fmt(p) for p in cr.get('premise', []))
                for cr in (crules or [])
            ]
            # ---- native fidelity: dt.score(Z, Yb) -- the surrogate scored on the SAME neighbourhood
            # it was fitted on, i.e. IN-SAMPLE. A decision tree evaluated on its own training data
            # saturates by construction: this is a second, independent reason for the ~0.97-0.98
            # ceiling (the first being n=6 explained samples).
            fid = exp.expDict.get('fidelity')

            # ---- repaired fidelity: redraw a FRESH neighbourhood with the same generator and
            # score the surrogate out-of-sample, on continuous outputs and balanced classes.
            oos = {'lore_fidelity_oos_acc': float('nan'), 'lore_fidelity_oos_bacc': float('nan'),
                   'lore_fidelity_oos_r2': float('nan'), 'neigh_pos_rate': float('nan')}
            try:
                from sklearn.metrics import balanced_accuracy_score, r2_score
                dt = exp.exp.dt                                             # the surrogate tree
                Z = lore_exp.lore_explainer.neighgen_fn(x_series.values, self.oos_samples)
                p_model = self._predictor(Z)                                # continuous model prob
                y_model = (p_model >= self.positive_threshold).astype(int)
                y_surr = dt.predict(Z)
                oos['lore_fidelity_oos_acc'] = float((y_surr == y_model).mean())
                oos['lore_fidelity_oos_r2'] = float(r2_score(p_model, dt.predict_proba(Z)[:, 1]))
                oos['neigh_pos_rate'] = float(y_model.mean())               # did Z cross the boundary?
                if len(set(y_model.tolist())) > 1:
                    oos['lore_fidelity_oos_bacc'] = float(balanced_accuracy_score(y_model, y_surr))
            except Exception as e:
                oos['oos_error'] = f'{type(e).__name__}: {e}'

            return {'rule_str': rule_str, 'rule_features': rule_features, 'crules_str': crules_str,
                    'fidelity': float(fid) if fid is not None else float('nan'),
                    'rule_len': len(premise), **oos}
        except Exception as e:
            return {'rule_str': f'[LORE skipped: {type(e).__name__}: {e}]', 'rule_features': [], 'crules_str': [],
                    'fidelity': float('nan'), 'rule_len': 0}

    # ── Public API ───────────────────────────────────────────────────────
    def explain(self, raw_df, images, labels, model, mode='context'):
        self._build_sub_models(model)

        # Label-encode categoricals → all-numeric for LORE
        encoded_df = self._label_encode_df(raw_df)
        feat_cols_no_target = [c for c in encoded_df.columns if c != 'target']

        n = len(encoded_df)
        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.sort(np.random.choice(n, self.max_explain_samples, replace=False))

        ex_df = encoded_df.iloc[ex_idx].reset_index(drop=True)
        ex_labels = labels[ex_idx].ravel()

        if mode == 'context':
            img = images.mean(axis=0, keepdims=True).astype(np.float32)
        elif mode == 'tabular_only':
            img = np.zeros((1, *images.shape[1:]), dtype=np.float32)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        image_embedding = self._image_encoder(img, training=False).numpy()[0]
        predictor = self._make_predictor(image_embedding, feat_cols_no_target)
        preds = predictor(ex_df)
        lore_exp = self._build_explainer(predictor, encoded_df)

        freq_all, freq_pos, freq_neg = defaultdict(int), defaultdict(int), defaultdict(int)
        rules = []

        for i in range(len(ex_df)):
            result = self._explain_one(lore_exp, ex_df.iloc[i])
            print(f"[LORE {i+1}/{len(ex_df)}] rule_features={result['rule_features']}")
            result['pred'] = float(preds[i])
            result['label'] = int(ex_labels[i])
            rules.append(result)
            for feat in result['rule_features']:
                raw_feat = feat.split("=")[0] if "=" in feat else feat
                freq_all[raw_feat] += 1
                (freq_pos if preds[i] >= self.positive_threshold else freq_neg)[raw_feat] += 1

        def _agg(key):
            v = [r[key] for r in rules
                 if isinstance(r.get(key), float) and not np.isnan(r[key])]
            return float(np.mean(v)) if v else float('nan')

        def _sem(key):
            v = [r[key] for r in rules
                 if isinstance(r.get(key), float) and not np.isnan(r[key])]
            return float(np.std(v, ddof=1) / np.sqrt(len(v))) if len(v) > 1 else float('nan')

        _lens = [r['rule_len'] for r in rules if r.get('rule_len')]
        return {
            'feature_freq_all': dict(freq_all),
            'feature_freq_pos': dict(freq_pos),
            'feature_freq_neg': dict(freq_neg),
            'explained_indices': ex_idx,
            'n_explained': len(rules),
            'fidelity': _agg('fidelity'),                       # native, IN-SAMPLE (saturates)
            'fidelity_sem': _sem('fidelity'),                   # <- with n=6 this did not exist
            'fidelity_oos_acc': _agg('lore_fidelity_oos_acc'),  # out-of-sample accuracy
            'fidelity_oos_bacc': _agg('lore_fidelity_oos_bacc'),# balanced (kills class inflation)
            'fidelity_oos_r2': _agg('lore_fidelity_oos_r2'),    # continuous (kills 0.5-threshold)
            'fidelity_oos_r2_sem': _sem('lore_fidelity_oos_r2'),
            'neigh_pos_rate': _agg('neigh_pos_rate'),
            'rule_len': float(np.mean(_lens)) if _lens else float('nan'),
        }, rules, preds


    from pathlib import Path


class LoreCallback(keras.callbacks.Callback):
    """
    Runs LoreEngine every N epochs on a frozen validation subset.
    Operates on RAW features (pre-FeatureSpace) — XAI-Lib handles
    categorical encoding internally.
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        raw_feature_df: pd.DataFrame,
        feat_cols: List[str],
        feature_space,
        base_dir: str,
        max_explain_samples: int = 12,
        positive_threshold: float = 0.5,
        top_k: int = 10,
        every_n_epochs: int = 2,
        max_samples: int = 256,
        max_batches: Optional[int] = None,
        lore_config: Optional[Dict] = None,
        rng_seed: int = 0,
        modes: tuple = ('context',),
    ):
        super().__init__()
        self.data_source = data_source
        self.raw_feature_df = raw_feature_df  # pre-FeatureSpace validation DataFrame
        self.feat_cols = feat_cols
        self.top_k = int(top_k)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        self._images = None
        self._raw_df = None
        self._labels = None
        self.last_quality = {}   # aggregate LORE quality (fidelity); merged into run diagnostics

        self.modes = tuple(modes)
        self._engine = LoreEngine(
            feature_space=feature_space,
            feat_cols=feat_cols,
            max_explain_samples=int(max_explain_samples),
            positive_threshold=float(positive_threshold),
            lore_config=lore_config,
        )

    def _freeze_batch(self, ds: tf.data.Dataset):
        """Collect images and labels from dataset, raw features from DataFrame."""
        imgs, labels = [], []
        n = 0
        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches and b_idx >= self.max_batches:
                break
            remaining = self.max_samples - n
            if remaining <= 0:
                break
            imgs.append(x['images'].numpy()[:remaining])
            labels.append(y.numpy()[:remaining])
            n += imgs[-1].shape[0]

        if n == 0:
            raise RuntimeError('No samples collected for LORE.')
        self._images = np.concatenate(imgs)
        self._labels = np.concatenate(labels)
        # Take corresponding rows from raw DataFrame (same order, no shuffle in val)
        self._raw_df = self.raw_feature_df.iloc[:n][self.feat_cols].reset_index(drop=True)

    def on_train_begin(self, logs=None):
        self._freeze_batch(self.data_source)

    def on_epoch_end(self, epoch: int, logs=None):
        if epoch % self.every_n_epochs != 0 or self._images is None:
            return

        np.random.seed(self.rng_seed)
        epoch_name = f'epoch{epoch:03d}'
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        row = {'epoch': int(epoch)}
        # 'tabular_only' doubled the cost but never reached the report -- the published
        # lore_fidelity is the 'context' one. Dropping it pays for max_explain_samples 6 -> 32.
        for mode in self.modes:
            mode_dir = epoch_dir / mode
            mode_dir.mkdir(parents=True, exist_ok=True)

            stats, rules, preds = self._engine.explain(
                raw_df=self._raw_df,
                images=self._images,
                labels=self._labels,
                model=self.model,
                mode=mode,
            )

            self.last_quality[f'lore_fidelity_{mode}'] = stats.get('fidelity')
            self.last_quality[f'lore_rule_len_{mode}'] = stats.get('rule_len')
            row[f'lore_fidelity_{mode}'] = stats.get('fidelity')

            plot_lore_feature_frequency(
                freq_all=stats['feature_freq_all'],
                freq_pos=stats['feature_freq_pos'],
                freq_neg=stats['feature_freq_neg'],
                k=self.top_k,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_feature_freq_top{self.top_k:02d}_{mode}_{epoch_name}.png'),
            )

            save_lore_rules_text(
                rules=rules,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_rules_{mode}_{epoch_name}.txt'),
            )

        # headline quality (context mode) + per-epoch trajectory CSV (one row/epoch when run each epoch)
        self.last_quality['lore_fidelity'] = self.last_quality.get('lore_fidelity_context')
        self.last_quality['lore_rule_len'] = self.last_quality.get('lore_rule_len_context')
        _csv = self.base_dir / 'lore_fidelity_per_epoch.csv'
        pd.DataFrame([row]).to_csv(_csv, mode='a', header=not _csv.exists(), index=False)

### Modality Attendance

In [ ]:
class AlphaLogger(keras.callbacks.Callback):
    """Logs LearnableAlpha state per epoch. Writes incrementally to CSV."""
    def __init__(self, csv_path, layer_name="learnable_alpha"):
        super().__init__()
        self.csv_path = csv_path
        self.layer_name = layer_name
        self.rows = []

    def _read(self):
        layer = self.model.get_layer(self.layer_name)
        logit = float(layer.logit.numpy())
        alpha = float(tf.sigmoid(layer.logit).numpy())
        return logit, alpha

    def on_train_begin(self, logs=None):
        logit, alpha = self._read()
        self.rows.append({"epoch": -1, "logit": logit, "alpha": alpha, "beta": 1.0 - alpha})
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)

    def on_epoch_end(self, epoch, logs=None):
        logit, alpha = self._read()
        self.rows.append({"epoch": int(epoch), "logit": logit, "alpha": alpha, "beta": 1.0 - alpha})
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)

In [ ]:
# Shared runner
import os
import json
import time
import gc
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

os.makedirs(RUNS_PATH, exist_ok=True)
os.makedirs(SUMMARIES_PATH, exist_ok=True)


def _materialize_probe(n=8):
    probe_ds = build_dataset(
        validation_ids[:n],
        training_validation_hdf5,
        {k: v[:n] for k, v in validation_features.items()},
        validation_labels[:n],
        batch_size=n, shuffle=False, augment=False, cache=False,
        drop_remainder=False,
    )
    probe_ds = probe_ds.map(
        lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y)
    )
    xb, yb = next(iter(probe_ds))
    return xb, yb


class RunLogger(keras.callbacks.Callback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = csv_path
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.rows.append({
            "epoch": int(epoch),
            "loss": float(logs.get("loss", np.nan)),
            "val_loss": float(logs.get("val_loss", np.nan)),
            "auc": float(logs.get("auc", np.nan)),
            "val_auc": float(logs.get("val_auc", np.nan)),
        })
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)


def make_train_callbacks(run_dir, model_name, patience=4, track_alpha_layer=None):
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"best_{model_name}.keras")
    ckpt_cb = keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="val_auc", save_best_only=True,
        save_weights_only=False, mode="max",
    )
    es_cb = keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max",
        patience=patience, restore_best_weights=True,
        verbose=1,
    )
    callbacks = [ckpt_cb, es_cb, RunLogger(os.path.join(run_dir, "epoch_metrics.csv"))]
    if "lr_cb" in globals():
        callbacks.insert(0, lr_cb)

    if track_alpha_layer is not None:
        callbacks.append(AlphaLogger(csv_path=os.path.join(run_dir, "alpha_trajectory.csv"),layer_name=track_alpha_layer,))

    return callbacks, ckpt_path


def make_xai_callbacks(run_dir, model_name):
    xai_dir = os.path.join(run_dir, "xai")
    shap_dir = os.path.join(xai_dir, "shap")
    gradcam_dir = os.path.join(xai_dir, "gradcam")
    lore_dir = os.path.join(xai_dir, "lore")
    os.makedirs(shap_dir, exist_ok=True)
    os.makedirs(gradcam_dir, exist_ok=True)
    os.makedirs(lore_dir, exist_ok=True)

    global x_probe, y_probe
    if "x_probe" not in globals() or x_probe is None:
        x_probe, y_probe = _materialize_probe(n=8)

    shap_cb = ShapCallback(
        data_source=validation_ds, validation_df=validation_df,
        feat_cols=FEAT_COLS, encoded_feature_names=encoded_feature_names,
        base_dir=shap_dir, background_size=64, max_explain_samples=128,
        top_k=10, positive_threshold=0.5, every_n_epochs=1,
        max_samples=128, max_batches=5, rng_seed=0,
    )
    gm_cb = MultiModalGradCAMCallback(
        engine=MultiModalGradCAMEngine(use_guided_grads=True, mode="image_only"),
        probe_batch=(x_probe, y_probe), class_index=0,
        save_dir=gradcam_dir, tag=f"{model_name}/GradCAM",
    )
    # max_explain_samples was 6 -- lore_fidelity was the mean over SIX instances (the same 6 for
    # all 110 runs), which is why it saturated at ~0.97-0.98 and could not discriminate between
    # strategies: with n=6 there is no usable standard error. Raised to 32. The 'random_state' key
    # is dropped because the xailib wrapper IGNORES it (LoreEngine sets it on the wrapper instead).
    lore_cb = LoreCallback(
        data_source=validation_ds, raw_feature_df=validation_df,
        feat_cols=FEAT_COLS, feature_space=feature_space, base_dir=lore_dir,
        max_explain_samples=32, positive_threshold=0.5, top_k=10,
        every_n_epochs=1, max_samples=256, max_batches=2,
        lore_config={'neigh_type': 'random', 'size': 300},
        rng_seed=0,
    )
    return [shap_cb, gm_cb, lore_cb]


def run_xai_on_best_epoch(best_model, xai_callbacks, best_epoch, best_val_auc):
    logs = {"val_auc": float(best_val_auc)}
    any_done = False
    for cb in xai_callbacks:
        cb_name = cb.__class__.__name__
        try:
            if hasattr(cb, "every_n_epochs"):
                cb.every_n_epochs = 1
            cb.set_model(best_model)
            cb.set_params({"epochs": int(best_epoch + 1), "steps": None, "verbose": 0})
            if hasattr(cb, "on_train_begin"):
                cb.on_train_begin(logs={})
            cb.on_epoch_end(int(best_epoch), logs=logs)
            if hasattr(cb, "on_train_end"):
                cb.on_train_end(logs={})
            any_done = True
        except Exception as e:
            print(f"[XAI WARN] {cb_name} failed at epoch {best_epoch}: {type(e).__name__}: {e}")
    return any_done


def _format_settings(extra_meta):
    if not extra_meta:
        return "none"
    parts = []
    for k, v in extra_meta.items():
        parts.append(f"{k}={v:.6g}" if isinstance(v, float) else f"{k}={v}")
    return ", ".join(parts)


def _make_exp_tag():
    return datetime.now().strftime("%Y%m%d-%H%M%S")

def _make_setting_folder(setting_tag, epochs):
    parts = [setting_tag] if setting_tag else ["run"]
    if epochs is not None:
        parts.append(f"e{int(epochs)}")
    return "__".join(parts)


# ---- Post-training diagnostics helpers (model-agnostic) -----------------
def _ablate_batch(x, mode, strength=1.0):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        x["features"] = (1.0 - strength) * feat  # zero-out
    return x


def _eval_auc(model, ds, transform=None):
    m = keras.metrics.AUC()
    for x, y in ds:
        xb = transform(x) if transform else x
        m.update_state(y, model(xb, training=False))
    return float(m.result().numpy())


def modality_contribution(model, ds):
    """AUC drops when each modality ablated. balance ∈ [0,1]: 0=image-driven, 1=tabular-driven."""
    base = _eval_auc(model, ds)
    img_drop = base - _eval_auc(model, ds, lambda x: _ablate_batch(x, "img"))
    tab_drop = base - _eval_auc(model, ds, lambda x: _ablate_batch(x, "tab"))
    return {
        "base_auc": base,
        "img_drop_auc": float(img_drop),
        "tab_drop_auc": float(tab_drop),
        "modality_balance": float(tab_drop / (img_drop + tab_drop + 1e-8)),
    }


def brier_ece(model, ds, n_bins=15):
    ys, ps = [], []
    for x, y in ds:
        ps.append(tf.squeeze(model(x, training=False), -1).numpy())
        ys.append(np.squeeze(y.numpy()))
    y, p = np.concatenate(ys), np.concatenate(ps)
    brier = float(np.mean((p - y) ** 2))
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p <= hi if i == n_bins - 1 else p < hi)
        if m.any():
            ece += (m.sum() / len(p)) * abs(p[m].mean() - y[m].mean())
    return {"brier": brier, "ece": float(ece)}


def inference_ms(model, ds, n_iter=20):
    x_probe, _ = next(iter(ds.take(1)))
    _ = model(x_probe, training=False)  # warmup
    t0 = time.time()
    for _ in range(n_iter):
        _ = model(x_probe, training=False)
    return float((time.time() - t0) / n_iter * 1000)


def run_experiment(
    impl_name,
    build_model_fn,
    enable_xai,
    epochs,
    exp_tag=None,
    setting_tag=None,
    extra_meta=None,
    seed=None,
    patience=4,
    extra_callbacks_fn=None
):
    keras.backend.clear_session()
    gc.collect()

    run_seed = int(getattr(CFG, "seed", 42) if seed is None else seed)
    tf.keras.utils.set_random_seed(run_seed)

    if exp_tag is None:
        exp_tag = _make_exp_tag()

    setting_folder = _make_setting_folder(setting_tag, epochs)
    run_id = f"{exp_tag}__{setting_folder}"
    run_dir = os.path.join(RUNS_PATH, impl_name, exp_tag, setting_folder)
    os.makedirs(run_dir, exist_ok=True)

    model = build_model_fn(run_eagerly=False)
    model_name = f"{impl_name}__{setting_tag}" if setting_tag else impl_name

    print("=" * 110)
    print(f"[RUN START] impl={impl_name} | tag={setting_tag or '-'} | run_id={run_id} "
          f"| xai={bool(enable_xai)} | epochs={epochs} | seed={run_seed}")
    print("-" * 110)

    cfg = {
        "impl_name": impl_name,
        "exp_tag": exp_tag,
        "setting_tag": setting_tag,
        "run_id": run_id,
        "model_name": model_name,
        "epochs": int(epochs),
        "batch_size": int(CFG.batch_size),
        "seed": run_seed,
        "enable_xai": bool(enable_xai),
        "patience": int(patience),
    }
    if extra_meta is not None:
        cfg.update(extra_meta)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    callbacks, ckpt_path = make_train_callbacks(run_dir, model_name, patience=patience)
    if extra_callbacks_fn is not None:
        callbacks.extend(extra_callbacks_fn(run_dir))
    t0 = time.time()
    history = model.fit(
        training_ds, epochs=epochs, callbacks=callbacks,
        validation_data=validation_ds, verbose=CFG.verbose,
        class_weight=class_weights if "class_weights" in globals() else None,
    )
    train_seconds = time.time() - t0

    val_auc_hist = history.history.get("val_auc", [])
    if len(val_auc_hist) > 0:
        val_auc_arr = np.asarray(val_auc_hist, dtype=np.float32)
        best_epoch = int(np.nanargmax(val_auc_arr))
        best_val_auc = float(np.nanmax(val_auc_arr))
    else:
        best_epoch = -1
        best_val_auc = float("nan")

    xai_done = False
    lore_quality = {}
    if bool(enable_xai) and best_epoch >= 0:
        if os.path.exists(ckpt_path):
            best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
        else:
            best_model = model
        xai_callbacks = make_xai_callbacks(run_dir, model_name)
        if len(xai_callbacks) > 0:
            xai_done = run_xai_on_best_epoch(best_model, xai_callbacks, best_epoch, best_val_auc)
            for _cb in xai_callbacks:
                if isinstance(_cb, LoreCallback):
                    lore_quality = getattr(_cb, "last_quality", {}) or {}

    # ---- Post-training diagnostics (model-agnostic) ----
    diag = {}
    if best_epoch >= 0:
        eval_model = best_model if (bool(enable_xai) and best_epoch >= 0) else model
        diag.update(modality_contribution(eval_model, validation_ds))
        diag.update(brier_ece(eval_model, validation_ds))
        diag["inference_ms_per_batch"] = inference_ms(eval_model, validation_ds)
        diag.update(lore_quality)  # LORE fidelity alongside the other quantitative XAI metrics
        if globals().get("ENABLE_FAITHFULNESS_IN_RUN", False) and bool(enable_xai) \
                and "explanation_faithfulness" in globals():
            try:
                diag.update(explanation_faithfulness(
                    eval_model, run_dir, best_epoch, save=True, verbose=False))
            except Exception as e:
                print(f"[FAITH WARN] {type(e).__name__}: {e}")
        with open(os.path.join(run_dir, "diagnostics.json"), "w") as f:
            json.dump(diag, f, indent=2)

    cfg.update({
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "xai_best_epoch_only": True,
        "xai_done": bool(xai_done),
        **diag,
    })

    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history.history, f, indent=2)

    summary_df = pd.DataFrame([{
        "impl_name": impl_name,
        "exp_tag": exp_tag,
        "setting_tag": setting_tag,
        "run_id": run_id,
        "seed": run_seed,
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "train_seconds": float(train_seconds),
        "xai_done": bool(xai_done),
        "run_dir": run_dir,
        "model_name": model_name,
        **diag,
        **(extra_meta or {}),
    }])
    summary_df.to_csv(os.path.join(run_dir, "single_run_summary.csv"), index=False)

    print("-" * 110)
    print(f"[RUN END]| best_epoch={best_epoch} | best_val_auc={best_val_auc:.6f} "
          f"| xai_done={xai_done} | time={train_seconds:.1f}s")
    print("=" * 110)

    keras.backend.clear_session()
    gc.collect()
    return summary_df, run_dir


def run_model_experiment(
    impl_name, settings, build_model_from_setting_fn, enable_xai_fn,
    epochs, extra_meta_fn=None, seed=None, patience=4, exp_tag=None, extra_callbacks_fn=None,
):
    rows = []
    histories = {}
    if exp_tag is None:                        # NEW
        exp_tag = _make_exp_tag()
    summary_dir = os.path.join(RUNS_PATH, impl_name, exp_tag)
    os.makedirs(summary_dir, exist_ok=True)

    print(f"\n[EXPERIMENT START] impl={impl_name} | n_settings={len(settings)} | exp_tag={exp_tag}")

    for i, setting in enumerate(settings, 1):
        tag = setting["tag"]
        meta = extra_meta_fn(setting) if extra_meta_fn is not None else {}
        print(f"\n[SETTING {i}/{len(settings)}] tag={tag} | xai={bool(enable_xai_fn(setting))} | settings={_format_settings(meta)}")

        summary_df, run_dir = run_experiment(
            impl_name=impl_name,
            build_model_fn=lambda run_eagerly, s=setting: build_model_from_setting_fn(s, run_eagerly),
            enable_xai=bool(enable_xai_fn(setting)),
            epochs=epochs,
            exp_tag=exp_tag,            # propagate macro tag
            setting_tag=tag,
            extra_meta=meta,
            seed=seed,
            patience=patience,
            extra_callbacks_fn=extra_callbacks_fn,
        )
        rows.append(summary_df)
        run_id = summary_df.iloc[0]["run_id"]
        hist_path = os.path.join(run_dir, "history.json")
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                histories[run_id] = json.load(f)   # key by run_id, matches plots

    full_summary = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if not full_summary.empty:
        full_summary.to_csv(os.path.join(summary_dir, "summary.csv"), index=False)

    print(f"\n[EXPERIMENT END] impl={impl_name} | completed_settings={len(rows)}")
    return full_summary, histories, summary_dir

def analyze_sweep_shap(study_dir, combined_df, param_col="alpha"):
    """Cross-setting Wilcoxon + SHAP concentration analysis. Reads saved arrays."""
    from scipy import stats as scistats
    import seaborn as sns

    plots_dir = os.path.join(study_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # Wilcoxon paired matrix
    params = sorted(combined_df[param_col].unique())
    pivot = combined_df.pivot_table(index="seed", columns=param_col, values="best_val_auc")
    n = len(params)
    pmat = np.full((n, n), np.nan)
    for i, a in enumerate(params):
        for j, b in enumerate(params):
            if i >= j: continue
            x, y = pivot[a].dropna().values, pivot[b].dropna().values
            if len(x) == len(y) >= 2:
                try: pmat[i, j] = scistats.wilcoxon(x, y).pvalue
                except ValueError: pass

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(pmat, xticklabels=[f"{a:.3f}" for a in params],
                yticklabels=[f"{a:.3f}" for a in params],
                annot=True, fmt=".3f", cmap="viridis_r", vmin=0, vmax=0.1, ax=ax,
                cbar_kws={"label": "Wilcoxon p-value (capped 0.1)"})
    ax.set_title(f"Pairwise Wilcoxon (paired by seed, n={pivot.shape[0]})")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"wilcoxon_{param_col}_pmatrix.png"), dpi=300)
    plt.close()
    pd.DataFrame(pmat, index=params, columns=params).to_csv(
        os.path.join(study_dir, f"wilcoxon_{param_col}_pmatrix.csv"))

    # SHAP concentration per setting
    rows = []
    for _, row in combined_df.iterrows():
        npz = os.path.join(row["run_dir"], "xai", "shap",
                           f"epoch_{int(row['best_epoch']):03d}",
                           f"shap_arrays_epoch_{int(row['best_epoch']):03d}.npz")
        if not os.path.exists(npz): continue
        m = np.load(npz)["mean_abs_all"]
        total = m.sum() + 1e-12
        rows.append({
            "seed": row["seed"], param_col: row[param_col],
            "shap_total": float(m.sum()),
            "shap_top5_share": float(np.sort(m)[::-1][:5].sum() / total),
            "shap_entropy": float(-(m/total * np.log(m/total + 1e-12)).sum()),
        })
    if not rows: return
    shap_df = pd.DataFrame(rows)
    shap_df.to_csv(os.path.join(study_dir, "shap_concentration.csv"), index=False)

    agg = shap_df.groupby(param_col).agg(["mean", "std"]).reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col, ylab in zip(axes,
        ["shap_top5_share", "shap_entropy", "shap_total"],
        ["SHAP top-5 share", "SHAP entropy", "Total |SHAP| mass"]):
        ax.errorbar(agg[param_col], agg[(col, "mean")], yerr=agg[(col, "std")],
                    marker="o", capsize=4)
        ax.set_xlabel(param_col); ax.set_ylabel(ylab); ax.grid(alpha=0.3)
    fig.suptitle(f"SHAP behavior vs {param_col} (mean ± std)")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{param_col}_vs_shap_concentration.png"), dpi=300)
    plt.close()

    # ---- Diagnostics vs param (modality_balance, ece, brier) ----
    diag_cols = [c for c in ["modality_balance", "brier", "ece", "inference_ms_per_batch",
                             "lore_fidelity", "faithfulness_gain"]
                 if c in combined_df.columns]
    if diag_cols:
        agg_d = combined_df.groupby(param_col)[diag_cols].agg(["mean", "std"]).reset_index()
        fig, axes = plt.subplots(1, len(diag_cols), figsize=(5 * len(diag_cols), 4), squeeze=False)
        for ax, col in zip(axes[0], diag_cols):
            ax.errorbar(agg_d[param_col], agg_d[(col, "mean")], yerr=agg_d[(col, "std")],
                        marker="o", capsize=4)
            ax.set_xlabel(param_col); ax.set_ylabel(col); ax.grid(alpha=0.3)
        fig.suptitle(f"Diagnostics vs {param_col} (mean ± std)")
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f"{param_col}_vs_diagnostics.png"), dpi=300)
        plt.close()

    # ---- SHAP fidelity R²: shap_sum -> model output ----
    from sklearn.linear_model import LinearRegression
    fid_rows = []
    for _, row in combined_df.iterrows():
        npz = os.path.join(row["run_dir"], "xai", "shap",
                           f"epoch_{int(row['best_epoch']):03d}",
                           f"shap_arrays_epoch_{int(row['best_epoch']):03d}.npz")
        if not os.path.exists(npz): continue
        d = np.load(npz)
        x = d["shap_values"].sum(axis=1).reshape(-1, 1)
        y = d["preds"].ravel()
        r2 = float(LinearRegression().fit(x, y).score(x, y))
        fid_rows.append({"seed": row["seed"], param_col: row[param_col], "shap_fidelity_r2": r2})
    if fid_rows:
        fid_df = pd.DataFrame(fid_rows)
        fid_df.to_csv(os.path.join(study_dir, "shap_fidelity.csv"), index=False)
        agg_f = fid_df.groupby(param_col)["shap_fidelity_r2"].agg(["mean", "std"]).reset_index()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(agg_f[param_col], agg_f["mean"], yerr=agg_f["std"], marker="o", capsize=4)
        ax.set_xlabel(param_col); ax.set_ylabel("SHAP fidelity R²")
        ax.set_title(f"SHAP linear fidelity vs {param_col}")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f"{param_col}_vs_shap_fidelity.png"), dpi=300)
        plt.close()

In [ ]:
# plot runner helpers

PLOT_SHOW = True

def _finalize_plot(out_dir=None, fname=None, show=None):
    plt.tight_layout()
    if out_dir and fname:
        os.makedirs(out_dir, exist_ok=True)
        plt.savefig(os.path.join(out_dir, fname), dpi=300)
    should_show = PLOT_SHOW if show is None else bool(show)
    if should_show:
        plt.show()
    plt.close()

def plot_param_vs_best_auc(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "best_val_auc" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["best_val_auc"], marker="o")
    plt.xlabel(param_col); plt.ylabel("best_val_auc")
    plt.title(title or f"Best val AUC vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_best_val_auc.png")

def plot_param_vs_best_epoch(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "best_epoch" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["best_epoch"], marker="o")
    plt.xlabel(param_col); plt.ylabel("best_epoch")
    plt.title(title or f"Best epoch vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_best_epoch.png")

def plot_param_vs_train_time(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "train_seconds" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["train_seconds"], marker="o")
    plt.xlabel(param_col); plt.ylabel("train_seconds")
    plt.title(title or f"Train time vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_train_seconds.png")

def plot_val_auc_vs_epoch_sweep(df, histories, setting_col, title=None, out_dir=None, fname=None, smooth=True):
    if df.empty or "run_id" not in df.columns or setting_col not in df.columns or not histories:
        return

    d = df[["run_id", setting_col]].drop_duplicates().sort_values(setting_col)
    fig, ax = plt.subplots(figsize=(8, 5))

    plotted = 0
    for _, r in d.iterrows():
        y = histories.get(r["run_id"], {}).get("val_auc", [])
        if len(y) == 0:
            continue
        x = np.arange(1, len(y) + 1)
        label = f"{setting_col}={r[setting_col]}"
        if smooth and len(y) >= 4:
            from scipy.interpolate import make_interp_spline
            xs = np.linspace(x.min(), x.max(), 200)
            ys = make_interp_spline(x, y, k=3)(xs)
            line, = ax.plot(xs, ys, linewidth=1.6, label=label)
            ax.scatter(x, y, s=18, color=line.get_color(), zorder=3)
        else:
            ax.plot(x, y, marker="o", linewidth=1.6, label=label)
        plotted += 1

    if plotted == 0:
        plt.close(fig)
        return

    ax.set_xlabel("epoch"); ax.set_ylabel("val_auc")
    ax.set_title(title or f"val_auc vs epoch by {setting_col}")
    ax.minorticks_on()
    ax.grid(which="major", alpha=0.3)
    ax.grid(which="minor", alpha=0.15, linestyle=":")
    ax.legend(loc="best")
    _finalize_plot(out_dir, fname or f"val_auc_vs_epoch_by_{setting_col}.png")



def plot_param_vs_lore_fidelity(df, param_col, title=None, out_dir=None):
    """LORE fidelity per strategy (mirror of plot_param_vs_best_auc)."""
    if df.empty or param_col not in df.columns or "lore_fidelity" not in df.columns:
        return
    d = df.dropna(subset=["lore_fidelity"]).sort_values(param_col)
    if d.empty:
        return
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["lore_fidelity"], marker="o", color="#c44e52")
    plt.xlabel(param_col); plt.ylabel("lore_fidelity")
    plt.title(title or f"LORE fidelity vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_lore_fidelity.png")


def plot_lore_fidelity_vs_val_auc(df, param_col=None, title=None, out_dir=None, fname=None):
    """Scatter (one point per run): x=best_val_auc (performance), y=lore_fidelity (explanation)."""
    if df.empty or not {"best_val_auc", "lore_fidelity"}.issubset(df.columns):
        return
    d = df.dropna(subset=["best_val_auc", "lore_fidelity"])
    if d.empty:
        return
    plt.figure(figsize=(7, 5))
    if param_col and param_col in d.columns:
        sc = plt.scatter(d["best_val_auc"], d["lore_fidelity"], c=d[param_col],
                         cmap="viridis", s=60, edgecolors="black")
        plt.colorbar(sc, label=param_col)
        for _, r in d.iterrows():
            plt.annotate(f"{r[param_col]:.3g}", (r["best_val_auc"], r["lore_fidelity"]),
                         fontsize=7, xytext=(3, 3), textcoords="offset points")
    else:
        plt.scatter(d["best_val_auc"], d["lore_fidelity"], s=60, edgecolors="black")
    plt.xlabel("best_val_auc (performance)"); plt.ylabel("lore_fidelity (explanation)")
    plt.title(title or "LORE fidelity vs validation AUC (per strategy)")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, fname or "lore_fidelity_vs_val_auc.png")


def plot_lore_fidelity_vs_val_auc_by_epoch(df, out_dir=None, mode="context"):
    """Single strategy: aligns per-epoch lore_fidelity (xai/lore/lore_fidelity_per_epoch.csv)
    with per-epoch val_auc (history.json) and traces the (val_auc, lore_fidelity) trajectory.
    Meaningful when the LORE callback runs every epoch; with best-epoch-only it shows one point."""
    if df.empty or "run_dir" not in df.columns:
        return
    fcol = f"lore_fidelity_{mode}"
    for _, r in df.iterrows():
        run_dir = r.get("run_dir")
        if not isinstance(run_dir, str):
            continue
        fcsv = os.path.join(run_dir, "xai", "lore", "lore_fidelity_per_epoch.csv")
        hjson = os.path.join(run_dir, "history.json")
        if not (os.path.exists(fcsv) and os.path.exists(hjson)):
            continue
        fdf = pd.read_csv(fcsv)
        if fcol not in fdf.columns or fdf.empty:
            continue
        val_auc = json.load(open(hjson)).get("val_auc", [])
        ep = fdf["epoch"].astype(int).values
        keep = (ep >= 0) & (ep < len(val_auc))
        if not keep.any():
            continue
        x = np.asarray(val_auc)[ep[keep]]
        y = fdf[fcol].values[keep]
        tag = str(r.get("setting_tag", r.get("run_id", "run")))
        plt.figure(figsize=(6, 5))
        plt.plot(x, y, marker="o", color="#4c72b0")
        for xi, yi, e in zip(x, y, ep[keep]):
            plt.annotate(f"e{e}", (xi, yi), fontsize=7, xytext=(3, 3), textcoords="offset points")
        plt.xlabel("val_auc"); plt.ylabel(f"lore_fidelity ({mode})")
        plt.title(f"LORE fidelity vs val_auc by epoch \u2014 {tag}")
        plt.grid(alpha=0.3)
        _finalize_plot(out_dir, f"lore_fid_vs_valauc_by_epoch_{tag}.png")


def plot_lore_xai_suite(df, param_col, out_dir=None, by_epoch=True):
    """One call: per-strategy LORE fidelity, fidelity-vs-val_auc scatter (all strategies),
    and (optional) per-strategy per-epoch fidelity-vs-val_auc trajectory."""
    plot_param_vs_lore_fidelity(df, param_col, out_dir=out_dir)
    plot_lore_fidelity_vs_val_auc(df, param_col, out_dir=out_dir)
    if by_epoch:
        plot_lore_fidelity_vs_val_auc_by_epoch(
            df, out_dir=os.path.join(out_dir, "lore_by_epoch") if out_dir else None)


In [ ]:
# =====================================================================
# Explanation-Guided Deletion / Insertion Faithfulness AUC
# ---------------------------------------------------------------------
# Primary explanation-quality metric, comparable across strategies.
#   s(x)            = logit(p_model(x))            (positive-class logit)
#   C_exp(q)        = mean_i[ s(x_i) - s(x_i with top-q% evidence removed) ]
#   C_rand(q)       = same budget removed at random (mean over n_random)
#   deletion_auc    = trapz_q C(q)   (q=0 -> 0 anchor)
#   faithfulness_gain = deletion_auc_exp - deletion_auc_rand   (higher = better)
# Insertion is the symmetric secondary metric (start from neutral baseline,
# re-insert top-q% evidence). Computed for tabular (SHAP), image (Grad-CAM)
# and joint (both modalities together). Reuses saved SHAP arrays so no
# GradientExplainer re-fit is needed.
# =====================================================================
import os, json, glob, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import cv2

# ---- Config (tweak to control cost) ---------------------------------
FAITH_N_SAMPLES    = 128
FAITH_BUDGETS      = (0.05, 0.10, 0.20, 0.40, 0.60)
FAITH_METHODS      = ("tabular", "image", "joint")
FAITH_DO_INSERTION = True          # set False if too heavy
FAITH_N_RANDOM     = 3             # random-baseline repetitions
FAITH_IMG_GRID     = 16            # 16x16 patches over 256x256
FAITH_IMG_REPLACE  = "blur"        # 'blur' | 'mean'
FAITH_RNG_SEED     = 0
FAITH_EVAL_BATCH   = 64
FAITH_SAVE_PLOTS   = False         # per-run deletion/insertion curve PNGs (off: keep only the aggregate per-strategy plots)
ENABLE_FAITHFULNESS_IN_RUN = True  # hook inside run_experiment for future runs

_FAITH_TAB_BASELINE = None         # lazily built {col: median/mode}
_TRAPZ = getattr(np, "trapezoid", np.trapz)  # numpy>=2 renamed trapz -> trapezoid


def _logit_np(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def _build_tabular_baseline():
    """Median for numeric raw features, most-frequent token for categoricals."""
    base = {}
    for col in FEAT_COLS:
        s = training_df[col]
        if pd.api.types.is_numeric_dtype(s):
            base[col] = float(s.median())
        else:
            base[col] = s.astype(str).mode().iloc[0]
    return base


def faith_subsets(k=5, n=FAITH_N_SAMPLES, seed=0, n_val=None):
    """`k` DISJOINT random subsets of `n` validation samples each.

    Subset 0 is the canonical first-`n` (so its numbers must reproduce the published
    faithfulness.json); subsets 1..k-1 are disjoint random draws from the rest. This is
    what answers "would a DIFFERENT batch of 128 give the same answer?" -- the published
    numbers rest on a single batch of 128 out of 1091 (11.7%)."""
    n_val = n_val or len(validation_df)
    assert k * n <= n_val, f"{k} x {n} > {n_val} validation samples: subsets cannot be disjoint"
    rest = np.setdiff1d(np.arange(n_val), np.arange(n))
    rest = np.random.default_rng(seed).permutation(rest)
    return [np.arange(n)] + [np.sort(rest[i * n:(i + 1) * n]) for i in range(k - 1)]


def _build_faith_subset(n, subset_idx=None):
    """Validation samples for the faithfulness run.

    subset_idx=None -> first `n` (the notebook's original, deterministic behaviour).
    subset_idx=array -> those specific validation rows (used by the 5-subset check).
    """
    raw_full = validation_df.reset_index(drop=True)
    take_n = n if subset_idx is None else int(np.max(subset_idx)) + 1
    imgs, labs, got = [], [], 0
    for x, y in validation_ds:
        b = int(x["images"].shape[0])
        take = min(b, take_n - got)
        imgs.append(x["images"][:take].numpy())
        labs.append(np.asarray(y)[:take].reshape(-1))
        got += take
        if got >= take_n:
            break
    imgs = np.concatenate(imgs, 0).astype(np.float32)
    labs = np.concatenate(labs, 0)
    if subset_idx is not None:
        subset_idx = np.asarray(subset_idx, int)
        return (raw_full.iloc[subset_idx][FEAT_COLS].reset_index(drop=True),
                imgs[subset_idx], labs[subset_idx])
    raw_df = raw_full.iloc[:got][FEAT_COLS].reset_index(drop=True)
    return raw_df, imgs, labs


def _load_shap_values(run_dir, best_epoch, model, raw_df, imgs):
    """Reuse the per-sample SHAP arrays saved by ShapCallback; fallback to recompute."""
    shap_dir = os.path.join(run_dir, "xai", "shap")
    cand = os.path.join(shap_dir, f"epoch_{int(best_epoch):03d}",
                        f"shap_arrays_epoch_{int(best_epoch):03d}.npz")
    if not os.path.exists(cand):
        hits = sorted(glob.glob(os.path.join(shap_dir, "epoch_*", "shap_arrays_epoch_*.npz")))
        cand = hits[-1] if hits else None
    if cand is not None and os.path.exists(cand):
        d = np.load(cand)
        sv = np.asarray(d["shap_values"])
        if sv.ndim == 3:
            sv = sv[..., 0] if sv.shape[-1] == 1 else sv[:, 0, :]
        ei = np.asarray(d["explained_indices"]).astype(int).ravel()
        keep = ei < len(raw_df)
        return sv[keep], ei[keep]
    # ---- fallback: recompute SHAP on the subset ----
    enc = np.asarray(feature_space(_to_feature_dict(raw_df)))
    eng = ShapEngine(background_size=min(64, len(raw_df)),
                     max_explain_samples=len(raw_df), top_k=10)
    stats, sv, _ = eng.explain((enc, imgs, np.zeros(len(raw_df))), model)
    return np.asarray(sv), np.asarray(stats["explained_indices"]).astype(int)


def _persample_raw_importance(shap_values):
    """(n_explain, n_encoded) |SHAP| -> (n_explain, n_raw_features) summed per raw feature."""
    n = shap_values.shape[0]
    imp = np.zeros((n, len(FEAT_COLS)), dtype=np.float64)
    for ci, col in enumerate(FEAT_COLS):
        idx = encoded_indices_for_raw(encoded_feature_names, col)
        if idx.size:
            imp[:, ci] = np.abs(shap_values[:, idx]).sum(axis=1)
    return imp


def _encode_raw(df):
    return np.asarray(feature_space(_to_feature_dict(df)))


def _score_logit(model, enc_feats, imgs, batch=None):
    batch = batch or FAITH_EVAL_BATCH
    enc = tf.convert_to_tensor(enc_feats)
    out = []
    for i in range(0, len(imgs), batch):
        p = model({"images": tf.convert_to_tensor(imgs[i:i + batch]),
                   "features": enc[i:i + batch]}, training=False)
        out.append(tf.squeeze(p, -1).numpy())
    return _logit_np(np.concatenate(out))


def _perturb_tabular_encoded(raw_df, order, k, deletion=True):
    """deletion: top-k raw features -> training baseline.
       insertion: start all-baseline, restore top-k real values. Returns encoded (n,128)."""
    global _FAITH_TAB_BASELINE
    if _FAITH_TAB_BASELINE is None:
        _FAITH_TAB_BASELINE = _build_tabular_baseline()
    base = _FAITH_TAB_BASELINE
    n = len(raw_df)
    sel = np.zeros((n, len(FEAT_COLS)), dtype=bool)
    if k > 0:
        rows = np.arange(n)[:, None]
        sel[rows, order[:, :k]] = True
    df = raw_df.copy()
    if deletion:
        for ci, col in enumerate(FEAT_COLS):
            m = sel[:, ci]
            if m.any():
                df.loc[m, col] = base[col]
    else:
        for col in FEAT_COLS:
            df[col] = base[col]
        for ci, col in enumerate(FEAT_COLS):
            m = sel[:, ci]
            if m.any():
                df.loc[m, col] = raw_df.loc[m, col].values
    return _encode_raw(df)


def _gradcam_maps(model, imgs, enc_feats, batch=None):
    """Per-sample Grad-CAM saliency upsampled to image size, in [0,1]."""
    batch = batch or FAITH_EVAL_BATCH
    conv = None
    for l in model.layers:
        if isinstance(l, keras.layers.GlobalAveragePooling2D):
            conv = l.input
            break
    if conv is None:
        for l in reversed(model.layers):
            shp = getattr(l, "output_shape", None)
            if isinstance(shp, tuple) and len(shp) == 4:
                conv = l.output
                break
    grad_model = keras.Model(model.inputs, [conv, model.output])
    enc = tf.convert_to_tensor(enc_feats)
    maps = []
    H, W = imgs.shape[1], imgs.shape[2]
    for i in range(0, len(imgs), batch):
        ib = tf.convert_to_tensor(imgs[i:i + batch])
        fb = enc[i:i + batch]
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model({"images": ib, "features": fb}, training=False)
            p = tf.clip_by_value(preds[:, 0], 1e-6, 1.0 - 1e-6)
            loss = tf.reduce_sum(tf.math.log(p) - tf.math.log(1.0 - p))
        grads = tape.gradient(loss, conv_out)
        grads = tf.cast(conv_out > 0, grads.dtype) * tf.cast(grads > 0, grads.dtype) * grads
        w = tf.reduce_mean(grads, axis=(1, 2))                       # (b, C)
        cam = tf.nn.relu(tf.reduce_sum(conv_out * w[:, None, None, :], axis=-1))  # (b,Hc,Wc)
        cam = cam.numpy()
        for j in range(cam.shape[0]):
            c = cam[j] - cam[j].min()
            mx = c.max()
            c = c / mx if mx > 0 else c
            maps.append(cv2.resize(c, (W, H)))
    return np.stack(maps, 0)


def _patch_saliency_order(cams, grid):
    n, H, W = cams.shape
    ph, pw = H // grid, W // grid
    s = cams[:, :grid * ph, :grid * pw].reshape(n, grid, ph, grid, pw).mean(axis=(2, 4))
    return np.argsort(-s.reshape(n, grid * grid), axis=1)           # desc, (n, grid*grid)


FAITH_BLUR_SIGMA = 8   # production value; Stage 3 sweeps this to test OOD sensitivity


def _image_replacement(imgs, sigma=None):
    if FAITH_IMG_REPLACE == "mean":
        m = imgs.mean(axis=(1, 2), keepdims=True)
        return np.broadcast_to(m, imgs.shape).astype(imgs.dtype).copy()
    sigma = FAITH_BLUR_SIGMA if sigma is None else sigma
    repl = np.empty_like(imgs)
    for i in range(len(imgs)):
        repl[i] = cv2.GaussianBlur(imgs[i], (0, 0), sigmaX=sigma)
    return repl


def _perturb_image(imgs, repl, order, k, grid, deletion=True):
    n, H, W, C = imgs.shape
    ph, pw = H // grid, W // grid
    out = imgs.copy() if deletion else repl.copy()
    src = repl if deletion else imgs
    for i in range(n):
        for pidx in order[i, :k]:
            r, c = divmod(int(pidx), grid)
            y0, x0 = r * ph, c * pw
            out[i, y0:y0 + ph, x0:x0 + pw, :] = src[i, y0:y0 + ph, x0:x0 + pw, :]
    return out


def _bootstrap_ci(vals, n_boot=2000, seed=0):
    vals = np.asarray(vals, float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return (float("nan"), float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    bs = rng.choice(vals, size=(n_boot, vals.size), replace=True).mean(axis=1)
    return float(vals.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def explanation_faithfulness(model, run_dir, best_epoch, n_samples=None, budgets=None,
                             methods=None, do_insertion=None, n_random=None, grid=None,
                             rng_seed=None, save=True, verbose=True,
                             subset_idx=None, blur_sigma=None, tab_imp_override=None,
                             out_name="faithfulness", out_dir=None):
    """Defaults reproduce the original behaviour exactly.

    subset_idx       : validation rows to explain (default: first FAITH_N_SAMPLES).
    blur_sigma       : image-removal blur strength (default: FAITH_BLUR_SIGMA = 8).
    tab_imp_override : (n, 39) per-sample raw importance from ANY attributor, replacing the
                       |SHAP| ranking loaded from disk. This is how Stage 2 feeds in the
                       aligned Permutation attributor.
    out_name         : basename prefix for every file written, so variants never collide.
    out_dir          : WHERE to write. Default (None) = run_dir/xai/faithfulness/, i.e. next to the
                       published artifacts. The robustness stages pass a directory under
                       runs/_robustness/ instead, so the new checks stay physically separate from
                       the published results and can be deleted wholesale without touching them.
    """
    """Compute deletion/insertion faithfulness for a trained model. Saves artifacts
    under run_dir/xai/faithfulness/ and returns a flat dict of headline metrics."""
    t0 = time.time()
    n_samples = n_samples or FAITH_N_SAMPLES
    budgets = tuple(budgets or FAITH_BUDGETS)
    methods = tuple(methods or FAITH_METHODS)
    do_insertion = FAITH_DO_INSERTION if do_insertion is None else do_insertion
    n_random = FAITH_N_RANDOM if n_random is None else n_random
    grid = grid or FAITH_IMG_GRID
    rng = np.random.default_rng(FAITH_RNG_SEED if rng_seed is None else rng_seed)
    q_grid = np.array((0.0,) + budgets, dtype=np.float64)

    need_tab = ("tabular" in methods) or ("joint" in methods)
    need_img = ("image" in methods) or ("joint" in methods)

    raw_df, imgs, labs = _build_faith_subset(n_samples, subset_idx=subset_idx)
    tab_imp = None
    if need_tab:
        if tab_imp_override is not None:
            tab_imp = np.asarray(tab_imp_override, float)
            assert tab_imp.shape == (len(raw_df), len(FEAT_COLS)), (
                f"tab_imp_override must be ({len(raw_df)}, {len(FEAT_COLS)}), got {tab_imp.shape}")
        else:
            sv, ei = _load_shap_values(run_dir, best_epoch, model, raw_df, imgs)
            raw_df = raw_df.iloc[ei].reset_index(drop=True)
            imgs, labs = imgs[ei], labs[ei]
            tab_imp = _persample_raw_importance(sv)[:len(raw_df)]
    n = len(raw_df)
    n_raw, n_patches = len(FEAT_COLS), grid * grid

    enc_orig = _encode_raw(raw_df)
    s_orig = _score_logit(model, enc_orig, imgs)
    preds_orig = 1.0 / (1.0 + np.exp(-s_orig))

    if need_img:
        cams = _gradcam_maps(model, imgs, enc_orig)
        img_order_exp = _patch_saliency_order(cams, grid)
        repl = _image_replacement(imgs, sigma=blur_sigma)

    def _tab_order(exp):
        if exp:
            return np.argsort(-tab_imp, axis=1)
        return np.argsort(rng.random((n, n_raw)), axis=1)

    def _img_order(exp):
        if exp:
            return img_order_exp
        return np.argsort(rng.random((n, n_patches)), axis=1)

    def _drops(method, exp, deletion):
        tab_o = _tab_order(exp) if method in ("tabular", "joint") else None
        img_o = _img_order(exp) if method in ("image", "joint") else None
        mat = np.zeros((n, len(q_grid)))
        for qi, q in enumerate(q_grid):
            enc, ims = enc_orig, imgs
            if method in ("tabular", "joint"):
                k = int(round(q * n_raw))
                enc = _perturb_tabular_encoded(raw_df, tab_o, k, deletion=deletion)
            if method in ("image", "joint"):
                k = int(round(q * n_patches))
                ims = _perturb_image(imgs, repl, img_o, k, grid, deletion=deletion)
            mat[:, qi] = _score_logit(model, enc, ims)
        return mat

    results = {"n_faith_samples": int(n), "faith_budgets": list(budgets),
               "blur_sigma": float(FAITH_BLUR_SIGMA if blur_sigma is None else blur_sigma),
               "tab_ranking": "shap_gradient" if tab_imp_override is None else "override"}
    npz = {"q_grid": q_grid, "s_orig": s_orig, "labels": labs, "preds_orig": preds_orig}
    curves_json = {}

    for method in methods:
        # ---- deletion: C(q) = s_orig - s_perturbed ----
        del_exp = s_orig[:, None] - _drops(method, exp=True, deletion=True)
        del_rand = np.zeros((n, len(q_grid)))
        for _ in range(n_random):
            del_rand += s_orig[:, None] - _drops(method, exp=False, deletion=True)
        del_rand /= max(1, n_random)
        auc_de = float(_TRAPZ(del_exp.mean(0), q_grid))
        auc_dr = float(_TRAPZ(del_rand.mean(0), q_grid))
        results[f"deletion_auc_{method}_exp"] = auc_de
        results[f"deletion_auc_{method}_rand"] = auc_dr
        results[f"faithfulness_{method}_del_gain"] = auc_de - auc_dr
        npz[f"del_exp_{method}"] = del_exp
        npz[f"del_rand_{method}"] = del_rand
        curves_json[f"del_exp_{method}"] = del_exp.mean(0).tolist()
        curves_json[f"del_rand_{method}"] = del_rand.mean(0).tolist()

        if do_insertion:
            # ---- insertion: recovery relative to all-baseline (q=0) ----
            ins_exp = _drops(method, exp=True, deletion=False)
            ins_exp = ins_exp - ins_exp[:, [0]]
            ins_rand = np.zeros((n, len(q_grid)))
            for _ in range(n_random):
                m = _drops(method, exp=False, deletion=False)
                ins_rand += m - m[:, [0]]
            ins_rand /= max(1, n_random)
            auc_ie = float(_TRAPZ(ins_exp.mean(0), q_grid))
            auc_ir = float(_TRAPZ(ins_rand.mean(0), q_grid))
            results[f"insertion_auc_{method}_exp"] = auc_ie
            results[f"insertion_auc_{method}_rand"] = auc_ir
            results[f"faithfulness_{method}_ins_gain"] = auc_ie - auc_ir
            npz[f"ins_exp_{method}"] = ins_exp
            npz[f"ins_rand_{method}"] = ins_rand
            curves_json[f"ins_exp_{method}"] = ins_exp.mean(0).tolist()
            curves_json[f"ins_rand_{method}"] = ins_rand.mean(0).tolist()

    # ---- primary headline (joint deletion if available) + per-sample bootstrap CI ----
    primary = "joint" if "joint" in methods else methods[0]
    results["faithfulness_primary_method"] = primary
    results["faithfulness_gain"] = results[f"faithfulness_{primary}_del_gain"]
    ps_gain = _TRAPZ(npz[f"del_exp_{primary}"] - npz[f"del_rand_{primary}"], q_grid, axis=1)
    m, lo, hi = _bootstrap_ci(ps_gain)
    results["faithfulness_gain_ci_lo"] = lo
    results["faithfulness_gain_ci_hi"] = hi
    npz[f"persample_gain_{primary}"] = ps_gain

    if save:
        fdir = out_dir or os.path.join(run_dir, "xai", "faithfulness")
        os.makedirs(fdir, exist_ok=True)
        np.savez_compressed(os.path.join(fdir, f"{out_name}_curves.npz"), **npz)
        with open(os.path.join(fdir, f"{out_name}.json"), "w") as f:
            json.dump({**results, "curves": curves_json, "q_grid": q_grid.tolist()}, f, indent=2)
        for method in (methods if FAITH_SAVE_PLOTS else ()):
            plt.figure(figsize=(6, 4))
            plt.plot(q_grid, curves_json[f"del_exp_{method}"], marker="o", label="explanation")
            plt.plot(q_grid, curves_json[f"del_rand_{method}"], marker="s", label="random")
            plt.xlabel("budget q removed"); plt.ylabel("mean logit drop C(q)")
            plt.title(f"Deletion faithfulness ({method}) gain={results[f'faithfulness_{method}_del_gain']:+.3f}")
            plt.legend(); plt.grid(alpha=0.3)
            _finalize_plot(fdir, f"{out_name}_deletion_curve_{method}.png")
            if do_insertion:
                plt.figure(figsize=(6, 4))
                plt.plot(q_grid, curves_json[f"ins_exp_{method}"], marker="o", label="explanation")
                plt.plot(q_grid, curves_json[f"ins_rand_{method}"], marker="s", label="random")
                plt.xlabel("budget q inserted"); plt.ylabel("mean logit recovery")
                plt.title(f"Insertion faithfulness ({method}) gain={results[f'faithfulness_{method}_ins_gain']:+.3f}")
                plt.legend(); plt.grid(alpha=0.3)
                _finalize_plot(fdir, f"{out_name}_insertion_curve_{method}.png")

    results["faith_seconds"] = round(time.time() - t0, 2)
    if verbose:
        print(f"[FAITH] n={n} | primary={primary} | gain={results['faithfulness_gain']:+.4f} "
              f"| {results['faith_seconds']}s")
    return results


In [ ]:
# import keras, glob, traceback, os
#
# REPO = "/home/claudio/isic24-dum"
# ck = glob.glob(os.path.join(
#   REPO,
#   "artifacts/enhanced/runs/loss_regularization/20260610-095420/"
#   "seed_47/lam1e-03__e20/checkpoints/best_*.keras"))
# assert ck, f"no checkpoint found (cwd={os.getcwd()})"
# ck = ck[0]
# print("loading:", ck)
# m = keras.models.load_model(ck, compile=False, safe_mode=False)
#
# # SAME kwargs as make_xai_callbacks (cell 34, lines 97–104) — incl. lore_config + rng_seed
# cb = LoreCallback(
#   data_source=validation_ds, raw_feature_df=validation_df,
#   feat_cols=FEAT_COLS, feature_space=feature_space,
#   base_dir="/tmp/lore_probe", max_explain_samples=6,
#   positive_threshold=0.5, top_k=10, every_n_epochs=1,
#   max_samples=256, max_batches=2,
#   lore_config={'neigh_type': 'random', 'size': 300, 'random_state': 42},
#   rng_seed=0,
# )
# cb.set_model(m); cb.on_train_begin({})
# try:
#   cb.on_epoch_end(4, logs={"val_auc": 0.9})   # NOT wrapped → real error surfaces
# except Exception:
#   traceback.print_exc()

In [ ]:
# import keras, glob, shutil, gc, traceback
# from pathlib import Path
# import pandas as pd
#
# RUN_ROOT = Path("/home/claudio/isic24-dum/artifacts/enhanced/runs/"
#                 "loss_regularization/20260610-095420")
#
# run_dirs = sorted({p.parents[0] for p in RUN_ROOT.glob("seed_*/lam*__e20/checkpoints")})
# print(f"found {len(run_dirs)} run dirs")
#
# ok = []
# for rd in run_dirs:
#     rel  = rd.relative_to(RUN_ROOT)
#     cks  = glob.glob(str(rd / "checkpoints" / "best_*.keras"))
#     summ = rd / "single_run_summary.csv"
#     if not cks or not summ.exists():
#         print(f"[SKIP] {rel} (ck={bool(cks)}, summary={summ.exists()})"); continue
#
#     s = pd.read_csv(summ).iloc[0]
#     best_epoch, best_auc = int(s["best_epoch"]), float(s["best_val_auc"])
#     lore_dir = rd / "xai" / "lore"
#     if lore_dir.exists():
#         shutil.rmtree(lore_dir)            # drop stale empty skeleton -> no orphan dirs
#
#     try:
#         m = keras.models.load_model(cks[0], compile=False, safe_mode=False)
#         cb = LoreCallback(
#             data_source=validation_ds, raw_feature_df=validation_df,
#             feat_cols=FEAT_COLS, feature_space=feature_space,
#             base_dir=str(lore_dir), max_explain_samples=6,
#             positive_threshold=0.5, top_k=10, every_n_epochs=1,
#             max_samples=256, max_batches=2,
#             lore_config={'neigh_type': 'random', 'size': 300, 'random_state': 42},
#             rng_seed=0,
#         )
#         cb.set_model(m); cb.on_train_begin({})
#         cb.on_epoch_end(best_epoch, logs={"val_auc": best_auc})
#         n = len(list(lore_dir.rglob("*.txt")))
#         print(f"[OK]   {rel}  epoch{best_epoch:03d}  rule_files={n}  fidelity={getattr(cb,'last_quality',{})}")
#         ok.append(str(rel))
#     except Exception:
#         print(f"[FAIL] {rel}"); traceback.print_exc()
#     finally:
#         keras.backend.clear_session()
#         gc.collect()
#
# print(f"\nbackfilled {len(ok)}/{len(run_dirs)} runs")

In [ ]:
raise StopExecution

### Image Only (Unimodal)

In [ ]:
def build_visual(optimizer, loss, metric, run_eagerly=False):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    inp = {"images": image_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.Dense(128, activation="selu")(x1)

    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x1)

    model = keras.models.Model(inp, out)

    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

### Tabulr Only (Unimodal)

In [ ]:
def build_tabular(optimizer, loss, metric, run_eagerly=False):
    feat_input = keras.Input(
        shape=(feature_space.get_encoded_features().shape[1],), name="features"
    )
    inp = {"features": feat_input}

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x2)

    model = keras.models.Model(inp, out)

    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

### Vanilla (No explicit modality adjustment)

In [ ]:
def build_vanilla(optimizer, loss, metric, run_eagerly=False):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    concat = keras.layers.Concatenate()([x1, x2])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.models.Model(inp, out)

    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

In [ ]:
# Vanilla run (single-seed)
EPOCHS = CFG.epochs
ENABLE_XAI = True

summary_df, run_dir = run_experiment(
    impl_name="vanilla",
    build_model_fn=lambda run_eagerly: build_vanilla(adam, bce, auc, run_eagerly),
    enable_xai=ENABLE_XAI,
    epochs=EPOCHS,
    # setting_tag="baseline",   # optional
)
print(summary_df)

In [ ]:
# Vanilla run (multi-seed)
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

VANILLA_SETTINGS = [{"tag": "baseline"}]
study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "vanilla", study_tag)
os.makedirs(study_dir, exist_ok=True)

all_results = []
for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")
    res, _, _ = run_model_experiment(
        impl_name="vanilla",
        settings=VANILLA_SETTINGS,
        build_model_from_setting_fn=lambda s, run_eagerly: build_vanilla(adam, bce, auc, run_eagerly),
        enable_xai_fn=lambda s: True,
        epochs=EPOCHS,
        seed=seed,
        exp_tag=seed_exp_tag,
    )
    all_results.append(res)

combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)
display(combined[["impl_name", "run_id", "seed", "best_val_auc", "best_epoch", "train_seconds", "xai_done"]])

# AUC mean ± std
print(f"AUC: {combined['best_val_auc'].mean():.4f} ± {combined['best_val_auc'].std():.4f}")

# Per-seed val_auc curves
import json
plots_dir = os.path.join(study_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

plt.figure(figsize=(8, 5))
for _, row in combined.iterrows():
    hist_path = os.path.join(row["run_dir"], "history.json")
    if not os.path.exists(hist_path): continue
    with open(hist_path) as f:
        h = json.load(f)
    y = h.get("val_auc", [])
    plt.plot(np.arange(1, len(y)+1), y, marker="o", label=f"seed={row['seed']}")
plt.xlabel("epoch"); plt.ylabel("val_auc")
plt.title("Vanilla: val_auc vs epoch (per seed)")
plt.minorticks_on()
plt.grid(which="major", alpha=0.3); plt.grid(which="minor", alpha=0.15, linestyle=":")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "val_auc_vs_epoch_per_seed.png"), dpi=300)
plt.show(); plt.close()

### Fixed modality scaling — simple approach
After obtaining embeddings **x1** (images) and **x2** (tabular), we scale them with **β = 1 − α** and **α**, respectively.

- If **α = 0** → model uses only images  
- If **α = 0.5** → balanced usage  
- If **α = 1** → model uses only tabular


In [ ]:
# Fixed scaling ---------------------------------------------------------------
@keras.utils.register_keras_serializable(package="modality_attendance")
class ConstantScale(keras.layers.Layer):
    def __init__(self, scale, **kwargs):
        super().__init__(**kwargs)
        self.scale = float(scale)

    def call(self, x):
        return x * keras.ops.cast(self.scale, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"scale": self.scale})
        return cfg


def build_fixed_scaling(alpha, optimizer, loss, metric, run_eagerly=False):
    alpha = float(alpha)
    beta = 1.0 - alpha

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)
    x1_weighted = ConstantScale(beta, name="image_weighting")(x1)
    x2_weighted = ConstantScale(alpha, name="tabular_weighting")(x2)

    concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.Model(inp, out, name="fixed_alpha")
    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=False,
    )
    return model

In [ ]:
# fixed scaling run (single-seed)
ALPHAS = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]
XAI_ALPHAS = ALPHAS
EPOCHS = CFG.epochs
SEED = CFG.seed

FIXED_SETTINGS = [{"tag": f"a{a:.3f}".replace(".", "p"), "alpha": float(a)} for a in ALPHAS]

fixed_results, fixed_histories, fixed_summary_dir = run_model_experiment(
    impl_name="fixed_scaling",
    settings=FIXED_SETTINGS,
    build_model_from_setting_fn=lambda s, run_eagerly: build_fixed_scaling(
        s["alpha"], adam, bce, auc, run_eagerly
    ),
    enable_xai_fn=lambda s: round(float(s["alpha"]), 3) in XAI_ALPHAS,
    epochs=EPOCHS,
    extra_meta_fn=lambda s: {"alpha": float(s["alpha"]), "beta": float(1.0 - s["alpha"])},
    seed=SEED,
)

display(
    fixed_results[
        ["impl_name", "run_id", "alpha", "best_val_auc", "best_epoch", "train_seconds", "seed", "xai_done"]
    ]
)

plots_dir = os.path.join(fixed_summary_dir, "plots")
plot_param_vs_best_auc(fixed_results, "alpha", out_dir=plots_dir)
plot_param_vs_best_epoch(fixed_results, "alpha", out_dir=plots_dir)
plot_param_vs_train_time(fixed_results, "alpha", out_dir=plots_dir)
plot_val_auc_vs_epoch_sweep(
    fixed_results, fixed_histories, setting_col="alpha",
    title="Fixed scaling: val_auc vs epoch by alpha",
    out_dir=plots_dir, fname="val_auc_vs_epoch_alpha_sweep.png",
)

plot_lore_xai_suite(fixed_results, "alpha", out_dir=plots_dir)

print("[plots]:", plots_dir)

In [ ]:
# fixed scaling run (multi-seed)
ALPHAS = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]
XAI_ALPHAS = ALPHAS
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

FIXED_SETTINGS = [{"tag": f"a{a:.3f}".replace(".", "p"), "alpha": float(a)} for a in ALPHAS]

# Single macro folder for entire multi-seed study
study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "fixed_scaling", study_tag)
os.makedirs(study_dir, exist_ok=True)

all_results = []
all_histories = {}

for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")   # nested folder under study
    fixed_results, fixed_histories, fixed_summary_dir = run_model_experiment(
        impl_name="fixed_scaling",
        settings=FIXED_SETTINGS,
        build_model_from_setting_fn=lambda s, run_eagerly: build_fixed_scaling(
            s["alpha"], adam, bce, auc, run_eagerly
        ),
        enable_xai_fn=lambda s: round(float(s["alpha"]), 3) in XAI_ALPHAS,
        epochs=EPOCHS,
        extra_meta_fn=lambda s: {"alpha": float(s["alpha"]), "beta": float(1.0 - s["alpha"])},
        seed=seed,
        exp_tag=seed_exp_tag,
    )
    all_results.append(fixed_results)
    all_histories.update({f"{seed}__{k}": v for k, v in fixed_histories.items()})

    # per-seed plots stay inside the seed folder
    seed_plots_dir = os.path.join(fixed_summary_dir, "plots")
    plot_param_vs_best_auc(fixed_results, "alpha", out_dir=seed_plots_dir)
    plot_lore_xai_suite(fixed_results, "alpha", out_dir=seed_plots_dir)
    plot_val_auc_vs_epoch_sweep(
        fixed_results, fixed_histories, setting_col="alpha",
        out_dir=seed_plots_dir, fname="val_auc_vs_epoch_alpha_sweep.png",
    )

# Cross-seed aggregation
combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)

agg = combined.groupby("alpha")["best_val_auc"].agg(["mean", "std", "count"]).reset_index()
agg.to_csv(os.path.join(study_dir, "alpha_aggregated.csv"), index=False)
display(agg)

# Plot mean ± std across seeds
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(agg["alpha"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
ax.set_xlabel("alpha"); ax.set_ylabel("best_val_auc (mean ± std)")
ax.set_title(f"Fixed scaling: AUC vs alpha across {len(SEEDS)} seeds")
ax.grid(alpha=0.3)
plt.tight_layout()
study_plots = os.path.join(study_dir, "plots")
os.makedirs(study_plots, exist_ok=True)
plt.savefig(os.path.join(study_plots, "alpha_vs_auc_mean_std.png"), dpi=300)
plt.show()
plt.close()

print("[study dir]:", study_dir)

### Learnable modality scaling — symmetric (ablation)

We learn a single trainable logit `w` and compute:

- `alpha = sigmoid(w)` with `alpha ∈ [0,1]`
- `beta = 1 - alpha` (symmetry constraint)

`init_alpha` sets the starting balance (typically `0.5`).  

During training, backprop updates `w`, so the model can shift attention between modalities while keeping total modality weight fixed.

- `alpha → 0`: image-dominant
- `alpha ≈ 0.5`: balanced
- `alpha → 1`: tabular-dominant

For each run, report both `init_alpha` and learned `alpha_final` (from the best checkpoint).

In [ ]:
# Learnable modality scaling (init-alpha sweep)

@keras.utils.register_keras_serializable(package="modality_attendance")
class LearnableAlpha(keras.layers.Layer):
    def __init__(self, init_alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        init_alpha = float(np.clip(init_alpha, 1e-4, 1.0 - 1e-4))
        self.init_alpha = init_alpha
        init_logit = np.log(init_alpha / (1.0 - init_alpha))
        self.logit = self.add_weight(
            name="alpha_logit",
            shape=(),
            initializer=keras.initializers.Constant(init_logit),
            trainable=True,
        )

    def call(self, x):
        alpha = tf.sigmoid(self.logit)
        beta = 1.0 - alpha
        return alpha, beta

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"init_alpha": self.init_alpha})
        return cfg


@keras.utils.register_keras_serializable(package="modality_attendance")
class ScalarMultiply(keras.layers.Layer):
    def call(self, inputs):
        x, s = inputs
        s = keras.ops.cast(s, x.dtype)
        return x * s

    def compute_output_shape(self, input_shape):
        return input_shape[0]


def build_learnable_scaling(setting, run_eagerly=False):
    init_alpha = float(setting.get("init_alpha", 0.5))

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    alpha, beta = LearnableAlpha(init_alpha=init_alpha, name="learnable_alpha")(x1)
    x1_weighted = ScalarMultiply(name="image_weighting")([x1, beta])
    x2_weighted = ScalarMultiply(name="tabular_weighting")([x2, alpha])

    concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.Model(inp, out, name="learnable_scaling")

    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


def read_alpha_from_best_ckpt(row):
    ckpt_path = os.path.join(row["run_dir"], "checkpoints", f"best_{row['model_name']}.keras")
    if not os.path.exists(ckpt_path):
        return np.nan
    m = keras.models.load_model(ckpt_path, compile=False)
    a_layer = m.get_layer("learnable_alpha")
    return float(tf.sigmoid(a_layer.logit).numpy())

In [ ]:
# Learnable modality scaling — multi-seed
INIT_ALPHA = [0.1, 0.3, 0.5, 0.7, 0.9]
XAI_INIT_ALPHA = {0.5}
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

LEARNABLE_SETTINGS = [
    {"tag": f"init_a{a:.3f}".replace(".", "p"), "init_alpha": float(a)}
    for a in INIT_ALPHA
]

# Single macro folder for entire multi-seed study
study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "learnable_scaling", study_tag)
os.makedirs(study_dir, exist_ok=True)

all_results = []
all_histories = {}

for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")
    learnable_results, learnable_histories, learnable_summary_dir = run_model_experiment(
        impl_name="learnable_scaling",
        settings=LEARNABLE_SETTINGS,
        build_model_from_setting_fn=build_learnable_scaling,
        enable_xai_fn=lambda s: round(float(s["init_alpha"]), 3) in XAI_INIT_ALPHA,
        epochs=EPOCHS,
        extra_meta_fn=lambda s: {"init_alpha": float(s["init_alpha"])},
        seed=seed,
        exp_tag=seed_exp_tag,
        extra_callbacks_fn=lambda run_dir: [
            AlphaLogger(
                csv_path=os.path.join(run_dir, "alpha_trajectory.csv"),
                layer_name="learnable_alpha",
            )
        ],
    )

    # learnable-specific: alpha finale da best ckpt
    learnable_results["alpha_final"] = learnable_results.apply(read_alpha_from_best_ckpt, axis=1)
    learnable_results["beta_final"] = 1.0 - learnable_results["alpha_final"]

    all_results.append(learnable_results)
    all_histories.update({f"{seed}__{k}": v for k, v in learnable_histories.items()})

    # per-seed plots dentro la cartella del seed
    seed_plots_dir = os.path.join(learnable_summary_dir, "plots")
    plot_param_vs_best_auc(learnable_results, "init_alpha", out_dir=seed_plots_dir)
    plot_lore_xai_suite(learnable_results, "init_alpha", out_dir=seed_plots_dir)
    plot_val_auc_vs_epoch_sweep(
        learnable_results, learnable_histories, setting_col="init_alpha",
        out_dir=seed_plots_dir, fname="val_auc_vs_epoch_init_alpha_sweep.png",
    )

    # trajectory α per-seed
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, row in learnable_results.sort_values("init_alpha").iterrows():
        traj = pd.read_csv(os.path.join(row["run_dir"], "alpha_trajectory.csv"))
        ax.plot(traj["epoch"], traj["alpha"], marker="o", label=f"init={row['init_alpha']:.2f}")
    ax.axhline(0.5, color="grey", ls="--", alpha=0.5)
    ax.set_xlabel("epoch"); ax.set_ylabel("α"); ax.set_ylim(0, 1)
    ax.set_title(f"Learnable α trajectory (seed {seed})")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(seed_plots_dir, "alpha_trajectory_by_init.png"), dpi=300)
    plt.close()

# Cross-seed aggregation
combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)

agg_auc = combined.groupby("init_alpha")["best_val_auc"].agg(["mean", "std", "count"]).reset_index()
agg_alpha = combined.groupby("init_alpha")["alpha_final"].agg(["mean", "std"]).reset_index()
agg_auc.to_csv(os.path.join(study_dir, "init_alpha_aggregated.csv"), index=False)
agg_alpha.to_csv(os.path.join(study_dir, "alpha_final_aggregated.csv"), index=False)
display(agg_auc)
display(agg_alpha)

study_plots = os.path.join(study_dir, "plots")
os.makedirs(study_plots, exist_ok=True)

# AUC mean ± std
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(agg_auc["init_alpha"], agg_auc["mean"], yerr=agg_auc["std"], marker="o", capsize=4)
ax.set_xlabel("init_alpha"); ax.set_ylabel("best_val_auc (mean ± std)")
ax.set_title(f"Learnable scaling: AUC vs init_alpha across {len(SEEDS)} seeds")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(study_plots, "init_alpha_vs_auc_mean_std.png"), dpi=300)
plt.show(); plt.close()

# alpha_final mean ± std (convergenza: dove atterra α partendo da init diversi)
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(agg_alpha["init_alpha"], agg_alpha["mean"], yerr=agg_alpha["std"], marker="o", capsize=4)
ax.plot([INIT_ALPHA[0], INIT_ALPHA[-1]], [INIT_ALPHA[0], INIT_ALPHA[-1]], "--", lw=1, label="y=x")
ax.set_xlabel("init_alpha"); ax.set_ylabel("alpha_final (mean ± std)")
ax.set_title(f"Learnable scaling: init sensitivity across {len(SEEDS)} seeds")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(study_plots, "init_alpha_vs_alpha_final_mean_std.png"), dpi=300)
plt.show(); plt.close()

print("[study dir]:", study_dir)

**### Loss regularization -- Refined approach

Here we do not modify the architecture, but add a penalty term to the loss that reduces the dependence on the tabular (or image) branch.

We measure the "dependence" through the L2 norm of the weights connecting the modality to the final head.

We apply modality-aware L2 regularization on final-layer modality-specific weights.

- **Per-sample mode**:  
  `L_i = BCE_i + λ_reg * R`
- **Per-batch mode**:  
  `L = mean(BCE_i) + λ_reg * R`

Both use the same regularizer `R`, but per-batch gives a single scalar objective per step and is usually more stable for logging/comparison.**

The narrative is "constrain the easier modality to force the model to

    use vision". Image-penalty isn't motivated; "both" is redundant with L2

    weight decay. Document the decision.

In [ ]:
# per sample reg
def weighted_loss_ps(y_true, y_pred):
    # BCE base
    base_loss = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)

    # Penalità sui pesi che connettono la parte tabellare all’output
    W = dense_final.kernel  # matrice [dim_img + dim_tab, 1]
    W_tab = W[dim_img:]     # seleziona solo la parte tabellare
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))

    return base_loss + reg_term

# per batch reg
def weighted_loss_pb(y_true, y_pred):
    base_loss = keras.losses.binary_crossentropy(
        y_true, y_pred, label_smoothing=0.02
    )
    base_loss = tf.reduce_mean(base_loss)  # per-batch mean
    W = dense_final.kernel
    W_tab = W[dim_img:]
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))
    return base_loss + reg_term


def build_loss_regularized(
    lambda_reg=1e-4,
    penalty_on="tabular",          # "tabular" | "image" | "both"
    reg_mode="per_batch",          # "per_sample" | "per_batch"
    lambda_sym=0,                  # symmetry strength
    sym_target_tab=0.5,            # 0.5 => equal image/tab influence
    run_eagerly=False,
):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.LayerNormalization(name="img_branch_norm")(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)
    x2 = keras.layers.LayerNormalization(name="tab_branch_norm")(x2)

    concat = keras.layers.Concatenate(name="concat")([x1, x2])
    out_layer = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="output")
    out = out_layer(concat)
    model = keras.Model(inp, out, name="loss_regularization")

    dim_img = int(x1.shape[-1])
    dim_tab = int(x2.shape[-1])

    def _modality_norms():
        W = out_layer.kernel  # [dim_img + dim_tab, 1]
        W_img = W[:dim_img]
        W_tab = W[dim_img:dim_img + dim_tab]
        # mean square (not sum) avoids bias from different branch dims
        n_img = tf.reduce_mean(tf.square(W_img))
        n_tab = tf.reduce_mean(tf.square(W_tab))
        return n_img, n_tab

    def _reg_term():
        n_img, n_tab = _modality_norms()

        if penalty_on == "tabular":
            mag_term = lambda_reg * n_tab
        elif penalty_on == "image":
            mag_term = lambda_reg * n_img
        elif penalty_on == "both":
            mag_term = lambda_reg * (n_img + n_tab)
        else:
            raise ValueError(f"Unsupported penalty_on={penalty_on}")

        # soft symmetry: push tab share toward target (0.5 = symmetric)
        tab_share = n_tab / (n_img + n_tab + 1e-8)
        sym_term = lambda_sym * tf.square(tab_share - sym_target_tab)

        return mag_term + sym_term

    def weighted_loss_ps(y_true, y_pred):
        base = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)
        return base + _reg_term()

    def weighted_loss_pb(y_true, y_pred):
        base = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)
        base = tf.reduce_mean(base)
        return base + _reg_term()

    loss_fn = weighted_loss_ps if reg_mode == "per_sample" else weighted_loss_pb

    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=loss_fn,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


In [ ]:
# Loss regularization — multi-seed sweep over lambda_reg
REG_MODE = "per_batch"
PENALTY_ON = "tabular"
LAMBDA_REGS = [0, 1e-5, 1e-4, 1e-3, 1e-2]
XAI_LAMBDAS = LAMBDA_REGS
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

REG_SETTINGS = [
    {"tag": f"lam{l:.0e}", "lambda_reg": float(l)}
    for l in LAMBDA_REGS
]

study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "loss_regularization", study_tag)
os.makedirs(study_dir, exist_ok=True)

all_results = []
all_histories = {}

for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")
    reg_results, reg_histories, reg_summary_dir = run_model_experiment(
        impl_name="loss_regularization",
        settings=REG_SETTINGS,
        build_model_from_setting_fn=lambda s, run_eagerly: build_loss_regularized(
            lambda_reg=s["lambda_reg"],
            penalty_on=PENALTY_ON,
            reg_mode=REG_MODE,
            run_eagerly=run_eagerly,
        ),
        enable_xai_fn=lambda s: float(s["lambda_reg"]) in XAI_LAMBDAS,
        epochs=EPOCHS,
        extra_meta_fn=lambda s: {
            "reg_mode": REG_MODE,
            "lambda_reg": float(s["lambda_reg"]),
            "penalty_on": PENALTY_ON,
        },
        seed=seed,
        exp_tag=seed_exp_tag,
    )
    all_results.append(reg_results)
    all_histories.update({f"{seed}__{k}": v for k, v in reg_histories.items()})

    # per-seed plots inside seed folder
    seed_plots_dir = os.path.join(reg_summary_dir, "plots")
    plot_param_vs_best_auc(reg_results, "lambda_reg", out_dir=seed_plots_dir)
    plot_lore_xai_suite(reg_results, "lambda_reg", out_dir=seed_plots_dir)
    plot_val_auc_vs_epoch_sweep(
        reg_results, reg_histories, setting_col="lambda_reg",
        out_dir=seed_plots_dir, fname="val_auc_vs_epoch_lambda_reg_sweep.png",
    )

# Cross-seed aggregation
combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)

agg = combined.groupby("lambda_reg")["best_val_auc"].agg(["mean", "std", "count"]).reset_index()
agg.to_csv(os.path.join(study_dir, "lambda_reg_aggregated.csv"), index=False)
display(agg)

# Plot mean ± std across seeds (log-x because lambda spans orders of magnitude)
fig, ax = plt.subplots(figsize=(7, 4))
# Replace lambda_reg=0 with a small floor for log scale display only
x_plot = agg["lambda_reg"].replace(0, 1e-7)
ax.errorbar(x_plot, agg["mean"], yerr=agg["std"], marker="o", capsize=4)
ax.set_xscale("log")
ax.set_xlabel("lambda_reg (log; 0 plotted at 1e-7)")
ax.set_ylabel("best_val_auc (mean ± std)")
ax.set_title(f"Loss regularization ({PENALTY_ON}, {REG_MODE}): "
             f"AUC vs lambda across {len(SEEDS)} seeds")
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
study_plots = os.path.join(study_dir, "plots")
os.makedirs(study_plots, exist_ok=True)
plt.savefig(os.path.join(study_plots, "lambda_reg_vs_auc_mean_std.png"), dpi=300)
plt.show()
plt.close()

print("[study dir]:", study_dir)

### Controlled fusion (post‑training weighting)  
This variant replaces the standard merge with a weighted sum of the two modality embeddings:  
`fused = gamma_img * z_img + gamma_tab * z_tab`.  
It lets you manually bias the model toward image or tabular features *after training* by changing `gamma_img` and `gamma_tab` (both non‑trainable).  
Note: `z_img` and `z_tab` must have the same dimensionality, since they are summed.

The narrative is "constrain the easier modality to force the model to use vision. Image-penalty isn't motivated; "both" is redundant with L2 weight decay. Document the decision.

In [ ]:
# Controlled fusion (post-training weighting)

@keras.utils.register_keras_serializable(package="modality_attendance")
class FusionWithGamma(keras.layers.Layer):
    def __init__(self, gamma_img=1.0, gamma_tab=1.0, **kwargs):
        super().__init__(**kwargs)
        self.initial_gamma_img = float(gamma_img)
        self.initial_gamma_tab = float(gamma_tab)
        self.gamma_img = self.add_weight(
            name="gamma_img", shape=(),
            initializer=keras.initializers.Constant(self.initial_gamma_img),
            trainable=False,
        )
        self.gamma_tab = self.add_weight(
            name="gamma_tab", shape=(),
            initializer=keras.initializers.Constant(self.initial_gamma_tab),
            trainable=False,
        )

    def call(self, inputs):
        z_img, z_tab = inputs
        return self.gamma_img * z_img + self.gamma_tab * z_tab

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "gamma_img": self.initial_gamma_img,
            "gamma_tab": self.initial_gamma_tab,
        })
        return cfg


def build_gamma_fusion(setting, run_eagerly=False):
    gamma_img = float(setting.get("gamma_img", 1.0))
    gamma_tab = float(setting.get("gamma_tab", 1.0))

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.Dense(128, activation="linear", name="img_proj")(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    fused = FusionWithGamma(gamma_img=gamma_img, gamma_tab=gamma_tab, name="fusion_with_gamma")([x1, x2])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(fused)

    model = keras.Model(inp, out, name="controlled_fusion")
    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

In [ ]:
# Controlled fusion — multi-seed
CF_TAG = "base"
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

CONTROLLED_FUSION_SETTINGS = [
    {"tag": CF_TAG, "gamma_img": 1.0, "gamma_tab": 1.0}
]

# Single macro folder for entire multi-seed study
study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "controlled_fusion", study_tag)
os.makedirs(study_dir, exist_ok=True)

cf_all_results = []
cf_all_histories = {}

for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")
    cf_seed_results, cf_seed_histories, cf_summary_dir = run_model_experiment(
        impl_name="controlled_fusion",
        settings=CONTROLLED_FUSION_SETTINGS,
        build_model_from_setting_fn=build_gamma_fusion,
        enable_xai_fn=lambda s: True,
        epochs=EPOCHS,
        extra_meta_fn=lambda s: {
            "gamma_img": float(s["gamma_img"]),
            "gamma_tab": float(s["gamma_tab"]),
        },
        seed=seed,
        exp_tag=seed_exp_tag,
    )
    cf_all_results.append(cf_seed_results)
    cf_all_histories.update({f"{seed}__{k}": v for k, v in cf_seed_histories.items()})

    # per-seed plots stay inside the seed folder
    seed_plots_dir = os.path.join(cf_summary_dir, "plots")
    plot_lore_xai_suite(cf_seed_results, "gamma_img", out_dir=seed_plots_dir)
    plot_val_auc_vs_epoch_sweep(
        cf_seed_results, cf_seed_histories, setting_col="gamma_img",
        out_dir=seed_plots_dir, fname="val_auc_vs_epoch_controlled_fusion.png",
    )

# Cross-seed aggregation
cf_combined = pd.concat(cf_all_results, ignore_index=True)
cf_combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)

cf_agg = (
    cf_combined.groupby(["setting_tag", "gamma_img", "gamma_tab"], dropna=False)["best_val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
cf_agg.to_csv(os.path.join(study_dir, "controlled_fusion_aggregated.csv"), index=False)
display(cf_combined[["impl_name", "run_id", "setting_tag", "seed", "best_val_auc", "best_epoch", "train_seconds", "xai_done"]])
display(cf_agg)
print(f"AUC: {cf_combined['best_val_auc'].mean():.4f} ± {cf_combined['best_val_auc'].std():.4f}")
print("[study dir]:", study_dir)

In [ ]:
# Per-run training curves (loss & val_auc), saved to each run's plots/ folder
for _, row in cf_combined.iterrows():
    cf_run_dir = row["run_dir"]
    with open(os.path.join(cf_run_dir, "history.json")) as f:
        cf_hist = json.load(f)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    if "val_auc" in cf_hist and len(cf_hist["val_auc"]) > 0:
        x = np.arange(1, len(cf_hist["val_auc"]) + 1)
        plt.figure(figsize=(7, 4))
        plt.plot(x, cf_hist["val_auc"], marker="o")
        plt.xlabel("epoch"); plt.ylabel("val_auc")
        plt.title("Controlled fusion: val_auc vs epoch")
        plt.grid(alpha=0.3)
        _finalize_plot(plots_dir, "val_auc_vs_epoch.png")

    if "loss" in cf_hist and "val_loss" in cf_hist and len(cf_hist["loss"]) > 0:
        x = np.arange(1, len(cf_hist["loss"]) + 1)
        plt.figure(figsize=(7, 4))
        plt.plot(x, cf_hist["loss"], marker="o", label="loss")
        plt.plot(x, cf_hist["val_loss"], marker="o", label="val_loss")
        plt.xlabel("epoch"); plt.ylabel("loss")
        plt.title("Controlled fusion: loss vs epoch")
        plt.grid(alpha=0.3)
        plt.legend()
        _finalize_plot(plots_dir, "loss_vs_epoch.png")

 After training, you can manually adjust them to bias the final prediction toward one modality without retraining

In [ ]:
# Post-hoc gamma sweep over every controlled-fusion run
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())


# gamma_tab > 1 enhances tabular / penalizes image (gamma_img < 1)
# gamma_tab < 1 penalizes tabular / enhances image (gamma_img > 1)
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

posthoc_rows = []
for _, row in cf_combined.iterrows():
    cf_run_dir = row["run_dir"]
    ckpt_path = os.path.join(cf_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
    fusion_layer = best_model.get_layer("fusion_with_gamma")

    run_rows = []
    for s in POSTHOC_SETTINGS:
        fusion_layer.gamma_img.assign(s["gamma_img"])
        fusion_layer.gamma_tab.assign(s["gamma_tab"])
        auc_val = eval_auc(best_model, validation_ds)

        rec = {
            "run_id": row["run_id"],
            "setting_tag": row["setting_tag"],
            "seed": int(row["seed"]),
            "run_dir": cf_run_dir,
            "posthoc_id": s["tag"],
            "delta": s["delta"],
            "gamma_img": s["gamma_img"],
            "gamma_tab": s["gamma_tab"],
            "mode": s["mode"],
            "val_auc": auc_val,
            "base_model_name": row["model_name"],
        }
        run_rows.append(rec)
        posthoc_rows.append(rec)

    posthoc_df = pd.DataFrame(run_rows).sort_values("gamma_tab").reset_index(drop=True)
    diagnostics_dir = os.path.join(cf_run_dir, "diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    posthoc_df.to_csv(os.path.join(diagnostics_dir, "posthoc_gamma_summary.csv"), index=False)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("gamma_tab")
    plt.ylabel("val_auc")
    plt.title(f"Controlled fusion post-hoc: val_auc vs gamma_tab (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
    for _, r in posthoc_df.iterrows():
        plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
    plt.xlabel("gamma_img")
    plt.ylabel("gamma_tab")
    plt.title(f"Controlled fusion post-hoc sweep (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
    plt.close()

    keras.backend.clear_session()
    gc.collect()

posthoc_all_df = pd.DataFrame(posthoc_rows).sort_values(["seed", "gamma_tab"]).reset_index(drop=True)
posthoc_all_df.to_csv(os.path.join(study_dir, "controlled_fusion_posthoc_gamma_summary.csv"), index=False)

posthoc_agg = (
    posthoc_all_df.groupby(["delta", "gamma_img", "gamma_tab"], dropna=False)["val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
posthoc_agg.to_csv(os.path.join(study_dir, "controlled_fusion_posthoc_gamma_aggregated.csv"), index=False)
display(posthoc_all_df.head(20))
display(posthoc_agg)

### Adaptive gating

In [ ]:
# # Per evitare che il gate diventi troppo deterministico (cioè sempre 0 o 1),
# # possiamo aggiungere un termine di entropia alla loss

# def gated_loss(y_true, y_pred):
#     base_loss = keras.losses.binary_crossentropy(y_true, y_pred,
#                                                  label_smoothing=0.02)
#     gate_out = model.get_layer("gate_weights").output
#     entropy = -tf.reduce_mean(tf.reduce_sum(gate_out * tf.math.log(gate_out + 1e-6), axis=1))
#     return base_loss + 1e-4 * entropy

In [ ]:
import keras.ops as K

@keras.utils.register_keras_serializable(package="modality_attendance")
class GateEntropyLoss(keras.layers.Layer):
    def __init__(self, weight=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.weight = float(weight)

    def call(self, gate_out):
        entropy = -K.mean(K.sum(gate_out * K.log(gate_out + 1e-6), axis=1))
        self.add_loss(-self.weight * entropy)  # maximize entropy
        return gate_out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"weight": self.weight})
        return cfg


def build_adaptive_gating(setting, run_eagerly=False):
    entropy_weight = float(setting.get("entropy_weight", 1e-4))

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.Dense(256, activation="selu", name="img_emb")(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)
    x2 = keras.layers.Dense(256, activation="selu", name="tab_emb")(x2)

    fusion_input = keras.layers.Concatenate(name="fusion_input")([x1, x2])
    gate = keras.layers.Dense(64, activation="relu")(fusion_input)
    gate = keras.layers.Dense(2, activation="softmax", name="gate_weights")(gate)
    gate = GateEntropyLoss(weight=entropy_weight, name="gate_entropy_loss")(gate)

    w_img = keras.layers.Lambda(lambda w: w[:, 0:1], name="w_img")(gate)
    w_tab = keras.layers.Lambda(lambda w: w[:, 1:2], name="w_tab")(gate)

    x1_w = keras.layers.Multiply(name="weighted_img")([x1, w_img])
    x2_w = keras.layers.Multiply(name="weighted_tab")([x2, w_tab])
    fused = keras.layers.Add(name="fused")([x1_w, x2_w])

    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="output")(fused)

    model = keras.Model(inp, out, name="adaptive_gating")
    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


In [ ]:
# Adaptive gating — multi-seed
AG_SETTING = {"tag": "base", "entropy_weight": 1e-4}
EPOCHS = CFG.epochs
SEEDS = [47, 53, 71, 8, 90]

ADAPTIVE_GATING_SETTINGS = [AG_SETTING]

# Single macro folder for entire multi-seed study
study_tag = _make_exp_tag()
study_dir = os.path.join(RUNS_PATH, "adaptive_gating", study_tag)
os.makedirs(study_dir, exist_ok=True)

ag_all_results = []
ag_all_histories = {}

for seed in SEEDS:
    seed_exp_tag = os.path.join(study_tag, f"seed_{seed}")
    ag_seed_results, ag_seed_histories, ag_summary_dir = run_model_experiment(
        impl_name="adaptive_gating",
        settings=ADAPTIVE_GATING_SETTINGS,
        build_model_from_setting_fn=build_adaptive_gating,
        enable_xai_fn=lambda s: True,
        epochs=EPOCHS,
        extra_meta_fn=lambda s: {"entropy_weight": float(s["entropy_weight"])},
        seed=seed,
        exp_tag=seed_exp_tag,
    )
    ag_all_results.append(ag_seed_results)
    ag_all_histories.update({f"{seed}__{k}": v for k, v in ag_seed_histories.items()})

    # per-seed plots stay inside the seed folder
    seed_plots_dir = os.path.join(ag_summary_dir, "plots")
    plot_lore_xai_suite(ag_seed_results, "entropy_weight", out_dir=seed_plots_dir)
    plot_val_auc_vs_epoch_sweep(
        ag_seed_results, ag_seed_histories, setting_col="entropy_weight",
        out_dir=seed_plots_dir, fname="val_auc_vs_epoch_adaptive_gating.png",
    )

# Cross-seed aggregation
ag_combined = pd.concat(ag_all_results, ignore_index=True)
ag_combined.to_csv(os.path.join(study_dir, "combined_summary.csv"), index=False)

ag_agg = ag_combined.groupby("entropy_weight")["best_val_auc"].agg(["mean", "std", "count"]).reset_index()
ag_agg.to_csv(os.path.join(study_dir, "entropy_weight_aggregated.csv"), index=False)
display(ag_combined[["impl_name", "run_id", "setting_tag", "seed", "best_val_auc", "best_epoch", "train_seconds", "xai_done"]])
display(ag_agg)
print(f"AUC: {ag_combined['best_val_auc'].mean():.4f} ± {ag_combined['best_val_auc'].std():.4f}")
print("[study dir]:", study_dir)


In [ ]:
for _, row in ag_combined.iterrows():
    ag_run_dir = row["run_dir"]
    with open(os.path.join(ag_run_dir, "history.json")) as f:
        ag_hist = json.load(f)

    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    if "val_auc" in ag_hist and len(ag_hist["val_auc"]) > 0:
        x = np.arange(1, len(ag_hist["val_auc"]) + 1)
        plt.figure(figsize=(7, 4))
        plt.plot(x, ag_hist["val_auc"], marker="o")
        plt.xlabel("epoch"); plt.ylabel("val_auc")
        plt.title("Adaptive gating: val_auc vs epoch")
        plt.grid(alpha=0.3)
        _finalize_plot(plots_dir, "val_auc_vs_epoch.png")

    if "loss" in ag_hist and "val_loss" in ag_hist and len(ag_hist["loss"]) > 0:
        x = np.arange(1, len(ag_hist["loss"]) + 1)
        plt.figure(figsize=(7, 4))
        plt.plot(x, ag_hist["loss"], marker="o", label="loss")
        plt.plot(x, ag_hist["val_loss"], marker="o", label="val_loss")
        plt.xlabel("epoch"); plt.ylabel("loss")
        plt.title("Adaptive gating: loss vs epoch")
        plt.grid(alpha=0.3)
        plt.legend()
        _finalize_plot(plots_dir, "loss_vs_epoch.png")

#### Sanity cheks

##### Modality Ablation

Evaluates the model's reliance on each modality (image and tabular data).

It quantifies the performance drop (AUC) when a specific modality is either fully removed or partially perturbed, helping to understand its contribution to the overall prediction.

##### **Gate Faithfulness**

Assesses how well the dynamically predicted gate weights align with the actual impact of each modality on the model's output.

A higher **Spearman correlation** between a modality's gate weight and its ablation impact suggests that the gate is faithfully representing the modality's importance.

##### **Calibration (Brier + ECE)**

Evaluates the model's calibration i.e. how well its predicted probabilities match the true probabilities.

**Brier Score** measures the mean squared difference between predicted probabilities and actual outcomes, indicating overall accuracy and calibration.

**Expected Calibration Error (ECE)** quantifies how far predicted probabilities are from true correctness across different confidence bins, revealing systematic over- or under-confidence.

In [ ]:
# Generic AUC evaluator on a dataset
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Build ablated inputs to remove/perturb one modality
def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)  # gray image
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

# Evaluate AUC after modality ablation
def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(ablate_batch(x, mode, strength), training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Simple Spearman (rank correlation) implementation
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean()
    rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))

# Convert probabilities to logits for impact comparison
def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1 - eps)
    return tf.math.log(p / (1 - p))

# Gate-faithfulness: do gate weights correlate with true modality impact?
def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []
    for x, _ in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # Impact after image ablation
        pi = tf.squeeze(model(ablate_batch(x, "img", strength), training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # Impact after tabular ablation
        pt = tf.squeeze(model(ablate_batch(x, "tab", strength), training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        # Predicted gate weights
        w = gate_model(x, training=False)
        wimg.append(w[:, 0].numpy())
        wtab.append(w[:, 1].numpy())
        dimg.append(di.numpy())
        dtab.append(dt.numpy())

    wimg = np.concatenate(wimg)
    wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg)
    dtab = np.concatenate(dtab)

    return {
        "spearman_w_img_delta_img": spearman(wimg, dimg),
        "spearman_w_tab_delta_tab": spearman(wtab, dtab),
    }

# Calibration metrics: Brier score + ECE
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)
    brier = float(np.mean((p - y) ** 2))

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.any(m):
            conf = float(np.mean(p[m]))
            acc = float(np.mean(y[m]))
            ece += (np.sum(m) / N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

In [ ]:
study_dir   = os.path.join(RUNS_PATH, "adaptive_gating","20260629-185216")
ag_combined = pd.read_csv(os.path.join(study_dir,"combined_summary.csv"))

In [ ]:
# post-training diagnostics

all_diag = []
for _, row in ag_combined.iterrows():
    ag_run_dir = row["run_dir"]
    # combined_summary.csv stores June paths (/home/claudio/...); remap the
    # "/runs/..." suffix onto this machine's RUNS_PATH so checkpoints load.
    if not os.path.isdir(ag_run_dir):
        _s = str(ag_run_dir).replace("\\", "/")
        _i = _s.find("/runs/")
        if _i >= 0:
            ag_run_dir = os.path.join(os.path.dirname(RUNS_PATH), _s[_i + 1:])
    best_ckpt = os.path.join(ag_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    ag_model = keras.models.load_model(best_ckpt, compile=False, safe_mode=False)

    gate_model = keras.Model(ag_model.inputs, ag_model.get_layer("gate_weights").output)
    gx, _ = next(iter(validation_ds))
    gw = gate_model(gx, training=False).numpy()

    base_auc = eval_auc(ag_model, validation_ds)
    img_auc = eval_auc_with_ablation(ag_model, validation_ds, "img", 1.0)
    tab_auc = eval_auc_with_ablation(ag_model, validation_ds, "tab", 1.0)

    diag = {
        "run_id": row["run_id"],
        "setting_tag": row["setting_tag"],
        "seed": int(row["seed"]),
        "entropy_weight": float(row["entropy_weight"]),
        "run_dir": ag_run_dir,
        "gate_weight_mean_img": float(np.mean(gw[:, 0])),
        "gate_weight_mean_tab": float(np.mean(gw[:, 1])),
        "base_auc": base_auc,
        "img_drop_auc": base_auc - img_auc,
        "tab_drop_auc": base_auc - tab_auc,
        **gate_faithfulness(ag_model, validation_ds, strength=1.0),
        **brier_ece(ag_model, validation_ds),
    }

    os.makedirs(os.path.join(ag_run_dir, "diagnostics"), exist_ok=True)
    with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
        json.dump(diag, f, indent=2)
    all_diag.append(diag)

    # ---- diagnostic plots (per run), saved to the run's plots/ folder ----
    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # gather full-validation gate weights, labels and predictions in one pass
    wimg_all, wtab_all, y_true_all, p_pred_all = [], [], [], []
    for xb, yb in validation_ds:
        wb = gate_model(xb, training=False).numpy()
        pb = tf.squeeze(ag_model(xb, training=False), -1).numpy()
        wimg_all.append(wb[:, 0]); wtab_all.append(wb[:, 1])
        y_true_all.append(np.asarray(tf.squeeze(yb)).reshape(-1))
        p_pred_all.append(np.asarray(pb).reshape(-1))
    wimg_all = np.concatenate(wimg_all); wtab_all = np.concatenate(wtab_all)
    y_true_all = np.concatenate(y_true_all); p_pred_all = np.concatenate(p_pred_all)

    # 1) gate-weight distribution
    plt.figure(figsize=(7, 4))
    plt.hist(wimg_all, bins=30, alpha=0.6, label="w_img")
    plt.hist(wtab_all, bins=30, alpha=0.6, label="w_tab")
    plt.xlabel("gate weight"); plt.ylabel("count")
    plt.title(f"Adaptive gating: gate-weight distribution (seed {int(row['seed'])})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "gate_weight_distribution.png")

    # 2) modality ablation: base AUC and AUC drop after ablating each modality
    plt.figure(figsize=(7, 4))
    labels = ["base_auc", "img_drop_auc", "tab_drop_auc"]
    vals = [diag["base_auc"], diag["img_drop_auc"], diag["tab_drop_auc"]]
    plt.bar(labels, vals, color=["#4c72b0", "#dd8452", "#55a868"])
    for j, v in enumerate(vals):
        plt.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("AUC"); plt.grid(alpha=0.3, axis="y")
    plt.title(f"Adaptive gating: modality ablation (seed {int(row['seed'])})")
    _finalize_plot(plots_dir, "modality_ablation_auc_drop.png")

    # 3) calibration reliability diagram
    n_bins = 15
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_conf, bin_obs = [], []
    for j in range(n_bins):
        lo, hi = edges[j], edges[j + 1]
        m = (p_pred_all >= lo) & (p_pred_all < hi) if j < n_bins - 1 else (p_pred_all >= lo) & (p_pred_all <= hi)
        if np.any(m):
            bin_conf.append(float(np.mean(p_pred_all[m])))
            bin_obs.append(float(np.mean(y_true_all[m])))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    plt.plot(bin_conf, bin_obs, marker="o", label="model")
    plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
    plt.title(f"Calibration (Brier={diag['brier']:.3f}, ECE={diag['ece']:.3f})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "calibration_reliability.png")

    keras.backend.clear_session()
    gc.collect()

ag_diag_df = pd.DataFrame(all_diag)
ag_diag_df.to_csv(os.path.join(study_dir, "adaptive_gating_diagnostics.csv"), index=False)
display(ag_diag_df)

In [ ]:
# ---------------------------------------------------------------------
# Register custom serializable layers for post-hoc model loading.
# These are the SAME classes defined in the training-runner cells (which sit
# below the StopExecution guard and are skipped on a top-to-bottom run), so
# keras.models.load_model() in the faithfulness loop can deserialize every
# strategy's checkpoint. No training side effects — safe to run anytime before
# the faithfulness cells.
# ---------------------------------------------------------------------
import numpy as np
import tensorflow as tf
import keras
import keras.ops as K

@keras.utils.register_keras_serializable(package="modality_attendance")
class ConstantScale(keras.layers.Layer):
    def __init__(self, scale, **kwargs):
        super().__init__(**kwargs)
        self.scale = float(scale)
    def call(self, x):
        return x * keras.ops.cast(self.scale, x.dtype)
    def get_config(self):
        cfg = super().get_config(); cfg.update({"scale": self.scale}); return cfg

@keras.utils.register_keras_serializable(package="modality_attendance")
class LearnableAlpha(keras.layers.Layer):
    def __init__(self, init_alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        init_alpha = float(np.clip(init_alpha, 1e-4, 1.0 - 1e-4))
        self.init_alpha = init_alpha
        init_logit = np.log(init_alpha / (1.0 - init_alpha))
        self.logit = self.add_weight(name="alpha_logit", shape=(),
            initializer=keras.initializers.Constant(init_logit), trainable=True)
    def call(self, x):
        alpha = tf.sigmoid(self.logit); beta = 1.0 - alpha
        return alpha, beta
    def get_config(self):
        cfg = super().get_config(); cfg.update({"init_alpha": self.init_alpha}); return cfg

@keras.utils.register_keras_serializable(package="modality_attendance")
class ScalarMultiply(keras.layers.Layer):
    def call(self, inputs):
        x, s = inputs; s = keras.ops.cast(s, x.dtype)
        return x * s
    def compute_output_shape(self, input_shape):
        return input_shape[0]

@keras.utils.register_keras_serializable(package="modality_attendance")
class FusionWithGamma(keras.layers.Layer):
    def __init__(self, gamma_img=1.0, gamma_tab=1.0, **kwargs):
        super().__init__(**kwargs)
        self.initial_gamma_img = float(gamma_img)
        self.initial_gamma_tab = float(gamma_tab)
        self.gamma_img = self.add_weight(name="gamma_img", shape=(),
            initializer=keras.initializers.Constant(self.initial_gamma_img), trainable=False)
        self.gamma_tab = self.add_weight(name="gamma_tab", shape=(),
            initializer=keras.initializers.Constant(self.initial_gamma_tab), trainable=False)
    def call(self, inputs):
        z_img, z_tab = inputs
        return self.gamma_img * z_img + self.gamma_tab * z_tab
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma_img": self.initial_gamma_img, "gamma_tab": self.initial_gamma_tab})
        return cfg

@keras.utils.register_keras_serializable(package="modality_attendance")
class GateEntropyLoss(keras.layers.Layer):
    def __init__(self, weight=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.weight = float(weight)
    def call(self, gate_out):
        entropy = -K.mean(K.sum(gate_out * K.log(gate_out + 1e-6), axis=1))
        self.add_loss(-self.weight * entropy)  # maximize entropy
        return gate_out
    def get_config(self):
        cfg = super().get_config(); cfg.update({"weight": self.weight}); return cfg

print("[XAI] registered custom layers:",
      [c.__name__ for c in (ConstantScale, LearnableAlpha, ScalarMultiply,
                            FusionWithGamma, GateEntropyLoss)])


In [ ]:
# ---------------------------------------------------------------------
# Post-hoc analysis setup: canonical study dirs + summaries loaded in memory.
# Lets the gate-diagnostics and gamma-sweep cells run WITHOUT retraining.
# ---------------------------------------------------------------------
CANONICAL = {
    "vanilla":             os.path.join(RUNS_PATH, "vanilla", "20260604-210304"),
    "fixed_scaling":       os.path.join(RUNS_PATH, "fixed_scaling", "20260622-112038"),
    "learnable_scaling":   os.path.join(RUNS_PATH, "learnable_scaling", "20260619-184157"),
    "loss_regularization": os.path.join(RUNS_PATH, "loss_regularization", "20260610-095420"),
    "controlled_fusion":   os.path.join(RUNS_PATH, "controlled_fusion", "20260616-102253"),
    "adaptive_gating":     os.path.join(RUNS_PATH, "adaptive_gating", "20260629-185216"),
}
cf_combined = pd.read_csv(os.path.join(CANONICAL["controlled_fusion"], "combined_summary.csv"))
ag_combined = pd.read_csv(os.path.join(CANONICAL["adaptive_gating"], "combined_summary.csv"))
print("loaded cf_combined", cf_combined.shape, "| ag_combined", ag_combined.shape)


## Explanation Faithfulness (Deletion / Insertion AUC)

`faithfulness_gain = deletion_auc_explanation - deletion_auc_random` (piu alto = meglio), calcolata per `tabular` (SHAP), `image` (Grad-CAM) e `joint`. Riusa gli array SHAP gia salvati (nessun re-fit del GradientExplainer). Le tre celle seguenti definiscono il core, lo eseguono post-hoc su tutti i run gia allenati e aggregano i risultati.

In [ ]:
# ---------------------------------------------------------------------
# Post-hoc faithfulness over ALL trained runs (no retraining).
# Discovers run dirs from single_run_summary.csv, loads each best_*.keras
# checkpoint, computes deletion/insertion faithfulness and saves per-run
# artifacts + a combined CSV.
# ---------------------------------------------------------------------
FAITH_RUN_GLOB = os.path.join(RUNS_PATH, "**", "single_run_summary.csv")  # any depth: 3-level single + 4-level multi-seed
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
os.makedirs(FAITH_OUT_DIR, exist_ok=True)

# Restrict which strategies to process (None = all)
FAITH_ONLY_IMPL = None       # e.g. ["vanilla", "adaptive_gating"]
FAITH_OVERWRITE = False       # reuse inline-computed faithfulness.json; only compute what is missing
FAITH_MAX_RUNS = None         # set e.g. 5 for a smoke test before running all discovered checkpoints
FAITH_DRY_RUN = False          # True: only report what would be processed, without loading models

_SETTING_PARAM_COLS = ("alpha", "init_alpha", "lambda_reg",
                       "gamma_img", "gamma_tab", "entropy_weight")

summ_paths = sorted(glob.glob(FAITH_RUN_GLOB, recursive=True))

selected_runs = []
n_reuse = 0
n_compute = 0
for sp in summ_paths:
    try:
        row = pd.read_csv(sp).iloc[0].to_dict()
    except Exception as e:
        print("  skip (bad csv):", sp, e)
        continue

    impl = row.get("impl_name")
    if FAITH_ONLY_IMPL and impl not in FAITH_ONLY_IMPL:
        continue

    run_dir = row.get("run_dir")
    if not isinstance(run_dir, str) or not os.path.isdir(run_dir):
        run_dir = os.path.dirname(sp)
    model_name = row.get("model_name")
    best_epoch = int(row.get("best_epoch", -1))
    ckpt = os.path.join(run_dir, "checkpoints", f"best_{model_name}.keras")
    if not os.path.exists(ckpt) or best_epoch < 0:
        print("  skip (no ckpt / best_epoch):", run_dir)
        continue

    fjson = os.path.join(run_dir, "xai", "faithfulness", "faithfulness.json")
    will_reuse = os.path.exists(fjson) and not FAITH_OVERWRITE
    n_reuse += int(will_reuse)
    n_compute += int(not will_reuse)
    selected_runs.append((sp, row, run_dir, model_name, best_epoch, ckpt, fjson))

if FAITH_MAX_RUNS is not None:
    selected_runs = selected_runs[:int(FAITH_MAX_RUNS)]
    n_reuse = sum(os.path.exists(r[6]) and not FAITH_OVERWRITE for r in selected_runs)
    n_compute = len(selected_runs) - n_reuse

print(f"[FAITH] discovered {len(summ_paths)} run summaries | selected={len(selected_runs)} "
      f"| reuse={n_reuse} | compute={n_compute}")
if FAITH_DRY_RUN:
    display(pd.DataFrame([{"impl_name": r[1].get("impl_name"),
                           "setting_tag": r[1].get("setting_tag"),
                           "seed": r[1].get("seed"),
                           "run_dir": r[2],
                           "action": "reuse" if os.path.exists(r[6]) and not FAITH_OVERWRITE else "compute"}
                          for r in selected_runs]).head(50))
    raise SystemExit("FAITH_DRY_RUN=True: no models loaded.")

faith_rows = []
for sp, row, run_dir, model_name, best_epoch, ckpt, fjson in selected_runs:
    impl = row.get("impl_name")
    if os.path.exists(fjson) and not FAITH_OVERWRITE:
        res = json.load(open(fjson))
        res = {k: v for k, v in res.items() if k not in ("curves", "q_grid")}
    else:
        t0 = time.time()
        try:
            keras.backend.clear_session()
            model = keras.models.load_model(ckpt, compile=False, safe_mode=False)
            res = explanation_faithfulness(model, run_dir, best_epoch, save=True, verbose=False)
        except Exception as e:
            print(f"  [WARN] {impl}/{row.get('setting_tag')}: {type(e).__name__}: {e}")
            keras.backend.clear_session(); gc.collect()
            continue
        keras.backend.clear_session(); gc.collect()
        res["faith_seconds"] = round(time.time() - t0, 2)

    rec = {
        "impl_name": impl,
        "setting_tag": row.get("setting_tag"),
        "seed": row.get("seed"),
        "best_val_auc": row.get("best_val_auc"),
        "run_dir": run_dir,
        **{c: row[c] for c in _SETTING_PARAM_COLS if c in row},
        **{k: v for k, v in res.items() if not isinstance(v, (list, dict))},
    }
    faith_rows.append(rec)
    print(f"  ok {impl:>18}/{str(row.get('setting_tag')):<14} "
          f"gain={res.get('faithfulness_gain', float('nan')):+.4f} "
          f"({res.get('faith_seconds', '?')}s)")

faith_df = pd.DataFrame(faith_rows)
out_csv = os.path.join(FAITH_OUT_DIR, "combined_faithfulness.csv")
# MERGE (not overwrite): keep prior rows for runs not recomputed in this pass,
# replace rows for run_dirs processed now. Safe with FAITH_ONLY_IMPL filtering.
if os.path.exists(out_csv) and len(faith_df):
    _prev = pd.read_csv(out_csv)
    _prev = _prev[~_prev["run_dir"].isin(faith_df["run_dir"])]
    faith_df = pd.concat([_prev, faith_df], ignore_index=True)
elif os.path.exists(out_csv) and not len(faith_df):
    faith_df = pd.read_csv(out_csv)
faith_df.to_csv(out_csv, index=False)
print(f"[FAITH] saved {out_csv}  shape={faith_df.shape}")
display(faith_df.head(20))


In [ ]:
# ---------------------------------------------------------------------
# Cross-strategy aggregation: mean/std over seeds, bootstrap CI,
# comparison plots. Run after the post-hoc loop above.
# ---------------------------------------------------------------------
faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
fdf = pd.read_csv(faith_csv)

# --- keep ONLY canonical runs: drop seed_49 smoke runs and the superseded
#     duplicate controlled_fusion run so nothing is double-counted here.
#     (CANONICAL is defined in the "canonical study dirs" cell above.)
_canon_ts = {impl: os.path.basename(p) for impl, p in CANONICAL.items()}
fdf["_ts"] = fdf["run_dir"].astype(str).str.extract(r"/(\d{8}-\d{6})/")[0]
fdf = fdf[fdf.apply(lambda r: _canon_ts.get(r["impl_name"]) == r["_ts"], axis=1)].copy()
fdf = fdf.drop(columns=["_ts"]).reset_index(drop=True)
print(f"[FAITH] aggregating over {len(fdf)} canonical runs: "
      f"{fdf['impl_name'].value_counts().to_dict()}")
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
FAITH_SHOW_AGG_PLOTS = False  # save aggregate PNGs without flooding notebook output

gain_cols = [c for c in fdf.columns
             if c.startswith("faithfulness_") and c.endswith("_gain")]
val_cols = [c for c in (gain_cols + ["faithfulness_gain", "best_val_auc"]) if c in fdf.columns]

# ---- mean / std / n per (impl_name, setting_tag) over seeds ----
agg = (fdf.groupby(["impl_name", "setting_tag"], dropna=False)[val_cols]
          .agg(["mean", "std", "count"]))
agg.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_summary.csv"))
print("[FAITH] per-setting summary:")
display(agg)

# ---- bootstrap CI of primary faithfulness_gain per strategy (resample runs) ----
ci_rows = []
for impl, sub in fdf.groupby("impl_name"):
    m, lo, hi = _bootstrap_ci(sub["faithfulness_gain"].values)
    ci_rows.append({"impl_name": impl, "mean_gain": m, "ci_lo": lo, "ci_hi": hi, "n_runs": len(sub)})
ci_df = pd.DataFrame(ci_rows).sort_values("mean_gain", ascending=False).reset_index(drop=True)
ci_df.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_gain_ci.csv"), index=False)
print("[FAITH] strategy ranking (primary deletion gain, bootstrap 95% CI):")
display(ci_df)

# ---- bar chart: faithfulness_gain per strategy with CI ----
plt.figure(figsize=(9, 5))
x = np.arange(len(ci_df))
yerr = [ci_df["mean_gain"] - ci_df["ci_lo"], ci_df["ci_hi"] - ci_df["mean_gain"]]
plt.bar(x, ci_df["mean_gain"], yerr=yerr, capsize=4, color="#4c72b0", edgecolor="black")
plt.axhline(0.0, color="gray", lw=1)
plt.xticks(x, ci_df["impl_name"], rotation=30, ha="right")
plt.ylabel("faithfulness_gain (primary deletion)")
plt.title("Explanation faithfulness gain per strategy (bootstrap 95% CI)")
_finalize_plot(FAITH_OUT_DIR, "faithfulness_gain_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)

# ---- deletion-curve overlay per strategy (mean across runs) ----
def _overlay_deletion(method):
    plt.figure(figsize=(9, 5))
    plotted = False
    for impl, sub in fdf.groupby("impl_name"):
        cs, qg = [], None
        for _, r in sub.iterrows():
            npz = os.path.join(str(r["run_dir"]), "xai", "faithfulness", "faithfulness_curves.npz")
            if not os.path.exists(npz):
                continue
            d = np.load(npz)
            key = f"del_exp_{method}"
            if key not in d.files:
                continue
            cs.append(d[key].mean(axis=0)); qg = d["q_grid"]
        if cs:
            plt.plot(qg, np.mean(cs, axis=0), marker="o", label=impl); plotted = True
    if plotted:
        plt.xlabel("budget q removed"); plt.ylabel("mean logit drop C_exp(q)")
        plt.title(f"Explanation-guided deletion curves ({method}) per strategy")
        plt.legend(); plt.grid(alpha=0.3)
        _finalize_plot(FAITH_OUT_DIR, f"deletion_curves_{method}_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)
    else:
        plt.close()

for _m in ("joint", "tabular", "image"):
    _overlay_deletion(_m)

print("[FAITH] aggregation done ->", FAITH_OUT_DIR)


In [ ]:
# ---------------------------------------------------------------------
# MASTER RESULTS TABLE -> artifacts/enhanced/summaries/
# One row per run + a (family,setting) mean/std/count summary. Paper Table 1
# source + AUC-vs-faithfulness trade-off scatter source.
# Requires: canonical loader cell + regenerated combined_faithfulness.csv.
# ---------------------------------------------------------------------
os.makedirs(SUMMARIES_PATH, exist_ok=True)
_frames = []
for _impl, _d in CANONICAL.items():
    _p = os.path.join(_d, "combined_summary.csv")
    if os.path.exists(_p):
        _df = pd.read_csv(_p); _df["impl_name"] = _impl; _frames.append(_df)
master = pd.concat(_frames, ignore_index=True)
_faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
if os.path.exists(_faith_csv):
    _fdf = pd.read_csv(_faith_csv)
    _keep = [c for c in ["run_dir","faithfulness_gain","faithfulness_tabular_del_gain",
             "faithfulness_image_del_gain"] if c in _fdf.columns]
    _fdf = _fdf[_keep].copy()
    # run_dir absolute prefixes differ across machines (June: /home/claudio/...,
    # now: /mnt/disk1/home/cgiovannoni/...). combined_summary.csv keeps the OLD
    # prefix, combined_faithfulness.csv the NEW one, so a raw run_dir join is
    # all-NaN. Join on the machine-independent suffix starting at "/runs/".
    def _relkey(p):
        p = str(p).replace("\\", "/")
        i = p.find("/runs/")
        return p[i:] if i >= 0 else p
    master["_key"] = master["run_dir"].map(_relkey)
    _fdf["_key"] = _fdf["run_dir"].map(_relkey)
    _fdf = _fdf.drop(columns=["run_dir"]).drop_duplicates("_key")
    _dupe = [c for c in _fdf.columns if c != "_key" and c in master.columns]
    master = master.drop(columns=_dupe).merge(_fdf, on="_key", how="left").drop(columns=["_key"])

# LORE fidelity: combined_summary.csv only carries it for families whose runner
# logged it inline. Fill vanilla + controlled_fusion from the post-hoc CSV and
# loss_regularization from each run's per-epoch CSV (last epoch, context mode --
# same semantics as the inline lore_fidelity).
import glob as _g
def _lkey(p):
    p = str(p).replace("\\", "/")
    i = p.find("/runs/")
    return p[i:] if i >= 0 else p
_lore_map = {}
_lp = os.path.join(RUNS_PATH, "_faithfulness", "lore_fidelity_posthoc.csv")
if os.path.exists(_lp):
    for _, _r in pd.read_csv(_lp).iterrows():
        _lore_map[_lkey(_r["run_dir"])] = float(_r["lore_fidelity"])
for _pe in _g.glob(os.path.join(CANONICAL["loss_regularization"], "**", "lore_fidelity_per_epoch.csv"), recursive=True):
    _e = pd.read_csv(_pe)
    if len(_e):
        _lore_map[_lkey(os.path.dirname(os.path.dirname(os.path.dirname(_pe))))] = float(_e.iloc[-1]["lore_fidelity_context"])
if _lore_map and "lore_fidelity" in master.columns:
    master["lore_fidelity"] = master["lore_fidelity"].fillna(master["run_dir"].map(_lkey).map(_lore_map))
_metrics = [c for c in ["best_val_auc","brier","ece","modality_balance","lore_fidelity",
            "faithfulness_gain","faithfulness_tabular_del_gain","faithfulness_image_del_gain"]
            if c in master.columns]
_group = [c for c in ["impl_name","setting_tag"] if c in master.columns]
summary = master.groupby(_group, dropna=False)[_metrics].agg(["mean","std","count"])
master.to_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"), index=False)
summary.to_csv(os.path.join(SUMMARIES_PATH, "master_results_summary.csv"))
print("[SUMMARY] wrote master_results_rows.csv + master_results_summary.csv ->", SUMMARIES_PATH)
display(summary)


In [ ]:
# %load notebooks/modality_attendance/feature_space_guard.py
# ============================================================================
# FeatureSpace anti-drift GUARD  (post-hoc safety, added 2026-07-15)
#
# The whole class of bug it closes: post-hoc (LORE / SHAP / deletion / Stage 3)
# silently runs a FeatureSpace that is NOT the one the checkpoints were trained
# with — a re-adapt (non-deterministic, cell 16) or a dim change (127 -> 132 on
# the bias-fixed pipeline). Same code, silently wrong numbers.
#
# Two cheap checks, both fail LOUD instead of producing wrong numbers:
#   assert_feature_space_canonical(feature_space)   -> run ONCE after cell 16.
#       Recomputes the live FS fingerprint and compares it to the recorded
#       canonical (feature_space.MANIFEST.json). Catches "the global FS drifted".
#   assert_fs_matches_model(model, feature_space)   -> after EVERY load_model
#       in a post-hoc loop. Compares the checkpoint's 'features' input width to
#       the live FS encoded dim. Catches "this checkpoint != this FS".
#
# NB: this is the "guard only" scope (Claudio, 2026-07-15). It does NOT bind a
# FS to a study-dir or snapshot per run — that is the later consolidation
# (roadmap pt 1). The guard just makes any mismatch impossible to miss.
# ============================================================================
import os, json, hashlib
import numpy as np


def feature_space_fingerprint(fs):
    """Save-invariant fingerprint of a FeatureSpace's ADAPT STATE.
    Complete: feature order + encoded dim + categorical vocabularies + all adapt weights.
    MUST stay byte-for-byte identical to the spec that produced feature_space.MANIFEST.json."""
    h = hashlib.sha256()
    order = list(fs.features.keys())
    dim = int(fs.get_encoded_features().shape[-1])
    h.update(("ORDER:" + "|".join(order)).encode())
    h.update(f"DIM:{dim}".encode())
    for name in order:                                  # categorical vocabularies (order-sensitive)
        pp = fs.preprocessors.get(name)
        if pp is not None and hasattr(pp, "get_vocabulary"):
            try:
                v = pp.get_vocabulary()
                h.update(f"VOCAB:{name}:".encode()); h.update("|".join(map(str, v)).encode())
            except Exception:
                pass
    for w in fs.get_weights():                          # normalization stats etc.
        w = np.asarray(w)
        h.update(str(w.shape).encode()); h.update(str(w.dtype).encode()); h.update(w.tobytes())
    return dim, h.hexdigest()


def _find_manifest(manifest_path=None):
    if manifest_path and os.path.exists(manifest_path):
        return manifest_path
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
    for cand in (
        "notebooks/modality_attendance/feature_space.MANIFEST.json",
        os.path.join(here, "feature_space.MANIFEST.json"),
        "feature_space.MANIFEST.json",
    ):
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(
        "feature_space.MANIFEST.json not found. It records the canonical fingerprint; "
        "regenerate it from the canonical FeatureSpace before trusting post-hoc numbers.")


def assert_feature_space_canonical(fs, manifest_path=None):
    """Run ONCE, right after the FeatureSpace is loaded (cell 16/17).
    Fails loud if the live FS is not the canonical one the checkpoints were trained with."""
    mp = _find_manifest(manifest_path)
    man = json.load(open(mp))
    dim, fp = feature_space_fingerprint(fs)
    if dim != man["encoded_dim"]:
        raise RuntimeError(
            f"[FS GUARD] encoded dim {dim} != canonical {man['encoded_dim']} ({mp}). "
            f"The FeatureSpace was re-adapted with a different pipeline (127 vs 132?). "
            f"Post-hoc against the existing checkpoints would be WRONG. Restore the canonical: "
            f"cp {man['canonical_file']} artifacts/enhanced/feature_space/feature_space.keras")
    if fp != man["fingerprint_sha256"]:
        raise RuntimeError(
            f"[FS GUARD] FeatureSpace fingerprint\n    {fp}\n  != canonical\n    {man['fingerprint_sha256']}\n"
            f"  (dim matches, so this is a silent re-adapt: same shape, different stats/vocab).\n"
            f"  Restore the canonical before any post-hoc: "
            f"cp {man['canonical_file']} artifacts/enhanced/feature_space/feature_space.keras")
    print(f"[FS GUARD] OK — live FeatureSpace matches canonical (dim {dim}, fp {fp[:16]}…, {os.path.basename(mp)})")
    return dim, fp


def _model_feature_dim(model):
    """Read the tabular 'features' input width off a loaded model — no hardcoded 127/128."""
    # prefer the input literally named 'features'
    for t in getattr(model, "inputs", []):
        nm = getattr(t, "name", "") or ""
        if "features" in nm and len(t.shape) == 2:
            return int(t.shape[-1])
    # fallback: the single rank-2 (batch, dim) input is the tabular one (image input is rank-4)
    rank2 = [t for t in getattr(model, "inputs", []) if len(t.shape) == 2]
    if len(rank2) == 1:
        return int(rank2[0].shape[-1])
    raise RuntimeError("[FS GUARD] could not locate the tabular 'features' input on the model.")


def assert_fs_matches_model(model, fs, run_id=None):
    """Call right after keras.models.load_model(...) in any post-hoc loop.
    Fails loud if this checkpoint's feature-input width != the live FS encoded dim."""
    mdim = _model_feature_dim(model)
    fdim = int(fs.get_encoded_features().shape[-1])
    if mdim != fdim:
        raise RuntimeError(
            f"[FS GUARD] {run_id or '<run>'}: checkpoint expects features dim {mdim} but the live "
            f"FeatureSpace emits {fdim}. This checkpoint was NOT trained with this FeatureSpace — "
            f"post-hoc numbers would be silently wrong.")
    return mdim


# When %load-ed and executed in the notebook, run Guard A immediately on the
# FeatureSpace already in scope (named `feature_space`), so a single run wires it in.
if "feature_space" in dir():
    assert_feature_space_canonical(feature_space)  # noqa: F821
else:
    print("[FS GUARD] functions defined. `feature_space` not in scope yet — call "
          "assert_feature_space_canonical(feature_space) after cell 16, and "
          "assert_fs_matches_model(model, feature_space) after each load_model.")


In [ ]:

import numpy as np, pandas as pd, os, json, gc

def relmap(rd):
  if os.path.isdir(rd): return rd
  s = str(rd).replace("\\","/"); i = s.find("/runs/")
  return os.path.join(os.path.dirname(RUNS_PATH), s[i+1:]) if i>=0 else rd

# congela un batch di validation (immagini+label) + raw_df allineato (val non è shufflata)
N_BG = 128; imgs, labs, n = [], [], 0
for x, y in validation_ds:
  take = min(N_BG-n, x["images"].shape[0])
  imgs.append(x["images"].numpy()[:take]); labs.append(y.numpy()[:take]); n += take
  if n >= N_BG: break
images = np.concatenate(imgs); labels = np.concatenate(labs)
raw_df = validation_df.iloc[:n][FEAT_COLS].reset_index(drop=True)

engine = LoreEngine(feature_space=feature_space, feat_cols=FEAT_COLS,
                  max_explain_samples=12, positive_threshold=0.5,
                  lore_config={"neigh_type":"random","size":300,"random_state":42,"verbose":False})

TARGETS = {"vanilla":"20260604-210304", "controlled_fusion":"20260616-102253"}
rows = []
for impl, ts in TARGETS.items():
  comb = pd.read_csv(os.path.join(RUNS_PATH, impl, ts, "combined_summary.csv"))
  for _, r in comb.iterrows():
      rd = relmap(r["run_dir"]); seed = int(r["seed"])
      ck = os.path.join(rd, "checkpoints", f"best_{r['model_name']}.keras")
      model = keras.models.load_model(ck, compile=False, safe_mode=False)
      np.random.seed(42); sc, _, _ = engine.explain(raw_df=raw_df, images=images, labels=labels, model=model,
  mode="context")
      np.random.seed(42); st, _, _ = engine.explain(raw_df=raw_df, images=images, labels=labels, model=model,
  mode="tabular_only")
      rec = {"impl_name":impl, "seed":seed, "run_dir":rd,
             "lore_fidelity_context":sc.get("fidelity"), "lore_rule_len_context":sc.get("rule_len"),
             "lore_fidelity_tabular_only":st.get("fidelity"), "lore_rule_len_tabular_only":st.get("rule_len"),
             "lore_fidelity":sc.get("fidelity"), "lore_rule_len":sc.get("rule_len")}
      rows.append(rec)
      dj = os.path.join(rd, "diagnostics.json")
      d = json.load(open(dj)) if os.path.exists(dj) else {}
      d.update({k:rec[k] for k in list(rec)[3:]})
      json.dump(d, open(dj,"w"), indent=2)
      print(f"{impl} seed {seed}: lore_fidelity={rec['lore_fidelity']:.4f}")
      keras.backend.clear_session(); gc.collect()

out = os.path.join(RUNS_PATH, "_faithfulness", "lore_fidelity_posthoc.csv")
os.makedirs(os.path.dirname(out), exist_ok=True)
pd.DataFrame(rows).to_csv(out, index=False)
print("scritto:", out, "-", len(rows), "righe")


In [ ]:
# ============================ STAGE 0 — zero GPU ============================
import numpy as np, pandas as pd, os
from scipy import stats

_rng = np.random.default_rng(0)
_d = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
assert len(_d) == 110, f"expected the 110 canonical runs, got {len(_d)}"

REWEIGHT = ["fixed_scaling", "learnable_scaling", "loss_regularization"]   # the attendance KNOBS
FUSION   = ["controlled_fusion", "adaptive_gating"]                        # the fusion ARCHITECTURES
_d["mech"] = np.where(_d.impl_name.isin(FUSION), "fusion",
              np.where(_d.impl_name.isin(REWEIGHT), "reweight", "baseline"))
_d["img_reliance"] = 1.0 - _d["modality_balance"]

TAB, IMG, JNT = ("faithfulness_tabular_del_gain", "faithfulness_image_del_gain", "faithfulness_gain")
_out = []

# ---- 1. H1: does boosting a modality make ITS OWN explanation more faithful? ----
print("=" * 78); print("H1 TEST"); print("=" * 78)
for label, sub in [("all runs", _d),
                   ("reweighting only (the explicit knobs)", _d[_d.mech == "reweight"]),
                   ("reweighting, excl. degenerate alpha 0/1", _d[(_d.mech == "reweight") & (~_d.alpha.isin([0.0, 1.0]))])]:
    print(f"--- {label} (n={len(sub)})")
    for yname, y in [("image_gain", sub[IMG]), ("tabular_gain", sub[TAB])]:
        r, pr = stats.pearsonr(sub.img_reliance, y)
        print(f"    img_reliance <-> {yname:<13s} r={r:+.3f} (p={pr:.2g})")
        _out.append(dict(analysis="H1", subset=label, n=len(sub), y=yname, r=r, p=pr))
print("\nIf H1 held, img_reliance <-> image_gain would be POSITIVE and strong.\n")

# ---- 2. mechanism contrast: fusion architecture vs attendance reweighting ----
print("=" * 78); print("MECHANISM CONTRAST — fusion vs reweighting"); print("=" * 78)
_fus, _rest = _d[_d.mech == "fusion"], _d[_d.mech != "fusion"]
for comp, col in [("image", IMG), ("joint", JNT), ("tabular", TAB)]:
    a, b = _fus[col].values, _rest[col].values
    bs = np.array([_rng.choice(a, len(a)).mean() - _rng.choice(b, len(b)).mean() for _ in range(10000)])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    mwu = stats.mannwhitneyu(a, b).pvalue
    print(f"  {comp:<8s} fusion={a.mean():+.3f}  rest={b.mean():+.3f}  diff={a.mean()-b.mean():+.3f}"
          f"  CI[{lo:+.3f},{hi:+.3f}]  MWU p={mwu:.2g}")
    _out.append(dict(analysis="mech_contrast", component=comp, diff=a.mean()-b.mean(),
                     ci_lo=lo, ci_hi=hi, mwu_p=mwu))

# ---- 3. paired-by-seed (free power: all families share seeds 47/53/71/8/90) ----
print("\n" + "=" * 78); print("PAIRED BY SEED"); print("=" * 78)
_van = _d[_d.impl_name == "vanilla"].set_index("seed")
for fam in FUSION:
    f = _d[_d.impl_name == fam].set_index("seed")
    seeds = sorted(set(f.index) & set(_van.index))
    for comp, col in [("image", IMG), ("joint", JNT), ("tabular", TAB)]:
        diffs = f.loc[seeds, col].values - _van.loc[seeds, col].values
        tp = stats.ttest_rel(f.loc[seeds, col], _van.loc[seeds, col]).pvalue
        up = stats.ttest_ind(f.loc[seeds, col], _van.loc[seeds, col], equal_var=False).pvalue
        print(f"  {fam:<18s} {comp:<8s} diff={diffs.mean():+.3f}  paired p={tp:.3g}  (unpaired {up:.3g})")
        _out.append(dict(analysis="paired", family=fam, component=comp,
                         mean_diff=diffs.mean(), paired_p=tp, unpaired_p=up))
print("  NOTE: at n=5 Wilcoxon signed-rank CANNOT go below p=0.0625, whatever the effect size.")

# ---- 4. variance decomposition + hierarchical bootstrap (per-sample curves from disk) ----
def _curves(run_dir):
    return os.path.join(ART_PATH, run_dir.split("artifacts/enhanced/")[-1],
                        "xai", "faithfulness", "faithfulness_curves.npz")

_ps, _Q = {}, None
_rows = []
for _, r in _d.iterrows():
    z = np.load(_curves(r.run_dir)); _Q = z["q_grid"]
    rec = dict(run_id=r.run_id, impl_name=r.impl_name, setting_tag=r.setting_tag, mech=r.mech)
    for comp in ("tabular", "image", "joint"):
        g = (np.trapezoid(z[f"del_exp_{comp}"], _Q, axis=1)
             - np.trapezoid(z[f"del_rand_{comp}"], _Q, axis=1))   # per-sample gain
        _ps[(r.run_id, comp)] = g
        rec[f"{comp}_mean"] = g.mean()
        rec[f"{comp}_sem"] = g.std(ddof=1) / np.sqrt(len(g))
    _rows.append(rec)
_psdf = pd.DataFrame(_rows)

# reproduction anchor: per-sample mean MUST equal the published gain
_m = _d.merge(_psdf, on="run_id")
assert len(_m) == 110
for comp, col in [("tabular", TAB), ("image", IMG), ("joint", JNT)]:
    err = (_m[f"{comp}_mean"] - _m[col]).abs().max()
    assert err < 1e-9, f"{comp}: per-sample curves do not reproduce the published gain ({err:.2e})"
print("\nreproduction anchor OK — per-sample curves reproduce the published gains exactly")

print("\n" + "=" * 78); print("VARIANCE: seeds vs samples"); print("=" * 78)
print(f"  {'family':<20s} {'comp':<8s} {'sd(seed)':>10s} {'sem(128)':>10s}  dominant")
for fam, gf in _psdf.groupby("impl_name"):
    for comp in ("tabular", "image", "joint"):
        btw = gf.groupby("setting_tag")[f"{comp}_mean"].std(ddof=1).mean()  # across seeds, within setting
        wit = gf[f"{comp}_sem"].mean()
        print(f"  {fam:<20s} {comp:<8s} {btw:10.4f} {wit:10.4f}  "
              f"{'SEED' if btw > wit else 'SAMPLES'} ({btw/wit:.1f}x)")
        _out.append(dict(analysis="variance", family=fam, component=comp,
                         sd_seed=btw, sem_samples=wit, ratio=btw/wit))
print("  -> if SEED dominates, the 128-sample batch is NOT the binding constraint; the 5 seeds are.")

print("\n" + "=" * 78); print("HIERARCHICAL BOOTSTRAP (resample seeds, then samples)"); print("=" * 78)
for comp in ("image", "joint", "tabular"):
    fk = [r.run_id for _, r in _d.iterrows() if r.mech == "fusion"]
    rk = [r.run_id for _, r in _d.iterrows() if r.mech != "fusion"]
    def _draw(keys):
        ks = [keys[i] for i in _rng.integers(0, len(keys), len(keys))]
        return np.mean([_ps[(k, comp)][_rng.integers(0, 128, 128)].mean() for k in ks])
    bs = np.array([_draw(fk) - _draw(rk) for _ in range(2000)])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    pt = np.mean([_ps[(k, comp)].mean() for k in fk]) - np.mean([_ps[(k, comp)].mean() for k in rk])
    print(f"  {comp:<8s} diff={pt:+.3f}  CI[{lo:+.3f},{hi:+.3f}]  "
          f"{'EXCLUDES 0' if lo > 0 or hi < 0 else 'includes 0'}")
    _out.append(dict(analysis="hier_boot", component=comp, diff=pt, ci_lo=lo, ci_hi=hi))

# ── ALL Stage-0 output lives under runs/_robustness/stage0/ ──────────────────────────
_S0_DIR = os.path.join(RUNS_PATH, "_robustness", "stage0")
os.makedirs(_S0_DIR, exist_ok=True)
_p = os.path.join(_S0_DIR, "stage0_stats.csv")
pd.DataFrame(_out).to_csv(_p, index=False)
print(f"\nsaved -> {_p}")

In [ ]:
# %load artifacts/_tmp_stage1_genetic_test.py
# ============================================================================
# QUICK TEST — does a GENETIC neighbourhood rescue the degenerate multimodal
# LORE fidelity, and how much does it cost per sample?
#
# Run in the SAME kernel that just ran Stage-1 Phase A (needs: LoreEngine,
# feature_space, FEAT_COLS, _build_faith_subset, ART_PATH, SUMMARIES_PATH,
# and the '[XAI] registered custom layers:' cell already executed).
# ============================================================================
import glob, gc, time, numpy as np, pandas as pd, os, keras

# ---- cost knobs: start SMALL. If seed_53 already answers it, Ctrl-C. --------
NEIGH_TYPE = "genetic"   # 'genetic' = fitness on hard labels (targets opposite class directly).
                         # try 'geneticp' (proba-based, smoother) if 'genetic' underperforms.
TEST_N     = 4           # explained samples / run (small = fast)
TEST_OOS   = 300         # fresh-neighbourhood size for oos fidelity (was 500 under random)
TEST_NGEN  = 10          # GA generations. Lower (e.g. 5) ~halves cost; 10 = library default.
TEST_RUNS  = [
    "20260604-210304/seed_53__baseline__e20",   # vanilla, DEGENERATE under random -> the decisive test
    "20260604-210304/seed_47__baseline__e20",   # vanilla, already healthy under random -> sanity control
]

# random baselines for reference (from lore_fidelity_v2.csv)
_BASE = {
    "20260604-210304/seed_53__baseline__e20": dict(pos=0.020, bacc=0.540, ins=0.9926),
    "20260604-210304/seed_47__baseline__e20": dict(pos=0.571, bacc=0.642, ins=0.9827),
}

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[_rows.run_id.isin(TEST_RUNS)].reset_index(drop=True)

print(f"GENETIC LORE TEST — neigh_type={NEIGH_TYPE}  n={TEST_N}  oos={TEST_OOS}  ngen={TEST_NGEN}\n")
for _, r in _rows.iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    if not ck:
        print(f"[SKIP] {r.run_id} (no checkpoint)"); continue
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)
        eng = LoreEngine(feature_space=feature_space, feat_cols=FEAT_COLS,
                         max_explain_samples=TEST_N, positive_threshold=0.5,
                         lore_random_state=0, oos_samples=TEST_OOS,
                         lore_config={"neigh_type": NEIGH_TYPE, "size": 300,
                                      "ngen": TEST_NGEN, "verbose": False})
        raw_df, imgs, labs = _build_faith_subset(128)
        t0 = time.time()
        stats_, rules, preds = eng.explain(raw_df=raw_df, images=imgs, labels=labs,
                                           model=model, mode="context")
        dt = (time.time() - t0) / 60.0
        n  = max(int(stats_["n_explained"]), 1)
        b  = _BASE.get(r.run_id, {})
        print(f"{r.run_id}")
        print(f"   RANDOM  (ref): pos_rate={b.get('pos', float('nan')):.3f}  "
              f"oos_bacc={b.get('bacc', float('nan')):.3f}  insample={b.get('ins', float('nan')):.4f}")
        print(f"   GENETIC (now): pos_rate={stats_['neigh_pos_rate']:.3f}  "
              f"oos_bacc={stats_['fidelity_oos_bacc']:.3f}  insample={stats_['fidelity']:.4f}"
              f"   [{dt:.1f} min / {n} samples -> {dt/n:.1f} min/sample]", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}", flush=True)
    finally:
        keras.backend.clear_session(); gc.collect()

print("\n" + "=" * 74)
print("HOW TO READ IT")
print("=" * 74)
print(" seed_53 (random pos_rate 0.020, bacc 0.540) is the decisive one:")
print("  - if GENETIC lifts pos_rate to ~0.3-0.5 AND bacc rises clearly (>~0.6)")
print("    => the boundary-crossing neighbourhood FIXES the confound => a scoped genetic")
print("       rerun buys a REAL cross-strategy LORE ranking (a genuinely new result). Worth it.")
print("  - if pos_rate stays ~0.02 => the model is genuinely tabular-invariant on these samples;")
print("    genetic cannot manufacture positives => LORE conclusion stands, and we upgrade")
print("    'plausibly structural' -> 'structural' (rules out the base-rate competing story).")
print(" seed_47 is the control: already healthy under random; genetic must NOT break it.")
print(" min/sample above x (16 samples x N runs) = price of the full rerun. Decide from that.")


In [ ]:
# ======================= STAGE 1 — LORE repaired =======================
import glob, gc, time, numpy as np, pandas as pd, os, keras

# --- GUARD: the custom-layer registration cell MUST have run in this kernel ---------
# Without it, every fixed_scaling / learnable_scaling / controlled_fusion / adaptive_gating
# checkpoint fails to deserialize and the loop silently [FAIL]s 70+ runs while looking busy.
_need = ["modality_attendance>ConstantScale", "modality_attendance>LearnableAlpha",
         "modality_attendance>ScalarMultiply", "modality_attendance>FusionWithGamma",
         "modality_attendance>GateEntropyLoss"]
_missing = [n for n in _need if n not in keras.saving.get_custom_objects()]
if _missing:
    raise RuntimeError(
        "Custom layers NOT registered: " + ", ".join(m.split(">")[1] for m in _missing) +
        "\n-> run the cell that prints '[XAI] registered custom layers:' first.")

LORE2_N           = 16      # explained samples/run. Published n=6 had no SEM; measured cost ~1.9 min/sample.
LORE2_OOS_SAMPLES = 500     # fresh neighbourhood for out-of-sample fidelity
LORE2_OVERWRITE   = False   # False = resume: skip run_ids already in the CSV

# ── STAGED DESIGN: the full 110-run sweep is ~110 h at n=32. Budget is 12 h. ──────────
# Phase A is a GATE, not a survey. A tabular explainer MUST be vacuous on a model that ignores
# the table (a0p000, image-only) and informative on one that is nothing but the table (a1p000).
# If LORE fidelity cannot separate those two, the metric is dead and NO parameter change saves it
# -- and we stop, having spent 8 h instead of 110. Only if the gate opens do we pay for Phase B.
LORE2_PHASE = "A"

_PHASES = {
    # 15 runs x ~30 min = ~8 h   <- fits the budget, and is decisive
    "A": {"vanilla":             ["baseline"],       # reference
          "fixed_scaling":       ["a0p000",          # image-only : LORE should be VACUOUS here
                                  "a1p000"]},        # tabular-only: LORE should be INFORMATIVE here
    # 25 runs x ~30 min = ~12 h  <- only if A opens the gate. Resume makes this additive.
    "B": {"fixed_scaling":       ["a0p500"],
          "learnable_scaling":   ["init_a0p500"],
          "loss_regularization": ["lam1e-03"],
          "controlled_fusion":   ["base"],
          "adaptive_gating":     ["base"]},
}
LORE2_SETTINGS = _PHASES[LORE2_PHASE]

# ── ALL Stage-1 output lives under runs/_robustness/stage1_lore/ ──────────────────────
_S1_DIR = os.path.join(RUNS_PATH, "_robustness", "stage1_lore")
os.makedirs(_S1_DIR, exist_ok=True)
_OUT = os.path.join(_S1_DIR, "lore_fidelity_v2.csv")
if LORE2_OVERWRITE and os.path.exists(_OUT):
    os.remove(_OUT)
_done = set(pd.read_csv(_OUT).run_id) if os.path.exists(_OUT) else set()

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[[t in LORE2_SETTINGS.get(f, []) for f, t in zip(_rows.impl_name, _rows.setting_tag)]]
_rows = _rows[~_rows.run_id.isin(_done)].reset_index(drop=True)
print(f"PHASE {LORE2_PHASE}: {len(_rows)} runs to do ({len(_done)} already in "
      f"{os.path.basename(_OUT)}), ~{LORE2_N * 2} min/run -> ~{len(_rows) * LORE2_N * 2 / 60:.1f} h\n")

_t0 = time.time()
for i, r in _rows.iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    if not ck:
        print(f"[SKIP] {r.run_id}"); continue
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)
        eng = LoreEngine(feature_space=feature_space, feat_cols=FEAT_COLS,
                         max_explain_samples=LORE2_N, positive_threshold=0.5,
                         lore_random_state=0, oos_samples=LORE2_OOS_SAMPLES)
        raw_df, imgs, labs = _build_faith_subset(128)
        stats_, rules, preds = eng.explain(raw_df=raw_df, images=imgs, labels=labs,
                                           model=model, mode="context")
        row = {
            "impl_name": r.impl_name, "run_id": r.run_id, "seed": r.seed,
            "setting_tag": r.setting_tag, "n_explained": stats_["n_explained"],
            "fidelity_insample": stats_["fidelity"],          # the old, saturating metric
            "fidelity_sem": stats_["fidelity_sem"],           # <- did not exist at n=6
            "fidelity_oos_acc": stats_["fidelity_oos_acc"],
            "fidelity_oos_bacc": stats_["fidelity_oos_bacc"], # <- PRIMARY (kills class inflation)
            "fidelity_oos_r2": stats_["fidelity_oos_r2"],     # <- fragile, see note below
            "fidelity_oos_r2_sem": stats_["fidelity_oos_r2_sem"],
            "neigh_pos_rate": stats_["neigh_pos_rate"],       # <- DIAGNOSTIC, read this first
            "rule_len": stats_["rule_len"],
            "modality_balance": r.modality_balance,
        }
        # incremental append: a kill now costs ONE run, not the whole loop
        pd.DataFrame([row]).to_csv(_OUT, mode="a", header=not os.path.exists(_OUT), index=False)
        el = time.time() - _t0
        print(f"[{i+1}/{len(_rows)}] {r.run_id}  insample={row['fidelity_insample']:.4f} "
              f"oos_bacc={row['fidelity_oos_bacc']:.4f} oos_r2={row['fidelity_oos_r2']:.2f} "
              f"pos_rate={row['neigh_pos_rate']:.3f}  "
              f"({el/60:.0f} min, ETA {el/(i+1)*(len(_rows)-i-1)/60:.0f} min)", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}", flush=True)
    finally:
        keras.backend.clear_session(); gc.collect()

# ============================== READ-OUT ==============================
_l = pd.read_csv(_OUT)

print("\n" + "=" * 78)
print("STEP 1 — IS THE NEIGHBOURHOOD DEGENERATE?  (read this BEFORE the fidelity numbers)")
print("=" * 78)
print("neigh_pos_rate = fraction of the fresh neighbourhood the MODEL calls positive.")
print("If it sits at ~0 or ~1, the neighbourhood never crosses the decision boundary: then a")
print("low oos_bacc (~0.5) and a very negative oos_r2 are re-confirming THAT degeneracy, not")
print("showing 'the surrogate does not generalize'. The fix would be the neighbourhood GENERATOR")
print("(genetic / class-balanced), not the fidelity metric.")
print()
print(_l.groupby("impl_name")["neigh_pos_rate"].agg(["mean", "min", "max"]).round(3).to_string())

print("\n" + "=" * 78)
print("STEP 2 — DOES LORE FIDELITY DISCRIMINATE BETWEEN STRATEGIES NOW?")
print("=" * 78)
print(_l.groupby("impl_name")[["fidelity_insample", "fidelity_sem", "fidelity_oos_acc",
                               "fidelity_oos_bacc", "fidelity_oos_r2", "neigh_pos_rate"]]
      .mean().round(4).to_string())
print("\nSpread across families (a dead metric has ~0 spread):")
for c in ("fidelity_insample", "fidelity_oos_acc", "fidelity_oos_bacc"):
    fam = _l.groupby("impl_name")[c].mean()
    print(f"  {c:<20s} range={fam.max()-fam.min():.4f}   sd={fam.std():.4f}")
print("  (oos_r2 deliberately omitted here: its denominator is the variance of the model prob")
print("   INSIDE the neighbourhood, so it explodes negative whenever that variance is small.")
print("   Report balanced accuracy; treat r2 as directional only.)")

print("\n" + "=" * 78)
print("STEP 3 — THE ACID TEST: two deliberately opposite models")
print("=" * 78)
_ext = _l[_l.setting_tag.isin(["a0p000", "a1p000"])]
if len(_ext):
    print(_ext.groupby("setting_tag")[["fidelity_insample", "fidelity_oos_bacc",
                                       "neigh_pos_rate"]].agg(["mean", "std"]).round(4).to_string())
    print("  a0p000 = image-only (ignores the table); a1p000 = tabular-only.")
    print("  A tabular explainer MUST be vacuous on a0p000 and informative on a1p000.")
    print("  If it cannot tell THESE apart, LORE fidelity is dead and no parameter fixes it.")
else:
    print("  [missing — the fixed_scaling runs did not load. Check the registration guard.]")


## Stage 2 — il controllo decisivo su SHAP

**La domanda**: l'anti-fedelta' tabellare e' una proprieta' del modello, o un artefatto di come e' impostato
lo spiegatore?

`GradientExplainer` e' *expected gradients*: integra il cammino dal background al campione **su tutti gli input,
immagine compresa**. Quindi lo SHAP tabellare attuale attribuisce rispetto a un baseline in cui **anche l'immagine
e' sostituita**. La deletion invece perturba **solo la tabella, tenendo l'immagine vera**.
**SHAP sta spiegando una funzione diversa da quella che la deletion sonda.**

Questa cella ricalcola il ranking tabellare con un attributore che spiega **esattamente** la funzione che la
deletion perturba, e cambia **quattro** cose insieme:

| # | da | a | esclude |
| --- | --- | --- | --- |
| 1 | GradientExplainer | **Permutation** | artefatto della famiglia di explainer |
| 2 | background 64 dalla validation (dentro l'insieme spiegato) | **dal training, disgiunto** | artefatto del background |
| 3 | spazio encoded (127) | **raw (39), dove la deletion agisce** | mismatch raw<->encoded |
| 4 | immagine interpolata verso il background | **tenuta fissa al valore vero** | il mismatch di funzione |

**Come leggerlo (design adattivo):**
- **anti-fedelta' SOPRAVVIVE** -> non e' un artefatto dello spiegatore, la claim e' rafforzata. Ma la conclusione
  onesta si ferma li': la causa residua piu' probabile diventa la **ridondanza** (25/33 feature derivate da 8),
  che resta **non testata**. Formulare come *"robusta rispetto al metodo di attribuzione"*, NON *"e' del modello"*.
- **SPARISCE o si inverte** -> la claim centrale dello studio e' un artefatto. Allora si ablaziona **un fattore
  alla volta** (4 -> 2 -> 3 -> 1) per isolare la causa.

In [ ]:
# ================= STAGE 2 — aligned attributor (the decisive test) =================
import shap, glob, gc, time, numpy as np, pandas as pd, os, keras, tensorflow as tf

# --- GUARD: the custom-layer registration cell MUST have run in this kernel ---------
# Without it, every fixed_scaling / learnable_scaling / controlled_fusion / adaptive_gating
# checkpoint fails to deserialize and the loop silently [FAIL]s 70+ runs while looking busy.
_need = ["modality_attendance>ConstantScale", "modality_attendance>LearnableAlpha",
         "modality_attendance>ScalarMultiply", "modality_attendance>FusionWithGamma",
         "modality_attendance>GateEntropyLoss"]
_missing = [n for n in _need if n not in keras.saving.get_custom_objects()]
if _missing:
    raise RuntimeError(
        "Custom layers NOT registered: " + ", ".join(m.split(">")[1] for m in _missing) +
        "\n-> run the cell that prints '[XAI] registered custom layers:' first.")


S2_BG_SIZE   = 100     # background rows drawn from TRAINING (disjoint from what we explain)
S2_MAX_EVALS = 2 * len(FEAT_COLS) + 1     # Permutation: exact antithetic pass over 39 features
S2_LIMIT     = None    # set to an int to smoke-test first

_bg_raw = training_df[FEAT_COLS].sample(S2_BG_SIZE, random_state=0).reset_index(drop=True)


# ── label encoding: SHAP needs an all-numeric matrix ──────────────────────────────────
# shap.explainers.Permutation does arithmetic on the background matrix, so an object array holding
# the 6 string columns blows up with "unsupported operand type(s) for -: 'str' and 'str'".
# Encode the categoricals as integer codes for SHAP, decode them back to strings inside f() right
# before the FeatureSpace -- the same trick LoreEngine already uses.
# SOUND for Permutation specifically: the explainer only ever SUBSTITUTES whole values taken from
# the background rows, it never interpolates between them, so the integer code is an opaque label
# and no arithmetic meaning is imputed to it.
_S2_CAT = [c for c in FEAT_COLS if not pd.api.types.is_numeric_dtype(training_df[c])]
_S2_LMAP, _S2_IMAP = {}, {}
for _c in _S2_CAT:
    # vocabulary as STRINGS, matching what _to_feature_dict feeds the FeatureSpace (NaN included:
    # it becomes the literal level "nan" there too)
    _lv = sorted(pd.concat([training_df[_c], _bg_raw[_c]]).astype(str).unique())
    _S2_LMAP[_c] = {v: i for i, v in enumerate(_lv)}
    _S2_IMAP[_c] = {i: v for v, i in _S2_LMAP[_c].items()}
print(f"[S2] label-encoded {len(_S2_CAT)} categorical cols: "
      f"{ {c: len(_S2_LMAP[c]) for c in _S2_CAT} }")


def _s2_encode(df):
    """raw DataFrame -> float matrix SHAP can do arithmetic on."""
    out = df[FEAT_COLS].copy()
    for c in FEAT_COLS:
        if c in _S2_CAT:
            out[c] = out[c].astype(str).map(_S2_LMAP[c]).fillna(0)
        else:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.to_numpy(dtype=np.float64)


def _s2_decode(X):
    """float matrix -> raw DataFrame with the dtypes the FeatureSpace expects."""
    df = pd.DataFrame(np.atleast_2d(np.asarray(X, dtype=np.float64)), columns=FEAT_COLS)
    for c in FEAT_COLS:
        if c in _S2_CAT:
            idx = df[c].round().fillna(0).astype(int).clip(0, len(_S2_IMAP[c]) - 1)
            df[c] = [_S2_IMAP[c][i] for i in idx]
        else:
            df[c] = df[c].astype("float32")
    return df


def aligned_tab_importance(model, raw_df, imgs):
    """Per-sample |attribution| over the 39 RAW features, from a Permutation explainer that
    scores the model head with THIS sample's image embedding held fixed -- i.e. exactly the
    function the tabular deletion perturbs."""
    gap = [l for l in model.layers if isinstance(l, keras.layers.GlobalAveragePooling2D)]
    img_emb, feat_in = gap[0].output, model.input["features"]
    img_enc = keras.Model(model.input["images"], img_emb)
    head = keras.Model([img_emb, feat_in], model.output)
    head_fn = tf.function(lambda e, f: head([e, f], training=False), reduce_retracing=True)

    embs = img_enc.predict(imgs, batch_size=32, verbose=0)      # backbone runs ONCE per sample

    def make_f(emb_i):
        e = tf.constant(emb_i[np.newaxis], dtype=tf.float32)
        def f(X):
            enc = feature_space(_to_feature_dict(_s2_decode(X)))
            n = int(enc.shape[0])
            return head_fn(tf.repeat(e, n, axis=0), enc).numpy()[:, 0]
        return f

    X  = _s2_encode(raw_df)
    BG = _s2_encode(_bg_raw)                                     # background = TRAINING rows
    imp = np.zeros((len(raw_df), len(FEAT_COLS)))
    for i in range(len(raw_df)):
        ex = shap.explainers.Permutation(make_f(embs[i]), BG)
        sv = ex(X[i:i + 1], max_evals=S2_MAX_EVALS, silent=True)
        imp[i] = np.abs(np.asarray(sv.values).reshape(-1))
    return imp


# ── ALL Stage-2 output lives under runs/_robustness/stage2_aligned_attr/ ──────────────
# Nothing is written into runs/<run_id>/, so these checks stay physically separate from the
# published artifacts and can be `rm -rf`'d without touching a single canonical file.
_S2_DIR = os.path.join(RUNS_PATH, "_robustness", "stage2_aligned_attr")
os.makedirs(_S2_DIR, exist_ok=True)
_OUT2 = os.path.join(_S2_DIR, "faithfulness_aligned_attr.csv")
_done2 = set(pd.read_csv(_OUT2).run_id) if os.path.exists(_OUT2) else set()

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[~_rows.run_id.isin(_done2)]
if S2_LIMIT:
    _rows = _rows.head(S2_LIMIT)
print(f"{len(_rows)} runs to do ({len(_done2)} already done)")

_t0 = time.time()
for i, r in _rows.reset_index(drop=True).iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)
        raw_df, imgs, labs = _build_faith_subset(FAITH_N_SAMPLES)
        imp = aligned_tab_importance(model, raw_df, imgs)
        out = explanation_faithfulness(
            model, rd, int(r.best_epoch),
            methods=("tabular",),          # only the tabular ranking changed
            tab_imp_override=imp,
            out_name="faithfulness_aligned_attr",
            out_dir=os.path.join(_S2_DIR, str(r.run_id)),   # NOT runs/<run_id>/xai/
            save=True, verbose=False,
        )
        row = {"impl_name": r.impl_name, "run_id": r.run_id, "seed": r.seed,
               "setting_tag": r.setting_tag,
               "tab_gain_aligned": out["faithfulness_tabular_del_gain"],
               "tab_gain_published": r.faithfulness_tabular_del_gain}
        pd.DataFrame([row]).to_csv(_OUT2, mode="a", header=not os.path.exists(_OUT2), index=False)
        el = time.time() - _t0
        print(f"[{i+1}/{len(_rows)}] {r.run_id}  aligned={out['faithfulness_tabular_del_gain']:+.4f}  "
              f"published={r.faithfulness_tabular_del_gain:+.4f}   "
              f"({el/60:.0f} min, ETA {el/(i+1)*(len(_rows)-i-1)/60:.0f} min)", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}")
    finally:
        keras.backend.clear_session(); gc.collect()

_a = pd.read_csv(_OUT2)

print("\n" + "=" * 78)
print("DOES THE TABULAR ANTI-FAITHFULNESS SURVIVE A COMPLETELY DIFFERENT ATTRIBUTOR?")
print("=" * 78)
print(_a.groupby("impl_name")[["tab_gain_published", "tab_gain_aligned"]].mean().round(4).to_string())
_n = (_a.tab_gain_aligned < 0).sum()
print(f"\nstill NEGATIVE (anti-faithful) in {_n}/{len(_a)} runs")
print(f"mean published = {_a.tab_gain_published.mean():+.4f}")
print(f"mean aligned   = {_a.tab_gain_aligned.mean():+.4f}")
print("\nSURVIVES  -> not an explainer artifact. Claim strengthened, BUT redundancy remains untested.")
print("FLIPS     -> the study's central claim is an artifact. Ablate one factor at a time.")

In [ ]:
# ============ STAGE 2b — control: SHAP originale, FeatureSpace attuale ============
# Il terzo angolo del 2x2. Stage 2 ha cambiato DUE cose insieme (attributore E FeatureSpace).
# Questa cella tiene il ranking SHAP ORIGINALE (gli array |SHAP| sono gia' su disco, salvati da
# ShapCallback durante il training) e lo valuta sotto il FeatureSpace ATTUALE.
#
#                        | ranking SHAP originale | ranking Permutation allineato
#   FeatureSpace pubbl.  | tab_gain_published     | --
#   FeatureSpace attuale | tab_gain_control  <--  | tab_gain_aligned
#
#   published vs control -> isola la DERIVA DEL FEATURESPACE
#   control   vs aligned -> isola l'effetto dell'ATTRIBUTORE (a FeatureSpace costante)
#
# Costo: solo la componente tabellare, ~36 s/run -> ~1 h su 110 run. Riprendibile.
import glob, gc, time, numpy as np, pandas as pd, os, keras

_need = ["modality_attendance>ConstantScale", "modality_attendance>LearnableAlpha",
         "modality_attendance>ScalarMultiply", "modality_attendance>FusionWithGamma",
         "modality_attendance>GateEntropyLoss"]
_missing = [n for n in _need if n not in keras.saving.get_custom_objects()]
if _missing:
    raise RuntimeError("Custom layers NOT registered: "
                       + ", ".join(m.split(">")[1] for m in _missing))

S2B_LIMIT = None

_S2B_DIR = os.path.join(RUNS_PATH, "_robustness", "stage2b_control")
os.makedirs(_S2B_DIR, exist_ok=True)
_OUT2B = os.path.join(_S2B_DIR, "faithfulness_control_fs.csv")
_done2b = set(pd.read_csv(_OUT2B).run_id) if os.path.exists(_OUT2B) else set()

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[~_rows.run_id.isin(_done2b)]
if S2B_LIMIT:
    _rows = _rows.head(S2B_LIMIT)
print(f"{len(_rows)} runs to do ({len(_done2b)} already done), ~36 s/run\n")

_t0 = time.time()
for i, r in _rows.reset_index(drop=True).iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)
        # NO tab_imp_override -> explanation_faithfulness carica gli |SHAP| da disco.
        # Stesso ranking della run pubblicata; cambia solo il FeatureSpace.
        out = explanation_faithfulness(
            model, rd, int(r.best_epoch),
            methods=("tabular",),
            out_name="faithfulness_control_fs",
            out_dir=os.path.join(_S2B_DIR, str(r.run_id)),   # NON runs/<run_id>/xai/
            save=True, verbose=False,
        )
        row = {"impl_name": r.impl_name, "run_id": r.run_id, "seed": r.seed,
               "setting_tag": r.setting_tag,
               "tab_gain_control": out["faithfulness_tabular_del_gain"],
               "tab_gain_published": r.faithfulness_tabular_del_gain,
               "modality_balance": r.modality_balance}
        pd.DataFrame([row]).to_csv(_OUT2B, mode="a", header=not os.path.exists(_OUT2B), index=False)
        el = time.time() - _t0
        print(f"[{i+1}/{len(_rows)}] {r.run_id}  control={row['tab_gain_control']:+.4f}  "
              f"published={row['tab_gain_published']:+.4f}   "
              f"({el/60:.0f} min, ETA {el/(i+1)*(len(_rows)-i-1)/60:.0f} min)", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}", flush=True)
    finally:
        keras.backend.clear_session(); gc.collect()

# ============================== IL 2x2 ==============================
from scipy import stats

_c = pd.read_csv(_OUT2B)
_apath = os.path.join(RUNS_PATH, "_robustness", "stage2_aligned_attr",
                      "faithfulness_aligned_attr.csv")
_a = pd.read_csv(_apath) if os.path.exists(_apath) else None

print("\n" + "=" * 78)
print("A. FEATURESPACE DRIFT   (published vs control -- stesso ranking, FeatureSpace diverso)")
print("=" * 78)
_r, _p = stats.pearsonr(_c.tab_gain_published, _c.tab_gain_control)
_rho, _ps = stats.spearmanr(_c.tab_gain_published, _c.tab_gain_control)
_d = (_c.tab_gain_control - _c.tab_gain_published)
print(f"  n={len(_c)}   pearson r={_r:+.3f} (p={_p:.2g})   spearman rho={_rho:+.3f} (p={_ps:.2g})")
print(f"  mean published={_c.tab_gain_published.mean():+.4f}   mean control={_c.tab_gain_control.mean():+.4f}")
print(f"  mean |differenza| = {_d.abs().mean():.4f}   max |differenza| = {_d.abs().max():.4f}")
print("  -> r vicino a 1 = la deriva del FeatureSpace NON e' cio' che muove i numeri.")

if _a is not None:
    _m = _c.merge(_a[["run_id", "tab_gain_aligned"]], on="run_id")

    print("\n" + "=" * 78)
    print("B. EFFETTO ATTRIBUTORE   (control vs aligned -- STESSO FeatureSpace, attributore diverso)")
    print("=" * 78)
    _r2, _p2 = stats.pearsonr(_m.tab_gain_control, _m.tab_gain_aligned)
    _rho2, _ps2 = stats.spearmanr(_m.tab_gain_control, _m.tab_gain_aligned)
    print(f"  n={len(_m)}   pearson r={_r2:+.3f} (p={_p2:.2g})   spearman rho={_rho2:+.3f} (p={_ps2:.2g})")
    print(f"  mean control={_m.tab_gain_control.mean():+.4f}   mean aligned={_m.tab_gain_aligned.mean():+.4f}")
    print(f"  anti-fedeli (<0):  control {(_m.tab_gain_control < 0).sum()}/{len(_m)}   "
          f"aligned {(_m.tab_gain_aligned < 0).sum()}/{len(_m)}")

    print("\n" + "=" * 78)
    print("C. LA CLAIM CHE DIPENDE DAL RANKING")
    print("=" * 78)
    print("Report attuale: modality_balance <-> tabular gain, rho = -0.63 (p~1e-13), cioe' 'piu' un")
    print("modello si appoggia al tabellare, piu' la sua spiegazione e' anti-fedele'. E' un'affermazione")
    print("sul RANKING: sopravvive solo se sopravvive in tutte e tre le colonne.")
    for _lab, _col in [("published", "tab_gain_published"), ("control", "tab_gain_control"),
                       ("aligned", "tab_gain_aligned")]:
        _rr, _pp = stats.spearmanr(_m.modality_balance, _m[_col])
        _flag = "REGGE" if (_pp < 0.05 and _rr < 0) else "NON REGGE"
        print(f"  modality_balance <-> {_lab:<10s} rho={_rr:+.3f} (p={_pp:.2g})   {_flag}")

    print("\n" + "=" * 78)
    print("D. MEDIE PER FAMIGLIA (tutte e tre le colonne)")
    print("=" * 78)
    print(_m.groupby("impl_name")[["tab_gain_published", "tab_gain_control",
                                   "tab_gain_aligned"]].mean().round(4).to_string())

    print("\n" + "=" * 78)
    print("E. IL TEST ACIDO, ISOLATO   (a1p000 = solo-tabellare: la tabella e' l'UNICO input)")
    print("=" * 78)
    _acid = _m[_m.setting_tag == "a1p000"]
    if len(_acid):
        print(_acid[["seed", "tab_gain_published", "tab_gain_control",
                     "tab_gain_aligned"]].round(4).to_string(index=False))
        print()
        print("  Su questo modello uno spiegatore tabellare DEVE essere fedele: non c'e' altro input.")
        print("  control NEGATIVO + aligned POSITIVO -> la colpa e' dell'ATTRIBUTORE, non del")
        print("     FeatureSpace, e la claim tabellare del report va ritirata.")
        print("  control POSITIVO come aligned       -> la colpa e' del FEATURESPACE: i numeri")
        print("     pubblicati sono inquinati dal bug di adapt, e vanno semplicemente rifatti.")
else:
    print("\n[CSV di Stage 2 non trovato -- lancia prima la cella aligned-attributor]")

## Stage 2b — il controllo che scioglie il confondente

Stage 2 cambia **due cose insieme**: l'attributore (Gradient -> Permutation) **e** il FeatureSpace
(quello pubblicato fu adattato in modo non deterministico; questo e' quello salvato e ricaricato).
Da solo non dice quale delle due muove i numeri.

Questa cella calcola il **terzo angolo** del 2x2: il ranking SHAP **originale** (gli array `|SHAP|`
sono gia' su disco, salvati da `ShapCallback`) sotto il FeatureSpace **attuale**.

| | ranking SHAP originale | ranking Permutation allineato |
| --- | --- | --- |
| **FeatureSpace pubblicato** | `tab_gain_published` (nel master CSV) | — (non esiste, ne' serve) |
| **FeatureSpace attuale** | **`tab_gain_control`** (questa cella) | `tab_gain_aligned` (Stage 2) |

- `published` vs `control` -> isola la **deriva del FeatureSpace**
- `control` vs `aligned`   -> isola l'effetto dell'**attributore** (a FeatureSpace costante)

**Costo**: solo la componente tabellare, ~36 s/run -> **~1 h** su 110 run.
Da lanciare **dopo** Stage 2 (ne rilegge il CSV per il confronto).

In [ ]:
# ============ STAGE 2b — control: SHAP originale, FeatureSpace attuale ============
import glob, gc, time, numpy as np, pandas as pd, os, keras

# --- GUARD: the custom-layer registration cell MUST have run in this kernel ---------
_need = ["modality_attendance>ConstantScale", "modality_attendance>LearnableAlpha",
         "modality_attendance>ScalarMultiply", "modality_attendance>FusionWithGamma",
         "modality_attendance>GateEntropyLoss"]
_missing = [n for n in _need if n not in keras.saving.get_custom_objects()]
if _missing:
    raise RuntimeError(
        "Custom layers NOT registered: " + ", ".join(m.split(">")[1] for m in _missing) +
        "\n-> run the cell that prints '[XAI] registered custom layers:' first.")

S2B_LIMIT = None

# ── ALL Stage-2b output lives under runs/_robustness/stage2b_control/ ─────────────────
_S2B_DIR = os.path.join(RUNS_PATH, "_robustness", "stage2b_control")
os.makedirs(_S2B_DIR, exist_ok=True)
_OUT2B = os.path.join(_S2B_DIR, "faithfulness_control_fs.csv")
_done2b = set(pd.read_csv(_OUT2B).run_id) if os.path.exists(_OUT2B) else set()

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[~_rows.run_id.isin(_done2b)]
if S2B_LIMIT:
    _rows = _rows.head(S2B_LIMIT)
print(f"{len(_rows)} runs to do ({len(_done2b)} already done), ~36 s/run\n")

_t0 = time.time()
for i, r in _rows.reset_index(drop=True).iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)
        # NO tab_imp_override -> explanation_faithfulness loads the |SHAP| arrays saved by
        # ShapCallback at train time. Same ranking as the published run; only the FeatureSpace
        # (and thus the model's predictions on perturbed rows) differs.
        out = explanation_faithfulness(
            model, rd, int(r.best_epoch),
            methods=("tabular",),
            out_name="faithfulness_control_fs",
            out_dir=os.path.join(_S2B_DIR, str(r.run_id)),   # NOT runs/<run_id>/xai/
            save=True, verbose=False,
        )
        row = {"impl_name": r.impl_name, "run_id": r.run_id, "seed": r.seed,
               "setting_tag": r.setting_tag,
               "tab_gain_control": out["faithfulness_tabular_del_gain"],
               "tab_gain_published": r.faithfulness_tabular_del_gain,
               "modality_balance": r.modality_balance}
        pd.DataFrame([row]).to_csv(_OUT2B, mode="a", header=not os.path.exists(_OUT2B), index=False)
        el = time.time() - _t0
        print(f"[{i+1}/{len(_rows)}] {r.run_id}  control={row['tab_gain_control']:+.4f}  "
              f"published={row['tab_gain_published']:+.4f}   "
              f"({el/60:.0f} min, ETA {el/(i+1)*(len(_rows)-i-1)/60:.0f} min)", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}", flush=True)
    finally:
        keras.backend.clear_session(); gc.collect()

# ============================== THE 2x2 READ-OUT ==============================
from scipy import stats

_c = pd.read_csv(_OUT2B)
_apath = os.path.join(RUNS_PATH, "_robustness", "stage2_aligned_attr",
                      "faithfulness_aligned_attr.csv")
_a = pd.read_csv(_apath) if os.path.exists(_apath) else None

print("\n" + "=" * 78)
print("A. FEATURESPACE DRIFT   (published vs control -- same SHAP ranking, different FeatureSpace)")
print("=" * 78)
_r, _p = stats.pearsonr(_c.tab_gain_published, _c.tab_gain_control)
_rho, _ps = stats.spearmanr(_c.tab_gain_published, _c.tab_gain_control)
_d = (_c.tab_gain_control - _c.tab_gain_published)
print(f"  n={len(_c)}   pearson r={_r:+.3f} (p={_p:.2g})   spearman rho={_rho:+.3f} (p={_ps:.2g})")
print(f"  mean published={_c.tab_gain_published.mean():+.4f}   mean control={_c.tab_gain_control.mean():+.4f}")
print(f"  mean |difference| = {_d.abs().mean():.4f}   max |difference| = {_d.abs().max():.4f}")
print("  -> r near 1 means the FeatureSpace drift is NOT what moves the numbers.")

if _a is not None:
    _m = _c.merge(_a[["run_id", "tab_gain_aligned"]], on="run_id")
    print("\n" + "=" * 78)
    print("B. ATTRIBUTOR EFFECT   (control vs aligned -- SAME FeatureSpace, different attributor)")
    print("=" * 78)
    _r2, _p2 = stats.pearsonr(_m.tab_gain_control, _m.tab_gain_aligned)
    _rho2, _ps2 = stats.spearmanr(_m.tab_gain_control, _m.tab_gain_aligned)
    print(f"  n={len(_m)}   pearson r={_r2:+.3f} (p={_p2:.2g})   spearman rho={_rho2:+.3f} (p={_ps2:.2g})")
    print(f"  mean control={_m.tab_gain_control.mean():+.4f}   mean aligned={_m.tab_gain_aligned.mean():+.4f}")
    print(f"  anti-faithful (<0):  control {(_m.tab_gain_control < 0).sum()}/{len(_m)}   "
          f"aligned {(_m.tab_gain_aligned < 0).sum()}/{len(_m)}")

    print("\n" + "=" * 78)
    print("C. THE CLAIM THAT DEPENDS ON THE RANKING")
    print("=" * 78)
    print("Published report: modality_balance <-> tabular gain, rho = -0.63 (p~1e-13), i.e. 'the more")
    print("a model leans on the tabular branch, the more anti-faithful its explanation'. That is a")
    print("claim about the RANKING, so it only survives if it survives in all three columns:")
    for _lab, _col in [("published", "tab_gain_published"), ("control", "tab_gain_control"),
                       ("aligned", "tab_gain_aligned")]:
        _rr, _pp = stats.spearmanr(_m.modality_balance, _m[_col])
        _flag = "HOLDS" if (_pp < 0.05 and _rr < 0) else "DOES NOT HOLD"
        print(f"  {_lab:<10s} rho={_rr:+.3f} (p={_pp:.2g})   {_flag}")

    print("\n" + "=" * 78)
    print("D. PER-FAMILY MEANS (all three columns)")
    print("=" * 78)
    print(_m.groupby("impl_name")[["tab_gain_published", "tab_gain_control",
                                   "tab_gain_aligned"]].mean().round(4).to_string())
    print("\nVERDICT TEMPLATE:")
    print("  sign anti-faithful in all three  -> robust to BOTH confounders. State it.")
    print("  A: r~1 but B: r low             -> the ATTRIBUTOR moves the ranking, not the FeatureSpace.")
    print("     Then any rho-based trade-off claim must be re-stated on the aligned column, or dropped.")
else:
    print("\n[Stage 2 CSV not found -- run the aligned-attributor cell first for the B/C/D blocks]")

## Stage 3 — immagine: 5 subset disgiunti + sweep sigma

Due domande:
1. **Il ranking tra famiglie in deletion e' stabile al campionamento?** La claim sulla fusione e' un'affermazione
   sul *ranking*, stimata su **un solo** batch di 128 su 1091. -> 5 subset **disgiunti**. Il subset 0 e' quello
   canonico: i suoi numeri **devono** riprodurre `faithfulness.json`.
2. **La divergenza deletion/insertion (r = +0.008) e' un artefatto OOD?** L'insertion parte dall'immagine
   **completamente sfocata** -> molto fuori distribuzione. -> sweep `sigma in {4, 8, 16}`.

**Costo**: ~9 min/run. I 5 subset su 110 run sono ~17 h; lo sweep sigma e' su **1 subset** (non 5), ~10 h.
Conviene lanciarli separatamente, di notte.

In [ ]:
  import pandas as pd                                                                                      
  p = "/mnt/disk1/home/cgiovannoni/isic24-dum/artifacts/enhanced/runs/_robustness/stage3_image/faithfulness_image_robustness.csv"
  df = pd.read_csv(p)                                                                                      
  partial = df.groupby("run_id").subset.nunique().pipe(lambda s: s[s < 3]).index.tolist()                  
  print("rimuovo run_id parziali:", partial)                                                               

In [ ]:
# ===================== STAGE 3 — image robustness =====================
import glob, gc, time, numpy as np, pandas as pd, os, keras

# --- GUARD: the custom-layer registration cell MUST have run in this kernel ---------
# Without it, every fixed_scaling / learnable_scaling / controlled_fusion / adaptive_gating
# checkpoint fails to deserialize and the loop silently [FAIL]s 70+ runs while looking busy.
_need = ["modality_attendance>ConstantScale", "modality_attendance>LearnableAlpha",
         "modality_attendance>ScalarMultiply", "modality_attendance>FusionWithGamma",
         "modality_attendance>GateEntropyLoss"]
_missing = [n for n in _need if n not in keras.saving.get_custom_objects()]
if _missing:
    raise RuntimeError(
        "Custom layers NOT registered: " + ", ".join(m.split(">")[1] for m in _missing) +
        "\n-> run the cell that prints '[XAI] registered custom layers:' first.")


S3_K          = 3         # disjoint subsets. 5 would be ~17 h on 110 runs; 3 is ~10 h (budget: 12 h).
S3_DO_SUBSETS = False      # subset 0 == the canonical published batch (reproduction anchor)
S3_DO_SIGMA   = True     # sigma sweep on subset 0 only (~10 h) — separate pass, separate night
S3_SIGMAS     = (4, 8, 16)
S3_LIMIT      = None

_subsets = faith_subsets(k=S3_K, n=FAITH_N_SAMPLES, seed=0)   # subset 0 == the canonical first 128
print("subsets:", [(len(s), int(s[0]), int(s[-1])) for s in _subsets])

# ── ALL Stage-3 output lives under runs/_robustness/stage3_image/ (see Stage 2 note) ──
_S3_DIR = os.path.join(RUNS_PATH, "_robustness", "stage3_image")
os.makedirs(_S3_DIR, exist_ok=True)
_OUT3 = os.path.join(_S3_DIR, "faithfulness_image_sigma.csv")
_done3 = set(pd.read_csv(_OUT3).run_id) if os.path.exists(_OUT3) else set()

_rows = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_rows = _rows[~_rows.run_id.isin(_done3)]
if S3_LIMIT:
    _rows = _rows.head(S3_LIMIT)
print(f"{len(_rows)} runs to do ({len(_done3)} already done)")

_t0 = time.time()
for i, r in _rows.reset_index(drop=True).iterrows():
    rd = os.path.join(ART_PATH, r.run_dir.split("artifacts/enhanced/")[-1])
    ck = glob.glob(os.path.join(rd, "checkpoints", "best_*.keras"))
    try:
        model = keras.models.load_model(ck[0], compile=False, safe_mode=False)

        jobs = []
        if S3_DO_SUBSETS:
            jobs += [dict(kind="subset", subset=k, sigma=8, idx=s) for k, s in enumerate(_subsets)]
        if S3_DO_SIGMA:
            jobs += [dict(kind="sigma", subset=0, sigma=sg, idx=_subsets[0])
                     for sg in S3_SIGMAS if not (S3_DO_SUBSETS and sg == 8)]

        for j in jobs:
            # NOTE: the |SHAP| arrays on disk only cover the canonical first 128, so for the other
            # subsets the tabular ranking is recomputed inside explanation_faithfulness (fallback).
            o = explanation_faithfulness(
                model, rd, int(r.best_epoch),
                methods=("image",) if j["kind"] == "sigma" else ("tabular", "image", "joint"),
                subset_idx=j["idx"], blur_sigma=j["sigma"],
                out_name=f"faith_s{j['subset']}_sig{j['sigma']}",
                out_dir=os.path.join(_S3_DIR, str(r.run_id)),   # NOT runs/<run_id>/xai/
                save=True, verbose=False,
            )
            row = {"impl_name": r.impl_name, "run_id": r.run_id, "seed": r.seed,
                   "setting_tag": r.setting_tag, "subset": j["subset"], "sigma": j["sigma"]}
            row.update({k: v for k, v in o.items() if k.startswith("faithfulness_")})
            pd.DataFrame([row]).to_csv(_OUT3, mode="a", header=not os.path.exists(_OUT3), index=False)

        el = time.time() - _t0
        print(f"[{i+1}/{len(_rows)}] {r.run_id}  ({el/60:.0f} min, "
              f"ETA {el/(i+1)*(len(_rows)-i-1)/60:.0f} min)", flush=True)
    except Exception as e:
        print(f"[FAIL] {r.run_id}: {type(e).__name__}: {e}")
    finally:
        keras.backend.clear_session(); gc.collect()

_s = pd.read_csv(_OUT3)

# ---- reproduction anchor: subset 0 at sigma=8 must match the published numbers ----
_anchor = _s[(_s.subset == 0) & (_s.sigma == 8)].merge(
    pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))[["run_id", "faithfulness_image_del_gain"]], on="run_id", suffixes=("", "_pub"))
if len(_anchor):
    err = (_anchor.faithfulness_image_del_gain - _anchor.faithfulness_image_del_gain_pub).abs().max()
    print(f"\nANCHOR subset0/sigma8 vs published: max diff = {err:.4f}")
    print("  (a small non-zero diff is EXPECTED: the FeatureSpace was re-adapted non-deterministically")
    print("   when the published numbers were produced — see the report's reproducibility section)")

# ---- is the family ranking stable across subsets? ----
if S3_DO_SUBSETS:
    print("\n" + "=" * 78); print("IS THE FAMILY RANKING STABLE ACROSS SUBSETS?"); print("=" * 78)
    piv = _s[_s.sigma == 8].pivot_table(index="impl_name", columns="subset",
                                        values="faithfulness_image_del_gain", aggfunc="mean")
    print(piv.round(3).to_string())
    print("\nrank per subset:"); print(piv.rank(ascending=False).astype(int).to_string())
    print("\nIf the ranking reshuffles across subsets, the fusion claim rests on one lucky batch.")

    print("\ndeletion <-> insertion correlation, per subset (published: r = +0.008 on the image side):")
    for k in sorted(_s.subset.unique()):
        d = _s[(_s.subset == k) & (_s.sigma == 8)]
        if {"faithfulness_image_del_gain", "faithfulness_image_ins_gain"} <= set(d.columns):
            print(f"  subset {k}: r = {d.faithfulness_image_del_gain.corr(d.faithfulness_image_ins_gain):+.3f}")

if S3_DO_SIGMA:
    print("\n" + "=" * 78); print("IS THE DIVERGENCE AN OOD ARTIFACT OF THE BLUR BASELINE?"); print("=" * 78)
    print(_s[_s.subset == 0].pivot_table(index="impl_name", columns="sigma",
                                         values="faithfulness_image_del_gain").round(3).to_string())